# BioHub Cell Tracking

One input: everything the run loads lives in the `antonoof/all_files` dataset.

| attach | for |
| --- | --- |
| `antonoof/all_files` | weights, runtimes, offline wheels |
| `competitions/biohub-cell-tracking-during-development` | test volumes, and `train/` for local scoring |

Accelerator **GPU T4 x2**. No internet needed: dependencies install from
`all_files/wheels`.

## Fixes in this notebook

**The gap closer no longer takes a node twice.** `isolated_by_t` is a snapshot
taken before the first gap is closed, and start frames are visited before
midpoint frames, so a node that had already taken a parent as one gap's start
could be taken again as the next gap's midpoint. That is in-degree 2, which
`validate_final_graph_contract` rejects, and it kills the video. Pool candidates
are now screened against the live `incoming`/`outgoing` sets
(`gap_reuse_stale_skipped`).

**A video with no native evidence keeps its combined pass.** Past the helper
deadline a video is predicted without the helper models, so no `.npz` is
written. The frozen code calls `_evidence()` unconditionally, raises
`FileNotFoundError`, and the worker falls back to a geometry-only shard,
losing the division decoder and every graft stage. `evidence_ready` now skips
only the stages that read those archives (`native_evidence_missing`).

## Measured offline

The real detector produces the nodes, the notebook's own `motion_relink_edges`
links them, and the competition metric scores the result
(`synthetic/offline_pipeline.py`, `synthetic/metric.py`). Six non-scored videos
tune, the scored ones are held out.

| mechanism | CFG key | verdict |
| --- | --- | --- |
| margin refusal | `relink_margin_um` | off: -0.0147 tuning, -0.0090 held out |
| multi-step prior | `relink_motion_steps` | off: -0.0025 tuning, 0.0000 held out |
| add-only divisions | `addonly_divisions` | see the division section below |

Thresholds (`relink_max_match_cost`, `relink_tight_bonus_um`, `min_track_len`,
`gap_close_um`) were swept and left frozen: what the tuning set prefers, the
held-out set loses, which is the same non-transfer already seen on the
leaderboard.


In [ ]:
"""CFG - the only place where the bundle layout and the pipeline knobs are set."""
import json
import os
import time
from pathlib import Path

NOTEBOOK_START = time.monotonic()

INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
PKG_DIR = WORKING_DIR / "biohub_pipeline"
PKG_DIR.mkdir(parents=True, exist_ok=True)


def find_bundle(root: Path) -> Path:
    """The one attached directory carrying BUNDLE.json. Every model lives there."""
    preferred = root / "datasets" / "antonoof" / "all_files"
    if (preferred / "BUNDLE.json").is_file():
        return preferred
    found = sorted({
        manifest.parent
        for depth in ("*", "*/*", "*/*/*")
        for manifest in root.glob(f"{depth}/BUNDLE.json")
    })
    if len(found) != 1:
        raise RuntimeError(
            f"Attach exactly one antonoof/all_files bundle; found {found or 'none'}"
        )
    return found[0]


BUNDLE = find_bundle(INPUT_DIR)

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = next(
    (
        path
        for path in (INPUT_DIR / "competitions" / COMPETITION, INPUT_DIR / COMPETITION)
        if path.exists()
    ),
    INPUT_DIR / "competitions" / COMPETITION,
)

CFG = {
    "experiment_tag": "biohub_public953_joint_assignment_v1",
    "voxel_scale_um": [1.625, 0.40625, 0.40625],

    # Every path points inside the one attached bundle. Layout: BUNDLE.json.
    "models": {
        "p1": {
            "root": str(BUNDLE / "p1"),
            "weights_rel": "weights/unet_transformer/split_0/edge_predictor_best.pth",
            "sha256": "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771",
        },
        "p2": {
            "weights": str(
                BUNDLE / "p2" / "weights" / "unet_transformer" / "split_0"
                / "edge_predictor_best.pth"
            ),
            "sha256": "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f",
        },
        "model_c": {
            "dir": str(BUNDLE / "model_c"),
            "checkpoint_rel": "weights/edge_predictor_best.pth",
        },
        "source_cardinality": {
            "dir": str(BUNDLE / "source_cardinality"),
            "enabled": True,
        },
        "unigraft_p1p2": {
            "dir": str(BUNDLE / "unigraft_p1p2"),
            "enabled": True,
        },
        "live_v2_global_bundle": {
            "dir": str(BUNDLE / "live_v2_bundle"),
            "manifest_version": "biohub-live-v2-global-bundle-v2",
            "enabled": True,
        },
        "ownership_full_population": {
            "dir": str(BUNDLE / "ownership"),
            "enabled": True,
        },
        "edgegraft": {
            "dir": str(BUNDLE / "edgegraft"),
            "enabled": True,
        },
        "candidategraft": {
            "dir": str(BUNDLE / "candidategraft"),
            "enabled": True,
        },
        "basegraph_repair": {
            "dir": str(BUNDLE / "basegraph_repair"),
            "manifest_version": "biohub-basegraph-repair-v1",
            "enabled": True,
        },
        "motion_corrector": {
            "path": str(BUNDLE / "motion_corrector" / "motion_corrector_best.pt"),
            "enabled": True,
            "strength": 1.0,
        },
        "deepcenter": {
            "path": str(BUNDLE / "deepcenter" / "best.pt"),
            "sha256": "8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0",
            "expected_epoch": 2,
            "enabled": True,
        },
    },

    # ===================== detection / association ========================
    "detection": {
        "threshold": 0.96875,
        "unet_batch_size": 4,
        # Parabolic sub-voxel refinement of every detection peak. Offsets are
        # clamped to +-0.49 voxel, so rounding still lands on the original
        # lattice cell and every tensor the networks see is unchanged.
        "subvoxel_refinement": True,
    },
    "association": {
        "secondary_edge_weight": 0.15,
        "secondary_detection_weight": 0.475,
        "secondary_link_mode": "low_margin_consensus",
        "secondary_mix_temperature": 1.0,
        "secondary_low_margin_max": 0.35,
        "dual_seed_edge_threshold": 0.48,
        "minimum_candidate_retention": 0.90,
        "guarded_secondary_edge_weight": 0.0,
        # Harmonic forward/reverse mutual-support fusion, in probability space.
        "bidirectional_edge_weight": 0.30,
    },
    "ilp": {
        "use": True,
        "edge_weight": -1.0,
        "appearance": 0.0,
        "disappearance": 1.5,
        "division": 1.0,
    },

    # ===================== output graph post-processing ====================
    "graph": {
        "edge_max_um": 14.0,
        "enforce_next_frame": True,
        "single_parent_repair": True,
        "prune_isolated": True,

        "motion_relink": True,
        "relink_tight_um": 6.0,
        "relink_relaxed_um": 10.0,
        # One solve over the relaxed candidate set, with a discount inside the
        # tight gate, instead of two greedy passes. Not gate widening.
        "relink_joint_assignment": True,
        # Discount, in um, applied to a candidate whose raw distance is inside
        # relink_tight_um. Large enough to keep "tight first" as a strong
        # preference; 0 degenerates to a plain single relaxed pass, which is
        # the configuration that lost.
        "relink_tight_bonus_um": 3.0,
        "relink_velocity_weight": 0.5,
        # Global frame shift, fed to the prediction and to the corrector's
        # shift_* features, which the frozen code left as zeros.
        "relink_frame_registration": True,
        "relink_registration_weight": 1.0,
        # Per-axis velocity weights; z is mostly localisation noise, so 0.
        "relink_velocity_weight_axes": [0.0, 0.45, 0.47],
        # Steps averaged into the velocity; 1 is the frozen single-step prior.
        # Measured: residual drops, linking does not move (-0.0025, 0.0000).
        "relink_motion_steps": 1,
        "relink_motion_axes": [0.5, 0.5, 0.5],
        "relink_learned_bonus": 1.0,
        # Refuse an assignment whose geometric cost exceeds this, leaving both
        # endpoints open. 7.5 clips 1.13 % of true continuations; 6.0 clips
        # 2.59 % and 4 of 6 division legs. 0 disables it.
        "relink_max_match_cost": 7.5,
        # Decline a match whose runner-up, for the source or the target, is
        # within this margin. Measured and rejected: -0.0147 tuning, -0.0090
        # held out - the impostor is closer than the true parent, not equal.
        "relink_margin_um": 0.0,
        # The cap is on geometry alone. Including the learned bonus and the
        # corrector residual would let them buy a long jump, which is the
        # failure being fixed rather than a way to fix it.
        "relink_cost_cap_includes_learned": False,

        # Per-target no-parent option priced by its best association evidence.
        "relink_orphan_prior": False,
        "relink_orphan_base_um": 7.5,
        "relink_orphan_scale": 1.0,
        # Floor on the decline cost; without it a target the ILP never
        # parented would be dropped unconditionally.
        "relink_orphan_floor": 0.5,
        "relink_raw_distance_weight": 0.05,
        # Frames larger than this fall back to the raw ILP edges. The gated,
        # vectorised assignment makes 4000 as cheap as 2600 used to be, so
        # dense videos now keep the learned re-link instead of dropping it.
        "relink_max_frame_nodes": 4000,
        "motion_feature_z_extent_um": 102.375,

        "gap_close": True,
        "gap_close_um": 5.8,
        "gap_density_adaptive": True,
        "gap_density_reference_um": 6.5,
        "gap_density_gain": 0.040,
        "gap_density_max_step_delta_um": 0.125,
        "gap_density_neighbors": 3,
        # Prefer re-using an already detected orphan over inventing a node:
        # a reused node costs nothing in the node-count penalty.
        "gap_reuse_existing": True,
        "gap_reuse_um": 4.0,
        "gap_max_added_frac": 0.03,
        "gap_max_added_abs": 2000,
        "gap_refine_synthetic": True,
        "gap_refine_win_z": 1,
        "gap_refine_win_yx": 3,
        "gap_refine_max_shift_um": 3.2,

        # Veto only long synthetic gaps. Real/reused detections and total
        # gap spans below 8.5 um retain the frozen .952 behavior.
        "deepcenter_gap_veto": True,
        "deepcenter_gap_threshold": 0.25,
        "deepcenter_gap_confirm_min_span_um": 8.5,
        "deepcenter_score_win_z": 1,
        "deepcenter_score_win_yx": 2,
        "deepcenter_score_cache_max_frames": 8,
        "deepcenter_device": "cpu",

        "safe_divisions": True,
        "safe_div_max_um": 4.66,
        "safe_div_sister_max_um": 8.5,
        "safe_div_existing_child_max_um": 7.65,
        "safe_div_sister_score_weight": 0.15,
        "safe_div_frame_frac_cap": 0.0076,
        "safe_div_global_frac_cap": 0.00375,

        "filter_short_tracks": True,
        "min_track_len": 6,
        "keep_division_components": True,
        # Preserve an already-linked short component only when the movie
        # boundary makes the normal six-node minimum impossible to observe.
        "boundary_track_rescue": True,
        "boundary_track_min_len": 2,

        "linefit_smooth": True,
        "linefit_weight": 0.8,
        "linefit_window": 3,

        # Node budget: invented nodes are charged by the adjusted Jaccard.
        # Components holding a fork are never dropped.
        "node_budget": True,
        "node_budget_ratio": 1.0,
        "node_budget_max_drop_frac": 0.03,

        "frame_cache_max_frames": 8,
    },
    "division": {
        "combined_decoder": True,
        "model_c_threshold": 0.96,
        "gbm_threshold": 0.65,
        "gbm_rescue_delta": 0.15,
        "gbm_steal_delta": 0.25,
        # Add-only second daughter on the final graph. Adds edges only: no new
        # node, only targets nothing else claims, only sources with one child,
        # one fork per lineage. Gates come from the 151 training divisions.
        "addonly_divisions": False,
        "addonly_div_min_um": 0.5,
        "addonly_div_max_um": 10.0,
        "addonly_div_sister_min_um": 4.0,
        "addonly_div_sister_max_um": 15.0,
        "addonly_div_max_cos": -0.5,
        "addonly_div_budget_frac": 0.002,
    },

    # =============================== speed ================================
    # Scheduling and implementation only. No entry below changes a model, a
    # threshold, a gate, or a weight.
    "speed": {
        # -- exact: identical graphs, identical shards ----------------------
        # Replaces the predictor's per-cell Python scan of the (n_src, n_tgt)
        # probability matrix. The softmax runs over sources, so at the 0.48
        # candidate threshold at most two cells per column can survive: the
        # scan was pure overhead. Ordering is reproduced exactly.
        "vectorized_candidates": True,
        # Columnar .geff -> dict conversion instead of per-row iteration.
        "fast_geff_reader": True,
        # Byte-identical CSV assembled without csv.DictWriter per row.
        "fast_shard_writer": True,
        # process_graph deep-copied every node and edge for the graft stages,
        # then the BaseGraph repair overwrote both copies.
        "skip_redundant_raw_copies": True,
        # The anytime baseline was a full second post-processing pass over
        # every video, run serially in the notebook process and discarded by
        # the upgrade that followed it. Produce it only where it is needed.
        "lazy_baseline": True,
        # Longest-processing-time-first for both GPU production and CPU
        # dispatch.
        "longest_first": True,
        # Per-stage seconds in run_stats.csv.
        "stage_timing": True,
        # Evidence archives are written once and re-read many times, so the
        # inflate dominates. Identical arrays; oversized files stay compressed.
        "uncompressed_evidence": True,
        "uncompressed_evidence_max_mb": 512,

        # -- artifact runtime accelerator -----------------------------------
        # Verified drop-in replacements for the hot helpers inside the frozen
        # artifact runtimes. Measured on EdgeGRAFT V3 at 71k nodes: 126 s ->
        # 62 s, bit-identical over 120,998 candidate rows x 184 columns.
        # Every replacement must reproduce the function it replaces bit for
        # bit on an adversarial battery before it is installed; anything that
        # fails is left frozen and named in the log.
        "runtime_accel": True,
        # "off" | "smallest" | "all". Wraps every function in the loaded
        # artifact runtimes with a wall-clock counter, reports self time, and
        # unwraps afterwards, so only the profiled video pays. This is how the
        # stages this repo cannot read get a target list.
        "function_profile": "smallest",

        # -- adaptive: same arithmetic, different thread counts -------------
        # Workers were pinned to one thread for the entire run, including the
        # long tail where one video holds the machine alone. The budget now
        # tracks live upgrades and live GPU producers.
        "thread_boost": True,
        "thread_pool_total": 0,          # 0 -> os.cpu_count()

        # -- GPU algorithm selection ---------------------------------------
        # Same math, cuDNN may pick a different convolution algorithm. Every
        # window of a video has the same shape, so one autotune amortises
        # over hundreds of identical launches.
        "cudnn_benchmark": True,
        # Opt-in and unscored: fp16 autocast changes the detection logits and
        # therefore the detected node set.
        "amp_fp16": False,
    },

    # ======================= local scoring / metrics =======================
    # Used by the final cell. `gt_dir` empty -> the labelled train/ split of
    # the attached competition data, when it is present.
    "metrics": {
        "enabled": True,
        "max_match_um": 7.0,
        "adjustment_alpha": 0.1,
        "division_weight": 0.1,
        "gt_dir": "",
        "max_videos": 0,
        "report_path": str(WORKING_DIR / "local_metrics.csv"),
    },

    # ============================== runtime ===============================
    "runtime": {
        "gpu_workers": 2,
        # 0 -> os.cpu_count(). The upgrade pool is what the makespan is made
        # of, so it tracks the machine instead of a constant.
        "cpu_workers": 0,
        # 0 -> os.cpu_count() - gpu_workers. The GPU producers need CPU as
        # well: four upgrades plus two producers on four vCPUs oversubscribed
        # both halves during the production phase.
        "cpu_workers_while_predicting": 0,
        "hard_limit_seconds": 11 * 3600 + 50 * 60,
        "finalize_reserve_seconds": 600,
        "min_upgrade_seconds": 60,
        # Last-resort shard from the raw producer graph, reachable only when
        # both the combined upgrade and the baseline pass have failed for a
        # video. A degraded shard still scores; a missing one makes the whole
        # submission unassemblable.
        "emergency_shard": True,
        "slice": "",              # e.g. ":1" for a smoke test
        "allow_pip_install": False,
    },
}

CFG["paths"] = {
    "competition_dir": str(COMP_DIR),
    "test_dir": os.environ.get("BIOHUB_TEST_DIR", "").strip() or str(COMP_DIR / "test"),
    "working_dir": str(WORKING_DIR),
    "package_dir": str(PKG_DIR),
    "repo_dir": str(WORKING_DIR / "tracking_repo"),
    "shard_dir": str(WORKING_DIR / "submission_shards"),
    "model_c_evidence_dir": str(WORKING_DIR / "evidence_model_c"),
    "p1_evidence_dir": str(WORKING_DIR / "evidence_p1"),
    "p2_evidence_dir": str(WORKING_DIR / "evidence_p2"),
    "submission": str(WORKING_DIR / "submission.csv"),
    "run_stats": str(WORKING_DIR / "run_stats.csv"),
    "cfg_json": str(PKG_DIR / "cfg.json"),
}

TEST_DIR = Path(CFG["paths"]["test_dir"])
REPO_DIR = Path(CFG["paths"]["repo_dir"])
SUBMISSION_PATH = Path(CFG["paths"]["submission"])
RUN_STATS_PATH = Path(CFG["paths"]["run_stats"])


def export_cfg() -> None:
    """Publish CFG to the child predictor process and to the .py modules."""
    association = CFG["association"]
    environment = {
        "BIOHUB_SECONDARY_EDGE_WEIGHT": association["secondary_edge_weight"],
        "BIOHUB_SECONDARY_DETECTION_WEIGHT": association["secondary_detection_weight"],
        "BIOHUB_SECONDARY_LINK_MODE": association["secondary_link_mode"],
        "BIOHUB_SECONDARY_MIX_TEMPERATURE": association["secondary_mix_temperature"],
        "BIOHUB_SECONDARY_LOW_MARGIN_MAX": association["secondary_low_margin_max"],
        "BIOHUB_DUAL_SEED_EDGE_THRESHOLD": association["dual_seed_edge_threshold"],
        "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": association["minimum_candidate_retention"],
        "BIOHUB_RETENTION_GUARD_SECONDARY_EDGE_WEIGHT": association["guarded_secondary_edge_weight"],
        "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": association["bidirectional_edge_weight"],
        "BIOHUB_SECONDARY_WEIGHTS": CFG["models"]["p2"]["weights"],
        "BIOHUB_MODEL_C_CHECKPOINT": str(
            Path(CFG["models"]["model_c"]["dir"]) / CFG["models"]["model_c"]["checkpoint_rel"]
        ),
        "BIOHUB_MODEL_C_EVIDENCE_DIR": CFG["paths"]["model_c_evidence_dir"],
        "BIOHUB_PUBLIC_P1_EVIDENCE_DIR": CFG["paths"]["p1_evidence_dir"],
        "BIOHUB_PUBLIC_P2_EVIDENCE_DIR": CFG["paths"]["p2_evidence_dir"],
        "BIOHUB_REPO_DIR": CFG["paths"]["repo_dir"],
        "BIOHUB_CFG_JSON": CFG["paths"]["cfg_json"],
        "BIOHUB_SUBVOXEL_DETECTION": int(bool(CFG["detection"]["subvoxel_refinement"])),
        "BIOHUB_FAST_CANDIDATES": int(bool(CFG["speed"]["vectorized_candidates"])),
        "BIOHUB_CUDNN_BENCHMARK": int(bool(CFG["speed"]["cudnn_benchmark"])),
        "BIOHUB_AMP_FP16": int(bool(CFG["speed"]["amp_fp16"])),
        "BIOHUB_UNCOMPRESSED_EVIDENCE": int(bool(CFG["speed"]["uncompressed_evidence"])),
        "BIOHUB_UNCOMPRESSED_EVIDENCE_MAX_BYTES": int(
            float(CFG["speed"]["uncompressed_evidence_max_mb"]) * (1 << 20)
        ),
    }
    for key, value in environment.items():
        os.environ[key] = str(value)
    Path(CFG["paths"]["cfg_json"]).write_text(json.dumps(CFG, indent=2))


export_cfg()
for _old_guard_log in WORKING_DIR.glob("retention_guard_*.jsonl"):
    _old_guard_log.unlink()

print(f"tag={CFG['experiment_tag']} | test_dir={TEST_DIR} (exists={TEST_DIR.exists()})")
print(f"bundle={BUNDLE}")
print(
    f"sub-voxel detection: {CFG['detection']['subvoxel_refinement']} | "
    f"line-fit window: {CFG['graph']['linefit_window']} | "
    f"node budget: {CFG['graph']['node_budget_ratio']:.2f} x detected"
)

_SPEED = CFG["speed"]
_CPU_TOTAL = int(_SPEED["thread_pool_total"] or 0) or (os.cpu_count() or 4)
print(
    f"speed: {_CPU_TOTAL} vCPU | lazy baseline={_SPEED['lazy_baseline']} | "
    f"longest-first={_SPEED['longest_first']} | thread boost={_SPEED['thread_boost']} | "
    f"cudnn.benchmark={_SPEED['cudnn_benchmark']} | fp16={_SPEED['amp_fp16']}"
)
print(
    f"runtime accelerator: {_SPEED['runtime_accel']} | "
    f"function profile: {_SPEED['function_profile']}"
)
if _SPEED["amp_fp16"]:
    print("WARNING: fp16 autocast is ON. Detection logits change; this path is unscored.")


In [ ]:
"""Runtime check, offline dependencies, and one verification pass over the bundle."""
import hashlib
import importlib
import importlib.util
import shutil
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required; enable the Kaggle GPU accelerator.")
print("CUDA:", torch.cuda.get_device_name(0), "| devices:", torch.cuda.device_count())

# --------------------------------------------------------------------------
# Offline dependencies. Every wheel ships in BUNDLE/wheels, and
# runtime.allow_pip_install stays False, so the run never needs the network.
# --------------------------------------------------------------------------
PACKAGE_SPECS = {
    "tracksdata": "tracksdata", "zarr": "zarr>=3.0.10,<4", "pyscipopt": "pyscipopt",
    "geff": "geff>=1.1.3.1.1", "geff_spec": "geff-spec<1.2", "ilpy": "ilpy>=0.5.1",
    "polars": "polars>=1.36", "blosc2": "blosc2", "dask": "dask",
    "imagecodecs": "imagecodecs", "skimage": "scikit-image>=0.24", "pyarrow": "pyarrow",
    "rustworkx": "rustworkx>=0.17.1", "sqlalchemy": "sqlalchemy>=2",
    "numcodecs": "numcodecs>=0.13,<0.16", "donfig": "donfig>=0.8",
    "google_crc32c": "google-crc32c>=1.5", "bidict": "bidict>=0.23.1",
    "psygnal": "psygnal>=0.14", "rich": "rich", "networkx": "networkx>=3.2.1",
    "pydantic": "pydantic>=2.11", "pydantic_core": "pydantic-core",
    "annotated_types": "annotated-types", "typing_extensions": "typing-extensions>=4.13",
    "typing_inspection": "typing-inspection", "markdown_it": "markdown-it-py",
    "pygments": "pygments", "click": "click", "cloudpickle": "cloudpickle",
    "fsspec": "fsspec", "partd": "partd", "locket": "locket", "toolz": "toolz",
    "yaml": "pyyaml", "ndindex": "ndindex", "msgpack": "msgpack", "numexpr": "numexpr",
    "deprecated": "deprecated", "wrapt": "wrapt", "imageio": "imageio", "PIL": "pillow",
    "tifffile": "tifffile", "lazy_loader": "lazy-loader", "tqdm": "tqdm",
    "threadpoolctl": "threadpoolctl",
}
EXTRA_SPECS = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"], "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"], "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13",
                 "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
os.environ.setdefault("POLARS_PREFER_PKG", "32")

WHEEL_DIR = BUNDLE / "wheels"


def _install(names: list[str]) -> None:
    specs, seen = [], set()
    for name in names:
        for spec in [PACKAGE_SPECS.get(name, name), *EXTRA_SPECS.get(name, [])]:
            if spec.lower() not in seen:
                seen.add(spec.lower())
                specs.append(spec)
    command = [sys.executable, "-m", "pip", "install", "--no-deps", "-q"]
    if {"polars", "zarr"} & set(names):
        command.append("--force-reinstall")
    offline = [*command, "--no-index", "--find-links", str(WHEEL_DIR)]
    if subprocess.run([*offline, *specs], text=True, capture_output=True).returncode == 0:
        return
    if CFG["runtime"]["allow_pip_install"]:
        if subprocess.run([*command, *specs], text=True, capture_output=True).returncode == 0:
            return
    raise ImportError(f"Could not install {names} from {WHEEL_DIR}")


def _importable(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


def _runtime_healthy() -> list[str]:
    """Packages that are importable but unusable in this kernel."""
    broken = []
    try:
        import polars as pl
        from polars._plr import PySeries  # noqa: F401

        if not hasattr(pl, "Float16"):
            broken.append("polars")
    except Exception:
        broken.append("polars")
    try:
        import zarr

        if int(str(zarr.__version__).split(".", 1)[0]) < 3:
            broken.append("zarr")
    except Exception:
        broken.append("zarr")
    return broken


def ensure_dependencies() -> None:
    for _ in range(5):
        broken = [name for name in PACKAGE_SPECS if not _importable(name)]
        broken += [name for name in _runtime_healthy() if name not in broken]
        if not broken:
            return
        _install(broken)
        for name in list(sys.modules):
            if name.split(".")[0] in {n.split(".")[0] for n in broken}:
                sys.modules.pop(name, None)
    raise ImportError("Dependency recovery did not converge.")


ensure_dependencies()


# --------------------------------------------------------------------------
# The bundle. One table of everything the run needs, checked once, up front:
# a missing or wrong artifact fails here instead of degrading a 1.5 h run.
# --------------------------------------------------------------------------
_M = CFG["models"]
P1_ROOT = Path(_M["p1"]["root"])
P1_WEIGHTS_REL = _M["p1"]["weights_rel"]
P2_WEIGHTS = Path(_M["p2"]["weights"])
MODEL_C_DIR = Path(_M["model_c"]["dir"])
CARDINALITY_DIR = Path(_M["source_cardinality"]["dir"])
UNIGRAFT_DIR = Path(_M["unigraft_p1p2"]["dir"])
LIVE_V2_DIR = Path(_M["live_v2_global_bundle"]["dir"])
OWNERSHIP_DIR = Path(_M["ownership_full_population"]["dir"])
EDGEGRAFT_DIR = Path(_M["edgegraft"]["dir"])
CANDIDATEGRAFT_DIR = Path(_M["candidategraft"]["dir"])
BASEGRAPH_DIR = Path(_M["basegraph_repair"]["dir"])

REQUIRED: list[tuple[str, Path]] = [
    ("P1 inference repo", P1_ROOT / "repo" / "scripts" / "predict_unet_transformer.py"),
    ("P1 weights", P1_ROOT / P1_WEIGHTS_REL),
    ("P1 config", (P1_ROOT / P1_WEIGHTS_REL).parent / "config.json"),
    ("P2 weights", P2_WEIGHTS),
    ("P2 config", P2_WEIGHTS.parent / "config.json"),
    ("DeepCenter veto", Path(_M["deepcenter"]["path"])),
    ("motion corrector", Path(_M["motion_corrector"]["path"])),
    ("Model C weights", MODEL_C_DIR / _M["model_c"]["checkpoint_rel"]),
    ("Model C config", (MODEL_C_DIR / _M["model_c"]["checkpoint_rel"]).parent / "config.json"),
    ("division GBM runtime", MODEL_C_DIR / "division_gbm_runtime.py"),
    ("division feature API", MODEL_C_DIR / "train_division_pair_model.py"),
    ("Model C safe runtime", MODEL_C_DIR / "model_c_safe_runtime.py"),
    ("division GBM gates", MODEL_C_DIR / "deploy_spec.json"),
    ("cardinality head", CARDINALITY_DIR / "source_cardinality_head.pt"),
    ("cardinality runtime", CARDINALITY_DIR / "source_cardinality_graph_runtime.py"),
    ("UniGRAFT head", UNIGRAFT_DIR / "p1p2_only_source_cardinality_head.pt"),
    ("UniGRAFT runtime", UNIGRAFT_DIR / "unigraft_p1p2_late_ensemble_runtime.py"),
    ("live V2 runtime", LIVE_V2_DIR / "live_v2_global_bundle_runtime_v1.py"),
    ("ownership runtime", OWNERSHIP_DIR / "ownership_full_population_runtime_v1.py"),
    ("ownership gates", OWNERSHIP_DIR / "deploy_spec.json"),
    ("EdgeGRAFT runtime", EDGEGRAFT_DIR / "edgegraft_v3_deploy_runtime_v5.py"),
    ("CandidateGRAFT runtime", CANDIDATEGRAFT_DIR / "candidategraft_direct_runtime_v1.py"),
    ("CandidateGRAFT model", CANDIDATEGRAFT_DIR / "candidategraft_direct.joblib"),
    ("BaseGraph runtime", BASEGRAPH_DIR / "basegraph_candidate_fusion_runtime_v1.py"),
    ("offline wheels", WHEEL_DIR),
]
REQUIRED += [
    (f"EdgeGRAFT {family} fold {fold}", EDGEGRAFT_DIR / family / f"fold_{fold}.joblib")
    for family in ("ranker", "metric")
    for fold in range(5)
]
REQUIRED += [
    (f"EdgeGRAFT {family} gates", EDGEGRAFT_DIR / family / "report.json")
    for family in ("ranker", "metric")
]
REQUIRED += [
    (f"ownership fold {fold}", OWNERSHIP_DIR / "models" / f"ownership_fold_{fold}.joblib")
    for fold in range(5)
]

missing = [f"{label}: {path}" for label, path in REQUIRED if not path.exists()]
if missing:
    raise FileNotFoundError(
        "The attached bundle is incomplete:\n  " + "\n  ".join(missing)
    )


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


for label, path, expected in (
    ("P1", P1_ROOT / P1_WEIGHTS_REL, _M["p1"]["sha256"]),
    ("P2", P2_WEIGHTS, _M["p2"]["sha256"]),
    ("DeepCenter", Path(_M["deepcenter"]["path"]), _M["deepcenter"]["sha256"]),
):
    digest = _sha256(path)
    if digest != expected:
        raise RuntimeError(f"{label} checksum mismatch: {digest} != {expected}")

# Version guards. The live V2 bundle has a quarantined v1 with the same file
# names, and the BaseGraph runtime is only valid against its own manifest.
for key in ("live_v2_global_bundle", "basegraph_repair"):
    manifest = json.loads(
        (Path(_M[key]["dir"]) / "manifest.json").read_text(encoding="utf-8")
    )
    if manifest.get("version") != _M[key]["manifest_version"]:
        raise RuntimeError(
            f"{key}: manifest version {manifest.get('version')} != "
            f"{_M[key]['manifest_version']}"
        )
if json.loads((LIVE_V2_DIR / "manifest.json").read_text(encoding="utf-8")).get("status") != "candidate":
    raise RuntimeError("live V2 bundle is not the candidate build")

# --------------------------------------------------------------------------
# Working copy of the inference repo. The predictor is patched in place by the
# next cell, so it cannot run from the read-only dataset. Weights stay in the
# bundle and are linked in, because `--weights` is resolved against the repo.
# --------------------------------------------------------------------------
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(P1_ROOT / "repo", REPO_DIR)
try:
    os.symlink(P1_ROOT / "weights", REPO_DIR / "weights", target_is_directory=True)
except OSError:
    shutil.copytree(P1_ROOT / "weights", REPO_DIR / "weights")

# Fresh evidence and shard directories for this run.
for key in ("model_c_evidence_dir", "p1_evidence_dir", "p2_evidence_dir", "shard_dir"):
    directory = Path(CFG["paths"][key])
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)

export_cfg()

TEST_STEMS = sorted(
    path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr")
)
if not TEST_STEMS:
    raise FileNotFoundError(f"No test .zarr volumes in {TEST_DIR}")


print(f"\nbundle: {BUNDLE}")
print(
    f"{len(REQUIRED)} required artifacts present, 3 checksums verified, "
    f"2 manifests matched"
)
print(f"test videos: {len(TEST_STEMS)} | repo: {REPO_DIR}")


## 1. Predictor patch

The inference script is copied into `/kaggle/working` and rewritten there.
Every replacement must match exactly once, so a drift fails the run instead of
degrading it silently.


In [ ]:
%%writefile /kaggle/working/biohub_pipeline/patch_predictor.py
"""In-place patches for the artifact copy of `predict_unet_transformer.py`.

Run as a script (the notebook does this once, before any GPU work):

    BIOHUB_REPO_DIR=/kaggle/working/tracking_repo python patch_predictor.py

Every replacement is asserted to match exactly once, so a silent mismatch
with a different artifact build fails the run instead of degrading it.
"""
from __future__ import annotations

import os
from pathlib import Path

_ps = Path(os.environ["BIOHUB_REPO_DIR"]) / "scripts" / "predict_unet_transformer.py"


def _apply(source: str, replacements, label: str) -> str:
    for index, (old, new) in enumerate(replacements, start=1):
        count = source.count(old)
        if count != 1:
            raise RuntimeError(f"{label} patch {index}: expected 1 match, found {count}")
        source = source.replace(old, new, 1)
    compile(source, str(_ps), "exec")
    return source


_s = _ps.read_text()

# 1. Eight-view (D4) planar detection TTA in place of the default four flips.
_old = """        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4"""
_new = """        if cfg.det_tta:
            _nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                _nv += 1
            for _k in (1, 3):
                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))
                del imgs_rot, det_rot
                _nv += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            _nv += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            _nv += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / _nv"""
_s = _apply(_s, [(_old, _new)], "detection TTA")

# 2. Calibrated dual-seed ensemble: second temporal model on the same graph.
_ensemble_replacements = [
    ('    downsample: tuple[int, ...] = (1, 4, 4),\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    downsample: tuple[int, ...] = (1, 4, 4),\n    secondary_model: UNetNodeTransformer | None = None,\n    secondary_edge_weight: float = 0.0,\n    secondary_detection_weight: float = 0.0,\n    secondary_link_mode: str = "raw",\n    secondary_mix_temperature: float = 1.0,\n    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'),
    ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        del imgs', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:\n                if cfg.det_tta:\n                    _secondary_nv = 1\n                    for dims in [(-1,), (-2,), (-2, -1)]:\n                        secondary_imgs_flip = imgs.flip(dims)\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\n                        for f in range(W):\n                            secondary_det_logits[f] = (\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\n                            )\n                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\n                        for f in range(W):\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\n                            )\n                        del secondary_imgs_rot, secondary_det_rot\n                        _secondary_nv += 1\n                    secondary_imgs_t = imgs.transpose(-1, -2)\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\n                    for f in range(W):\n                        secondary_det_logits[f] = (\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\n                        )\n                    del secondary_imgs_t, secondary_det_t\n                    _secondary_nv += 1\n                    secondary_imgs_at = torch.rot90(\n                        imgs, 1, dims=(-2, -1)\n                    ).transpose(-1, -2)\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                            secondary_det_at[f].transpose(-1, -2),\n                            -1,\n                            dims=(-2, -1),\n                        )\n                    del secondary_imgs_at, secondary_det_at\n                    _secondary_nv += 1\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):\n                    primary_det = det_logits[f]\n                    secondary_det = secondary_det_logits[f]\n                    primary_mean = primary_det.mean()\n                    secondary_mean = secondary_det.mean()\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_det_aligned = (\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\n                    )\n                    det_logits[f] = (\n                        (1.0 - secondary_detection_weight) * primary_det\n                        + secondary_detection_weight * secondary_det_aligned\n                    )\n\n            del secondary_det_logits\n\n        del imgs'),
    ('            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            raw = edge_logits_pair[0]', '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n                if secondary_unet_out is None:\n                    raise RuntimeError("Secondary model is loaded but its feature map is missing")\n                secondary_feat_src = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\n                )\n                secondary_feat_tgt = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n                )\n                secondary_logits_pair = secondary_model.predict_edges(\n                    secondary_feat_src, secondary_feat_tgt,\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                    p_pos_src, p_pos_tgt,\n                    p_mask_src, p_mask_tgt,\n                )\n\n                if secondary_link_mode == "raw":\n                    secondary_for_mix = secondary_logits_pair\n                    blend_weight = secondary_edge_weight\n                elif secondary_link_mode in {\n                    "calibrated", "adaptive", "low_margin_consensus"\n                }:\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    primary_scale = edge_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\n                    secondary_scale = secondary_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_for_mix = (\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\n                        + primary_center\n                    )\n                    if secondary_link_mode == "calibrated":\n                        blend_weight = secondary_edge_weight\n                    elif secondary_link_mode == "adaptive":\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\n                            local_weight = (\n                                secondary_edge_weight + secondary_margin - primary_margin\n                            ).clamp(0.15, 0.75)\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            local_weight = torch.where(\n                                same_parent,\n                                torch.maximum(\n                                    local_weight,\n                                    torch.full_like(local_weight, secondary_edge_weight),\n                                ),\n                                local_weight,\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = secondary_edge_weight\n                    else:\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            uncertainty = (\n                                (secondary_low_margin_max - primary_margin)\n                                / secondary_low_margin_max\n                            ).clamp(0.0, 1.0)\n                            local_weight = secondary_edge_weight * uncertainty\n                            local_weight = torch.where(\n                                same_parent,\n                                local_weight,\n                                torch.zeros_like(local_weight),\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = 0.0\n                else:\n                    raise ValueError(f"Unsupported secondary link mode: {secondary_link_mode}")\n\n                edge_logits_pair = (\n                    (1.0 - blend_weight) * edge_logits_pair\n                    + blend_weight * secondary_for_mix\n                )\n                if secondary_mix_temperature != 1.0:\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    edge_logits_pair = mixed_center + (\n                        edge_logits_pair - mixed_center\n                    ) / secondary_mix_temperature\n\n            raw = edge_logits_pair[0]'),
    ('        del unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n'),
    ('    model, window_size, downsample = load_model(weights_path, device)\n    print(', '    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None\n    secondary_weights_text = os.environ.get("BIOHUB_SECONDARY_WEIGHTS", "").strip()\n    secondary_edge_weight = float(os.environ.get("BIOHUB_SECONDARY_EDGE_WEIGHT", "0"))\n    secondary_detection_weight = float(\n        os.environ.get("BIOHUB_SECONDARY_DETECTION_WEIGHT", "0")\n    )\n    secondary_link_mode = os.environ.get("BIOHUB_SECONDARY_LINK_MODE", "raw").strip()\n    secondary_mix_temperature = float(\n        os.environ.get("BIOHUB_SECONDARY_MIX_TEMPERATURE", "1")\n    )\n    secondary_low_margin_max = float(\n        os.environ.get("BIOHUB_SECONDARY_LOW_MARGIN_MAX", "0.2")\n    )\n    edge_candidate_threshold = float(\n        os.environ.get("BIOHUB_DUAL_SEED_EDGE_THRESHOLD", str(cfg.threshold))\n    )\n    if secondary_weights_text:\n        if not 0.0 < secondary_edge_weight < 1.0:\n            raise ValueError("BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1")\n        if not 0.0 <= secondary_detection_weight < 1.0:\n            raise ValueError(\n                "BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)"\n            )\n        if secondary_link_mode not in {\n            "raw", "calibrated", "adaptive", "low_margin_consensus"\n        }:\n            raise ValueError(\n                "BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, "\n                "or low_margin_consensus"\n            )\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\n            raise ValueError("BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]")\n        if not 0.0 < edge_candidate_threshold < 1.0:\n            raise ValueError("BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1")\n        if not 0.0 < secondary_low_margin_max <= 1.0:\n            raise ValueError("BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]")\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\n            Path(secondary_weights_text), device,\n        )\n        if secondary_window_size != window_size or secondary_downsample != downsample:\n            raise ValueError(\n                "Primary and secondary models have incompatible inference grids: "\n                f"primary=(window={window_size}, downsample={downsample}), "\n                f"secondary=(window={secondary_window_size}, downsample={secondary_downsample})"\n            )\n        cfg.threshold = edge_candidate_threshold\n        print(\n            f"Secondary model: {secondary_weights_text} | "\n            f"edge weight={secondary_edge_weight:.3f} | "\n            f"detection weight={secondary_detection_weight:.3f} | "\n            f"link mode={secondary_link_mode} | "\n            f"temperature={secondary_mix_temperature:.3f} | "\n            f"low-margin max={secondary_low_margin_max:.3f} | "\n            f"edge threshold={cfg.threshold:.3f}",\n            flush=True,\n        )\n\n    print('),
    ('                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n            )', '                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n                secondary_model=secondary_model,\n                secondary_edge_weight=secondary_edge_weight,\n                secondary_detection_weight=secondary_detection_weight,\n                secondary_link_mode=secondary_link_mode,\n                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )'),
]
_s = _apply(_s, _ensemble_replacements, "dual-seed")


# Hidden-proven label-free frame-retention guard.
_retention_old = '''                    det_logits[f] = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )'''
_retention_new = '''                    blended_det = (
                        (1.0 - secondary_detection_weight) * primary_det
                        + secondary_detection_weight * secondary_det_aligned
                    )
                    primary_candidates = len(_detect_cells_pooled(
                        primary_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    blended_candidates = len(_detect_cells_pooled(
                        blended_det[0],
                        int(frame_indices[f]),
                        cfg.det_threshold,
                        pool_k,
                    ))
                    minimum_retention = float(os.environ.get(
                        "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION",
                        "0.90",
                    ))
                    candidate_retention = (
                        blended_candidates / primary_candidates
                        if primary_candidates
                        else 1.0
                    )
                    use_primary_detection = bool(
                        primary_candidates > 0
                        and candidate_retention < minimum_retention
                    )
                    det_logits[f] = (
                        primary_det if use_primary_detection else blended_det
                    )
                    if int(frame_indices[f]) not in seen_frames:
                        shard = os.environ.get(
                            "BIOHUB_GPU_SHARD", "single"
                        ).replace("/", "_")
                        guard_log = (
                            Path("/kaggle/working")
                            / f"retention_guard_{shard}.jsonl"
                        )
                        guard_record = {
                            "dataset": ds_path.stem,
                            "frame": int(frame_indices[f]),
                            "primary_candidates": int(primary_candidates),
                            "blended_candidates": int(blended_candidates),
                            "retention": float(candidate_retention),
                            "minimum_retention": float(minimum_retention),
                            "use_primary": bool(use_primary_detection),
                        }
                        with guard_log.open("a") as guard_handle:
                            guard_handle.write(
                                json.dumps(guard_record, sort_keys=True)
                                + "\\n"
                            )
                        if use_primary_detection:
                            print(
                                "BIOHUB_RETENTION_GUARD "
                                + json.dumps(guard_record, sort_keys=True),
                                flush=True,
                            )'''
_s = _apply(_s, [(_retention_old, _retention_new)], "frame-retention guard")

# 3. Bidirectional harmonic-probability association fusion.
_bi_old = '''            edge_logits_pair = model.predict_edges(
                unet_feat_src, unet_feat_tgt,
                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,
                p_pos_src, p_pos_tgt,
                p_mask_src, p_mask_tgt,
            )  # (1, n_src, n_tgt)

            if secondary_model is not None:
'''
_bi_new = '''            edge_logits_pair = model.predict_edges(
                unet_feat_src, unet_feat_tgt,
                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,
                p_pos_src, p_pos_tgt,
                p_mask_src, p_mask_tgt,
            )  # (1, n_src, n_tgt)

            _bidirectional_weight = float(
                os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")
            )
            if _bidirectional_weight > 0.0:
                reverse_logits_native = model.predict_edges(
                    unet_feat_tgt, unet_feat_src,
                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,
                    p_pos_tgt, p_pos_src,
                    p_mask_tgt, p_mask_src,
                )  # (1, n_tgt, n_src)
                reverse_logits_pair = reverse_logits_native.transpose(1, 2)

                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)
                forward_scale = edge_logits_pair.float().std(
                    dim=1, keepdim=True, unbiased=False
                ).clamp_min(1e-4)
                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)
                reverse_scale = reverse_logits_pair.float().std(
                    dim=1, keepdim=True, unbiased=False
                ).clamp_min(1e-4)
                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)
                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)
                reverse_aligned = (
                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio
                    + forward_center
                )
                # Biohub 145: require mutual forward/reverse support in probability space.
                # The harmonic mean penalizes a candidate when either temporal direction
                # assigns it very low probability, while calibration preserves the forward
                # logit scale used by the unchanged downstream candidate threshold and ILP.
                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)
                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)
                harmonic_prob = 1.0 / (
                    (1.0 - _bidirectional_weight) / forward_prob
                    + _bidirectional_weight / reverse_prob
                )
                harmonic_prob = harmonic_prob / harmonic_prob.sum(
                    dim=1, keepdim=True
                ).clamp_min(1e-8)
                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))
                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)
                harmonic_scale = harmonic_logits.std(
                    dim=1, keepdim=True, unbiased=False
                ).clamp_min(1e-4)
                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)
                edge_logits_pair = (
                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio
                    + forward_center
                ).to(reverse_aligned.dtype)
                del (
                    reverse_logits_native,
                    reverse_logits_pair,
                    reverse_aligned,
                    forward_prob,
                    reverse_prob,
                    harmonic_prob,
                    harmonic_logits,
                )

            if secondary_model is not None:
'''
_s = _apply(_s, [(_bi_old, _bi_new)], "bidirectional fusion")

# 4. Native Model-C division evidence emitted inside the same GPU pass.
_model_c_predictor_replacements = [('import sys\n', 'import sys\nimport time\n'), ('    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    secondary_low_margin_max: float = 0.2,\n    division_model: UNetNodeTransformer | None = None,\n    division_evidence_path: Path | None = None,\n    division_det_threshold: float = 0.99,\n    division_pool_um: float = 3.0,\n    division_radius_um: float = 20.0,\n    division_topk: int = 16,\n    division_min_probability: float = 0.01,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'), ('    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\n\n    # Running node registry', '    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\n    division_pool_k = pool_kernel_from_um(division_pool_um, voxel_size)\n    division_ds_arr_t = torch.from_numpy(ds_arr).to(device)\n    original_spacing = np.asarray(ds.scale, dtype=np.float32)\n    division_seen_frames: set[int] = set()\n    division_seen_pairs: set[tuple[int, int]] = set()\n    division_coord_by_t: dict[int, np.ndarray] = {}\n    division_offset: dict[int, tuple[int, int]] = {}\n    division_node_count = 0\n    division_evidence = {\n        key: [] for key in (\n            "source_id", "target_id", "probability",\n            "alternative_parent_probability", "is_target_winner",\n            "distance_um", "source_target_rank",\n        )\n    }\n\n    # Running node registry'), ('            del secondary_det_logits\n\n        del imgs', '            del secondary_det_logits\n\n        division_unet_out = None\n        division_det_logits = None\n        if division_model is not None:\n            division_unet_out, division_det_logits = division_model.encode(imgs)\n            # Frozen Model-C parity path: identity + x/y/xy flips.\n            division_nv = 1\n            for division_dims in [(-1,), (-2,), (-2, -1)]:\n                division_imgs_flip = imgs.flip(division_dims)\n                _, division_det_flip = division_model.encode(division_imgs_flip)\n                for division_frame in range(W):\n                    division_det_logits[division_frame] = (\n                        division_det_logits[division_frame]\n                        + division_det_flip[division_frame].flip(division_dims)\n                    )\n                del division_imgs_flip, division_det_flip\n                division_nv += 1\n            for division_frame in range(W):\n                division_det_logits[division_frame] = (\n                    division_det_logits[division_frame] / division_nv\n                )\n\n        del imgs'), ('                seen_frames.add(t)\n\n        coords_so_far = (', '                seen_frames.add(t)\n\n        if division_model is not None:\n            for division_f_idx, division_t in enumerate(frame_indices):\n                if division_t in division_seen_frames:\n                    continue\n                native = _detect_cells_pooled(\n                    division_det_logits[division_f_idx][0],\n                    division_t,\n                    division_det_threshold,\n                    division_pool_k,\n                ).astype(np.int32)\n                native[:, 1:] *= np.asarray(downsample, np.int32)\n                division_coord_by_t[division_t] = native\n                division_offset[division_t] = (\n                    division_node_count,\n                    division_node_count + len(native),\n                )\n                division_node_count += len(native)\n                division_seen_frames.add(division_t)\n\n        coords_so_far = ('), ('            seen_pairs.add((t_src, t_tgt))\n\n            if t_src not in coord_offset or t_tgt not in coord_offset:', '            seen_pairs.add((t_src, t_tgt))\n\n            if (\n                division_model is not None\n                and (t_src, t_tgt) not in division_seen_pairs\n            ):\n                division_seen_pairs.add((t_src, t_tgt))\n                native_src = division_coord_by_t.get(\n                    t_src, np.empty((0, 4), np.int32)\n                )\n                native_tgt = division_coord_by_t.get(\n                    t_tgt, np.empty((0, 4), np.int32)\n                )\n                if len(native_src) and len(native_tgt):\n                    source_raw = native_src[:, 1:].astype(np.float32)\n                    target_raw = native_tgt[:, 1:].astype(np.float32)\n                    source_ds = source_raw / ds_arr\n                    target_ds = target_raw / ds_arr\n                    n_div_source, n_div_target = len(source_ds), len(target_ds)\n                    div_src = torch.from_numpy(source_ds).unsqueeze(0).to(device)\n                    div_tgt = torch.from_numpy(target_ds).unsqueeze(0).to(device)\n                    source_rel = np.column_stack([\n                        np.full(n_div_source, f_idx, np.float32), source_ds\n                    ])\n                    target_rel = np.column_stack([\n                        np.full(n_div_target, f_idx + 1, np.float32), target_ds\n                    ])\n                    div_window_shape = (W,) + image_shape[1:]\n                    div_pos_src = torch.from_numpy(\n                        extract_pos_features(source_rel, div_window_shape)\n                    ).unsqueeze(0).to(device)\n                    div_pos_tgt = torch.from_numpy(\n                        extract_pos_features(target_rel, div_window_shape)\n                    ).unsqueeze(0).to(device)\n                    div_mask_src = torch.ones(\n                        1, n_div_source, dtype=torch.bool, device=device\n                    )\n                    div_mask_tgt = torch.ones(\n                        1, n_div_target, dtype=torch.bool, device=device\n                    )\n                    div_feat_src = division_model._index_features(\n                        division_unet_out[:, f_idx], div_src, div_mask_src\n                    )\n                    div_feat_tgt = division_model._index_features(\n                        division_unet_out[:, f_idx + 1], div_tgt, div_mask_tgt\n                    )\n                    div_logits = division_model.predict_edges(\n                        div_feat_src,\n                        div_feat_tgt,\n                        div_src * division_ds_arr_t,\n                        div_tgt * division_ds_arr_t,\n                        div_pos_src,\n                        div_pos_tgt,\n                        div_mask_src,\n                        div_mask_tgt,\n                    )[0]\n                    div_probability = torch.softmax(div_logits, dim=0).float()\n                    div_source_um = torch.as_tensor(\n                        source_raw * original_spacing, device=device\n                    ).float()\n                    div_target_um = torch.as_tensor(\n                        target_raw * original_spacing, device=device\n                    ).float()\n                    div_distance = torch.cdist(div_source_um, div_target_um)\n                    div_eligible = div_distance <= division_radius_um\n                    div_masked = div_probability.masked_fill(\n                        ~div_eligible, -1.0\n                    )\n                    div_k = min(division_topk, n_div_target)\n                    div_values, div_targets = torch.topk(\n                        div_masked, div_k, dim=1, sorted=True\n                    )\n                    div_keep = div_values >= division_min_probability\n                    if n_div_source == 1:\n                        div_best_prob = div_probability[0]\n                        div_best_row = torch.zeros_like(\n                            div_best_prob, dtype=torch.long\n                        )\n                        div_second_prob = torch.zeros_like(div_best_prob)\n                    else:\n                        div_top2 = torch.topk(\n                            div_probability, 2, dim=0, sorted=True\n                        )\n                        div_best_prob = div_top2.values[0]\n                        div_best_row = div_top2.indices[0]\n                        div_second_prob = div_top2.values[1]\n                    div_source_rows = torch.arange(\n                        n_div_source, device=device\n                    ).unsqueeze(1).expand_as(div_targets)\n                    picked_source = div_source_rows[div_keep]\n                    picked_target = div_targets[div_keep]\n                    target_best = div_best_row[picked_target]\n                    div_alternative = torch.where(\n                        target_best == picked_source,\n                        div_second_prob[picked_target],\n                        div_best_prob[picked_target],\n                    )\n                    source_start = division_offset[t_src][0]\n                    target_start = division_offset[t_tgt][0]\n                    division_evidence["source_id"].append(\n                        (picked_source + source_start).cpu().numpy().astype(\n                            np.int64\n                        )\n                    )\n                    division_evidence["target_id"].append(\n                        (picked_target + target_start).cpu().numpy().astype(\n                            np.int64\n                        )\n                    )\n                    division_evidence["probability"].append(\n                        div_values[div_keep].cpu().numpy().astype(np.float32)\n                    )\n                    division_evidence[\n                        "alternative_parent_probability"\n                    ].append(\n                        div_alternative.cpu().numpy().astype(np.float32)\n                    )\n                    division_evidence["is_target_winner"].append(\n                        (target_best == picked_source).cpu().numpy().astype(\n                            np.uint8\n                        )\n                    )\n                    division_evidence["distance_um"].append(\n                        div_distance[\n                            picked_source, picked_target\n                        ].cpu().numpy().astype(np.float32)\n                    )\n                    div_rank_grid = torch.arange(\n                        div_k, device=device\n                    ).unsqueeze(0).expand_as(div_targets)\n                    division_evidence["source_target_rank"].append(\n                        div_rank_grid[div_keep].cpu().numpy().astype(np.int16)\n                    )\n\n            if t_src not in coord_offset or t_tgt not in coord_offset:'), ('        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n        if division_unet_out is not None:\n            del division_unet_out\n        if division_det_logits is not None:\n            del division_det_logits\n'), ('    coords = coords.astype(np.int16)\n    return coords, all_edges\n', '    coords = coords.astype(np.int16)\n    if division_model is not None and division_evidence_path is not None:\n        native_counts = np.asarray(\n            [\n                len(division_coord_by_t.get(\n                    frame, np.empty((0, 4), np.int32)\n                ))\n                for frame in range(T)\n            ],\n            np.int32,\n        )\n        native_offsets = np.concatenate([\n            [0], np.cumsum(native_counts, dtype=np.int64)\n        ])\n        native_chunks = [\n            division_coord_by_t.get(frame, np.empty((0, 4), np.int32))\n            for frame in range(T)\n        ]\n        native_coords = (\n            np.concatenate(native_chunks).astype(np.int16)\n            if native_chunks\n            else np.empty((0, 4), np.int16)\n        )\n        evidence_dtypes = {\n            "source_id": np.int64,\n            "target_id": np.int64,\n            "probability": np.float32,\n            "alternative_parent_probability": np.float32,\n            "is_target_winner": np.uint8,\n            "distance_um": np.float32,\n            "source_target_rank": np.int16,\n        }\n        evidence_arrays = {}\n        for evidence_key, evidence_chunks in division_evidence.items():\n            evidence_arrays[evidence_key] = (\n                np.concatenate(evidence_chunks).astype(\n                    evidence_dtypes[evidence_key], copy=False\n                )\n                if evidence_chunks\n                else np.empty(0, evidence_dtypes[evidence_key])\n            )\n        division_evidence_path.parent.mkdir(parents=True, exist_ok=True)\n        evidence_tmp = division_evidence_path.with_suffix(".npz.tmp")\n        with evidence_tmp.open("wb") as evidence_handle:\n            np.savez_compressed(\n                evidence_handle,\n                **evidence_arrays,\n                native_node_coords=native_coords,\n                native_frame_offsets=native_offsets,\n                spacing_um=original_spacing.astype(np.float32),\n            )\n        evidence_tmp.replace(division_evidence_path)\n    return coords, all_edges\n'), ('    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None', '    model, window_size, downsample = load_model(weights_path, device)\n\n    division_model = None\n    division_evidence_dir = None\n    division_weights_text = os.environ.get(\n        "BIOHUB_MODEL_C_CHECKPOINT", ""\n    ).strip()\n    division_evidence_text = os.environ.get(\n        "BIOHUB_MODEL_C_EVIDENCE_DIR", ""\n    ).strip()\n    division_helper_deadline_epoch = float(\n        os.environ.get("BIOHUB_HELPER_DEADLINE_EPOCH", "inf")\n    )\n    if division_weights_text:\n        division_model, division_window_size, division_downsample = load_model(\n            Path(division_weights_text), device\n        )\n        if division_window_size != window_size:\n            raise ValueError(\n                f"Model C window {division_window_size} != public {window_size}"\n            )\n        if division_downsample != downsample:\n            raise ValueError(\n                f"Model C downsample {division_downsample} != public {downsample}"\n            )\n        if not division_evidence_text:\n            raise ValueError("BIOHUB_MODEL_C_EVIDENCE_DIR is required")\n        division_evidence_dir = Path(division_evidence_text)\n        division_evidence_dir.mkdir(parents=True, exist_ok=True)\n        print(\n            f"Integrated native Model C: {division_weights_text} | "\n            f"evidence={division_evidence_dir}",\n            flush=True,\n        )\n\n    secondary_model = None'), ('    for name in tqdm(test_names, desc="Predicting", disable=not INTERACTIVE):\n        ds_path = data_dir / name\n        coords, edges = predict_video(', '    for name in tqdm(test_names, desc="Predicting", disable=not INTERACTIVE):\n        ds_path = data_dir / name\n        use_division_model = (\n            division_model is not None\n            and time.time() < division_helper_deadline_epoch\n        )\n        if division_model is not None and not use_division_model:\n            print(\n                f"Helper deadline reached; {name} keeps public baseline only",\n                flush=True,\n            )\n        coords, edges = predict_video('), ('                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )', '                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n                division_model=(\n                    division_model if use_division_model else None\n                ),\n                division_evidence_path=(\n                    division_evidence_dir / f"{name}.npz"\n                    if use_division_model else None\n                ),\n                division_det_threshold=0.99,\n                division_pool_um=3.0,\n                division_radius_um=20.0,\n                division_topk=16,\n                division_min_probability=0.01,\n            )'), ('    output_dir.mkdir(parents=True, exist_ok=True)\n\n    device = torch.device', '    output_dir.mkdir(parents=True, exist_ok=True)\n    for stale_marker in output_dir.glob("*.ready*"):\n        stale_marker.unlink()\n\n    device = torch.device')]
_s = _apply(_s, _model_c_predictor_replacements, "Model-C evidence")

# 5. Atomic per-video ready marker, written only after save_graph() returns.
_ready_old = '        save_graph(graph, output_dir / f"{name}.geff")'
_ready_new = '''        output_path = output_dir / f"{name}.geff"
        save_graph(graph, output_path)
        ready_tmp = output_dir / f"{name}.ready.tmp"
        ready_tmp.write_text("complete\\n")
        ready_tmp.replace(output_dir / f"{name}.ready")'''
_s = _apply(_s, [(_ready_old, _ready_new)], "ready marker")

# 6. In-pass P1/P2 source-cardinality evidence (no second GPU sweep).
_inpass_native_evidence_replacements = [('@torch.no_grad()\ndef predict_video(\n', '\n\ndef _new_native_evidence_state():\n    return {\n        "coords_by_t": {},\n        "offsets": {},\n        "node_count": 0,\n        "seen_frames": set(),\n        "seen_pairs": set(),\n        "evidence": {\n            key: [] for key in (\n                "source_id", "target_id", "probability",\n                "alternative_parent_probability", "is_target_winner",\n                "distance_um", "source_target_rank",\n            )\n        },\n    }\n\n\ndef _detect_native_evidence_frames(\n    state,\n    detection,\n    frame_indices,\n    threshold,\n    pool_kernel,\n    downsample,\n):\n    scale = np.asarray(downsample, np.int32)\n    for local_frame, absolute_frame in enumerate(frame_indices):\n        if absolute_frame in state["seen_frames"]:\n            continue\n        coord = _detect_cells_pooled(\n            detection[local_frame][0],\n            absolute_frame,\n            threshold,\n            pool_kernel,\n        ).astype(np.int32)\n        coord[:, 1:] *= scale\n        start = int(state["node_count"])\n        state["coords_by_t"][absolute_frame] = coord\n        state["offsets"][absolute_frame] = (start, start + len(coord))\n        state["node_count"] = start + len(coord)\n        state["seen_frames"].add(absolute_frame)\n\n\ndef _append_native_evidence_pair(\n    state,\n    model,\n    feature_maps,\n    f_idx,\n    t_src,\n    t_tgt,\n    downsample,\n    spacing,\n    window_shape,\n    device,\n    radius_um,\n    topk_per_source,\n    min_probability,\n):\n    if (t_src, t_tgt) in state["seen_pairs"]:\n        return\n    state["seen_pairs"].add((t_src, t_tgt))\n    source_native = state["coords_by_t"].get(t_src)\n    target_native = state["coords_by_t"].get(t_tgt)\n    if source_native is None or target_native is None:\n        return\n    if not len(source_native) or not len(target_native):\n        return\n    downsample_array = np.asarray(downsample, np.float32)\n    source_raw = source_native[:, 1:].astype(np.float32)\n    target_raw = target_native[:, 1:].astype(np.float32)\n    source_downsampled = source_raw / downsample_array\n    target_downsampled = target_raw / downsample_array\n    n_source, n_target = len(source_downsampled), len(target_downsampled)\n    source_tensor = torch.from_numpy(source_downsampled).unsqueeze(0).to(device)\n    target_tensor = torch.from_numpy(target_downsampled).unsqueeze(0).to(device)\n    source_relative = np.column_stack([\n        np.full(n_source, f_idx, np.float32), source_downsampled,\n    ])\n    target_relative = np.column_stack([\n        np.full(n_target, f_idx + 1, np.float32), target_downsampled,\n    ])\n    source_position = torch.from_numpy(\n        extract_pos_features(source_relative, window_shape).astype(np.float32)\n    ).unsqueeze(0).to(device)\n    target_position = torch.from_numpy(\n        extract_pos_features(target_relative, window_shape).astype(np.float32)\n    ).unsqueeze(0).to(device)\n    source_mask = torch.ones(1, n_source, dtype=torch.bool, device=device)\n    target_mask = torch.ones(1, n_target, dtype=torch.bool, device=device)\n    source_features = model._index_features(\n        feature_maps[:, f_idx], source_tensor, source_mask,\n    )\n    target_features = model._index_features(\n        feature_maps[:, f_idx + 1], target_tensor, target_mask,\n    )\n    downsample_tensor = torch.as_tensor(\n        downsample_array, dtype=torch.float32, device=device,\n    )\n    logits = model.predict_edges(\n        source_features,\n        target_features,\n        source_tensor * downsample_tensor,\n        target_tensor * downsample_tensor,\n        source_position,\n        target_position,\n        source_mask,\n        target_mask,\n    )[0]\n    probability = torch.softmax(logits, dim=0).float()\n    spacing_array = np.asarray(spacing, np.float32)\n    source_um = torch.as_tensor(\n        source_raw * spacing_array, dtype=torch.float32, device=device,\n    )\n    target_um = torch.as_tensor(\n        target_raw * spacing_array, dtype=torch.float32, device=device,\n    )\n    distance = torch.cdist(source_um, target_um)\n    eligible = distance <= radius_um\n    masked = probability.masked_fill(~eligible, -1.0)\n    k = min(topk_per_source, n_target)\n    values, targets = torch.topk(masked, k, dim=1, sorted=True)\n    keep = values >= min_probability\n    if n_source == 1:\n        best_probability = probability[0]\n        best_parent = torch.zeros_like(best_probability, dtype=torch.long)\n        second_probability = torch.zeros_like(best_probability)\n    else:\n        top2 = torch.topk(probability, 2, dim=0, sorted=True)\n        best_probability = top2.values[0]\n        best_parent = top2.indices[0]\n        second_probability = top2.values[1]\n    source_rows = torch.arange(\n        n_source, device=device,\n    ).unsqueeze(1).expand_as(targets)\n    picked_source = source_rows[keep]\n    picked_target = targets[keep]\n    target_best_parent = best_parent[picked_target]\n    alternative = torch.where(\n        target_best_parent == picked_source,\n        second_probability[picked_target],\n        best_probability[picked_target],\n    )\n    source_start = int(state["offsets"][t_src][0])\n    target_start = int(state["offsets"][t_tgt][0])\n    evidence = state["evidence"]\n    evidence["source_id"].append(\n        (picked_source + source_start).cpu().numpy().astype(np.int64)\n    )\n    evidence["target_id"].append(\n        (picked_target + target_start).cpu().numpy().astype(np.int64)\n    )\n    evidence["probability"].append(\n        values[keep].cpu().numpy().astype(np.float32)\n    )\n    evidence["alternative_parent_probability"].append(\n        alternative.cpu().numpy().astype(np.float32)\n    )\n    evidence["is_target_winner"].append(\n        (target_best_parent == picked_source).cpu().numpy().astype(np.uint8)\n    )\n    evidence["distance_um"].append(\n        distance[picked_source, picked_target].cpu().numpy().astype(np.float32)\n    )\n    rank_grid = torch.arange(\n        k, device=device,\n    ).unsqueeze(0).expand_as(targets)\n    evidence["source_target_rank"].append(\n        rank_grid[keep].cpu().numpy().astype(np.int16)\n    )\n\n\ndef _save_native_evidence(state, output_path, frames, spacing):\n    if output_path is None:\n        return\n    empty_coords = np.empty((0, 4), np.int32)\n    chunks = [\n        state["coords_by_t"].get(frame, empty_coords)\n        for frame in range(frames)\n    ]\n    counts = np.asarray([len(chunk) for chunk in chunks], np.int32)\n    offsets = np.concatenate([[0], np.cumsum(counts, dtype=np.int64)])\n    native_coords = (\n        np.concatenate(chunks).astype(np.int16, copy=False)\n        if chunks else np.empty((0, 4), np.int16)\n    )\n    dtypes = {\n        "source_id": np.int64,\n        "target_id": np.int64,\n        "probability": np.float32,\n        "alternative_parent_probability": np.float32,\n        "is_target_winner": np.uint8,\n        "distance_um": np.float32,\n        "source_target_rank": np.int16,\n    }\n    arrays = {}\n    for key, evidence_chunks in state["evidence"].items():\n        arrays[key] = (\n            np.concatenate(evidence_chunks).astype(dtypes[key], copy=False)\n            if evidence_chunks else np.empty(0, dtypes[key])\n        )\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_suffix(".npz.tmp")\n    with temporary.open("wb") as handle:\n        np.savez_compressed(\n            handle,\n            **arrays,\n            native_node_coords=native_coords,\n            native_frame_offsets=offsets,\n            spacing_um=np.asarray(spacing, np.float32),\n        )\n    temporary.replace(output_path)\n\n\n@torch.no_grad()\ndef predict_video(\n'), ('    division_min_probability: float = 0.01,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    division_min_probability: float = 0.01,\n    primary_native_evidence_path: Path | None = None,\n    secondary_native_evidence_path: Path | None = None,\n    native_evidence_det_threshold: float = 0.96875,\n    native_evidence_pool_um: float = 5.0,\n    native_evidence_radius_um: float = 20.0,\n    native_evidence_topk: int = 16,\n    native_evidence_min_probability: float = 0.01,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'), ('    division_evidence = {\n        key: [] for key in (\n            "source_id", "target_id", "probability",\n            "alternative_parent_probability", "is_target_winner",\n            "distance_um", "source_target_rank",\n        )\n    }\n\n    # Running node registry', '    division_evidence = {\n        key: [] for key in (\n            "source_id", "target_id", "probability",\n            "alternative_parent_probability", "is_target_winner",\n            "distance_um", "source_target_rank",\n        )\n    }\n    native_evidence_pool_kernel = pool_kernel_from_um(\n        native_evidence_pool_um, voxel_size,\n    )\n    primary_native_state = (\n        _new_native_evidence_state()\n        if primary_native_evidence_path is not None else None\n    )\n    secondary_native_state = (\n        _new_native_evidence_state()\n        if secondary_native_evidence_path is not None else None\n    )\n\n    # Running node registry'), ('        if cfg.det_tta:\n            _nv = 1', '        primary_native_detection = None\n        if cfg.det_tta:\n            _nv = 1'), ('                del imgs_flip, det_flip\n                _nv += 1\n            for _k in (1, 3):', '                del imgs_flip, det_flip\n                _nv += 1\n            primary_native_detection = [value / _nv for value in det_logits]\n            for _k in (1, 3):'), ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n        if primary_native_detection is None:\n            primary_native_detection = [value.clone() for value in det_logits]\n\n        secondary_unet_out = None'), ('        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:', '        secondary_native_detection = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:'), ('                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):', '                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    secondary_native_detection = [\n                        value / _secondary_nv for value in secondary_det_logits\n                    ]\n                    for _k in (1, 3):'), ('                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):', '                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n                if secondary_native_detection is None:\n                    secondary_native_detection = [\n                        value.clone() for value in secondary_det_logits\n                    ]\n\n                for f in range(W):'), ('        del imgs\n\n        # --- Detect cells in each frame (dedup across windows) ---', '        del imgs\n\n        if primary_native_state is not None:\n            _detect_native_evidence_frames(\n                primary_native_state,\n                primary_native_detection,\n                frame_indices,\n                native_evidence_det_threshold,\n                native_evidence_pool_kernel,\n                downsample,\n            )\n        if secondary_native_state is not None:\n            if secondary_native_detection is None:\n                raise RuntimeError("Secondary native evidence requested without model")\n            _detect_native_evidence_frames(\n                secondary_native_state,\n                secondary_native_detection,\n                frame_indices,\n                native_evidence_det_threshold,\n                native_evidence_pool_kernel,\n                downsample,\n            )\n\n        # --- Detect cells in each frame (dedup across windows) ---'), ('            seen_pairs.add((t_src, t_tgt))\n\n            if (\n                division_model is not None', '            seen_pairs.add((t_src, t_tgt))\n\n            native_window_shape = (W,) + image_shape[1:]\n            if primary_native_state is not None:\n                _append_native_evidence_pair(\n                    primary_native_state, model, unet_out,\n                    f_idx, t_src, t_tgt, downsample, ds.scale,\n                    native_window_shape, device,\n                    native_evidence_radius_um,\n                    native_evidence_topk,\n                    native_evidence_min_probability,\n                )\n            if secondary_native_state is not None:\n                if secondary_model is None or secondary_unet_out is None:\n                    raise RuntimeError("Secondary native evidence feature map missing")\n                _append_native_evidence_pair(\n                    secondary_native_state, secondary_model, secondary_unet_out,\n                    f_idx, t_src, t_tgt, downsample, ds.scale,\n                    native_window_shape, device,\n                    native_evidence_radius_um,\n                    native_evidence_topk,\n                    native_evidence_min_probability,\n                )\n\n            if (\n                division_model is not None'), ('        if division_det_logits is not None:\n            del division_det_logits\n\n    coords = np.concatenate(coord_lists)', '        if division_det_logits is not None:\n            del division_det_logits\n        if primary_native_detection is not None:\n            del primary_native_detection\n        if secondary_native_detection is not None:\n            del secondary_native_detection\n\n    coords = np.concatenate(coord_lists)'), ('        evidence_tmp.replace(division_evidence_path)\n    return coords, all_edges', '        evidence_tmp.replace(division_evidence_path)\n    _save_native_evidence(\n        primary_native_state,\n        primary_native_evidence_path,\n        T,\n        ds.scale,\n    ) if primary_native_state is not None else None\n    _save_native_evidence(\n        secondary_native_state,\n        secondary_native_evidence_path,\n        T,\n        ds.scale,\n    ) if secondary_native_state is not None else None\n    return coords, all_edges'), ('    division_model = None\n    division_evidence_dir = None', '    primary_native_evidence_dir = None\n    secondary_native_evidence_dir = None\n    primary_native_evidence_text = os.environ.get(\n        "BIOHUB_PUBLIC_P1_EVIDENCE_DIR", ""\n    ).strip()\n    secondary_native_evidence_text = os.environ.get(\n        "BIOHUB_PUBLIC_P2_EVIDENCE_DIR", ""\n    ).strip()\n    if primary_native_evidence_text:\n        primary_native_evidence_dir = Path(primary_native_evidence_text)\n        primary_native_evidence_dir.mkdir(parents=True, exist_ok=True)\n    if secondary_native_evidence_text:\n        secondary_native_evidence_dir = Path(secondary_native_evidence_text)\n        secondary_native_evidence_dir.mkdir(parents=True, exist_ok=True)\n\n    division_model = None\n    division_evidence_dir = None'), ('                division_min_probability=0.01,\n            )', '                division_min_probability=0.01,\n                primary_native_evidence_path=(\n                    primary_native_evidence_dir / f"{name}.npz"\n                    if use_division_model and primary_native_evidence_dir is not None\n                    else None\n                ),\n                secondary_native_evidence_path=(\n                    secondary_native_evidence_dir / f"{name}.npz"\n                    if use_division_model and secondary_native_evidence_dir is not None\n                    else None\n                ),\n                native_evidence_det_threshold=0.96875,\n                native_evidence_pool_um=5.0,\n                native_evidence_radius_um=20.0,\n                native_evidence_topk=16,\n                native_evidence_min_probability=0.01,\n            )')]
_s = _apply(_s, _inpass_native_evidence_replacements, "in-pass P1/P2 evidence")


# 7. Sub-voxel detection geometry.
#
# Peaks come off a (1, 4, 4) downsampled grid, so every y/x centroid is
# snapped to a 4-voxel lattice (1.625 um) before it is written out. A
# parabolic fit on the detection logits recovers the fractional peak, and the
# offset is clamped to +-0.49 voxel so `round()` still returns the original
# lattice cell: every tensor the networks see is bit-identical to the frozen
# pipeline, only the emitted coordinates gain sub-voxel precision.
_subvoxel_replacements = [
    (
        '''def _detect_cells_pooled(''',
        '''def _subvoxel_offsets(
    logits: torch.Tensor,
    peak_idx: torch.Tensor,
    max_shift: float = 0.49,
) -> torch.Tensor:
    """Parabolic peak offsets, one per axis, in downsampled voxels."""
    offsets = torch.zeros(peak_idx.shape, dtype=torch.float32, device=logits.device)
    for axis in range(3):
        size = logits.shape[axis]
        if size < 3:
            continue
        interior = (peak_idx[:, axis] > 0) & (peak_idx[:, axis] < size - 1)
        if not bool(interior.any()):
            continue
        rows = torch.nonzero(interior, as_tuple=False)[:, 0]
        centre = peak_idx[rows]
        lower = centre.clone()
        upper = centre.clone()
        lower[:, axis] = centre[:, axis] - 1
        upper[:, axis] = centre[:, axis] + 1
        value_c = logits[centre[:, 0], centre[:, 1], centre[:, 2]].float()
        value_l = logits[lower[:, 0], lower[:, 1], lower[:, 2]].float()
        value_r = logits[upper[:, 0], upper[:, 1], upper[:, 2]].float()
        denominator = value_l - 2.0 * value_c + value_r
        shift = torch.where(
            denominator.abs() > 1e-6,
            0.5 * (value_l - value_r) / denominator,
            torch.zeros_like(denominator),
        )
        offsets[rows, axis] = shift.clamp(-max_shift, max_shift)
    return offsets


def _detect_cells_pooled(''',
    ),
    (
        '''    if peak_idx.shape[0] == 0:
        return np.empty((0, 4), dtype=np.int16)

    coords = peak_idx.float().cpu().numpy()
    t_col = np.full((len(coords), 1), t, dtype=np.float32)
    return np.concatenate([t_col, coords], axis=1).astype(np.int16)''',
        '''    if peak_idx.shape[0] == 0:
        return np.empty((0, 4), dtype=np.float32)

    refined = peak_idx.float() + _subvoxel_offsets(det_logits[0], peak_idx)
    coords = refined.cpu().numpy()
    t_col = np.full((len(coords), 1), t, dtype=np.float32)
    return np.concatenate([t_col, coords], axis=1).astype(np.float32)''',
    ),
    (
        '''        coords_so_far = (
            np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), dtype=np.int16)
        )''',
        '''        coords_so_far = (
            np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), dtype=np.float32)
        )''',
    ),
    (
        '''            p_coords_src = torch.from_numpy(c_src[:, 1:].astype(np.float32)).unsqueeze(0).to(device)
            p_coords_tgt = torch.from_numpy(c_tgt[:, 1:].astype(np.float32)).unsqueeze(0).to(device)''',
        '''            p_coords_src = torch.from_numpy(
                np.rint(c_src[:, 1:]).astype(np.float32)
            ).unsqueeze(0).to(device)
            p_coords_tgt = torch.from_numpy(
                np.rint(c_tgt[:, 1:]).astype(np.float32)
            ).unsqueeze(0).to(device)''',
    ),
    (
        '''            c_src_rel = c_src.copy()
            c_src_rel[:, 0] = f_idx
            c_tgt_rel = c_tgt.copy()
            c_tgt_rel[:, 0] = f_idx + 1''',
        '''            c_src_rel = np.rint(c_src).astype(np.float32)
            c_src_rel[:, 0] = f_idx
            c_tgt_rel = np.rint(c_tgt).astype(np.float32)
            c_tgt_rel[:, 0] = f_idx + 1''',
    ),
    (
        '''    coords = np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), dtype=np.int16)
    # Scale spatial coords back to original resolution.
    coords = coords.astype(np.float32)
    coords[:, 1:] *= ds_arr
    coords = coords.astype(np.int16)''',
        '''    coords = np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), dtype=np.float32)
    # Scale spatial coords back to original resolution, sub-voxel part kept.
    coords = coords.astype(np.float32)
    coords[:, 1:] *= ds_arr''',
    ),
    # Native Model-C evidence: fractional centroids, rounded model inputs.
    (
        '''                native = _detect_cells_pooled(
                    division_det_logits[division_f_idx][0],
                    division_t,
                    division_det_threshold,
                    division_pool_k,
                ).astype(np.int32)
                native[:, 1:] *= np.asarray(downsample, np.int32)''',
        '''                native = _detect_cells_pooled(
                    division_det_logits[division_f_idx][0],
                    division_t,
                    division_det_threshold,
                    division_pool_k,
                ).astype(np.float32)
                native[:, 1:] *= np.asarray(downsample, np.float32)''',
    ),
    (
        '''                native_src = division_coord_by_t.get(
                    t_src, np.empty((0, 4), np.int32)
                )
                native_tgt = division_coord_by_t.get(
                    t_tgt, np.empty((0, 4), np.int32)
                )''',
        '''                native_src = division_coord_by_t.get(
                    t_src, np.empty((0, 4), np.float32)
                )
                native_tgt = division_coord_by_t.get(
                    t_tgt, np.empty((0, 4), np.float32)
                )''',
    ),
    (
        '''                    source_ds = source_raw / ds_arr
                    target_ds = target_raw / ds_arr''',
        '''                    source_ds = np.rint(source_raw / ds_arr).astype(np.float32)
                    target_ds = np.rint(target_raw / ds_arr).astype(np.float32)''',
    ),
    (
        '''                len(division_coord_by_t.get(
                    frame, np.empty((0, 4), np.int32)
                ))''',
        '''                len(division_coord_by_t.get(
                    frame, np.empty((0, 4), np.float32)
                ))''',
    ),
    (
        '''        native_chunks = [
            division_coord_by_t.get(frame, np.empty((0, 4), np.int32))
            for frame in range(T)
        ]
        native_coords = (
            np.concatenate(native_chunks).astype(np.int16)
            if native_chunks
            else np.empty((0, 4), np.int16)
        )''',
        '''        native_chunks = [
            division_coord_by_t.get(frame, np.empty((0, 4), np.float32))
            for frame in range(T)
        ]
        native_coords = (
            np.concatenate(native_chunks).astype(np.float32)
            if native_chunks
            else np.empty((0, 4), np.float32)
        )''',
    ),
    # In-pass P1/P2 evidence: same treatment.
    (
        '''    scale = np.asarray(downsample, np.int32)''',
        '''    scale = np.asarray(downsample, np.float32)''',
    ),
    (
        '''        coord = _detect_cells_pooled(
            detection[local_frame][0],
            absolute_frame,
            threshold,
            pool_kernel,
        ).astype(np.int32)''',
        '''        coord = _detect_cells_pooled(
            detection[local_frame][0],
            absolute_frame,
            threshold,
            pool_kernel,
        ).astype(np.float32)''',
    ),
    (
        '''    source_downsampled = source_raw / downsample_array
    target_downsampled = target_raw / downsample_array''',
        '''    source_downsampled = np.rint(source_raw / downsample_array).astype(np.float32)
    target_downsampled = np.rint(target_raw / downsample_array).astype(np.float32)''',
    ),
    (
        '''    empty_coords = np.empty((0, 4), np.int32)''',
        '''    empty_coords = np.empty((0, 4), np.float32)''',
    ),
    (
        '''    native_coords = (
        np.concatenate(chunks).astype(np.int16, copy=False)
        if chunks else np.empty((0, 4), np.int16)
    )''',
        '''    native_coords = (
        np.concatenate(chunks).astype(np.float32, copy=False)
        if chunks else np.empty((0, 4), np.float32)
    )''',
    ),
]
if os.environ.get("BIOHUB_SUBVOXEL_DETECTION", "1") == "1":
    _s = _apply(_s, _subvoxel_replacements, "sub-voxel detection")
    print("sub-voxel detection geometry: enabled")
else:
    print("sub-voxel detection geometry: disabled by CFG")



# 8. Guard-aware continuation-edge ownership.
#
# The hidden-proven retention guard already chooses P1 detections for frames
# where the P1+P2 blend retains less than 90% of P1's candidates.  Preserve
# that ownership during low-margin continuation fusion: P2 remains capped at
# 0.15 normally and at 0.00 only for a transition touching a guarded frame.
_guard_state_old = '''    seen_frames: set[int] = set()
    seen_pairs: set[tuple[int, int]] = set()'''
_guard_state_new = '''    seen_frames: set[int] = set()
    seen_pairs: set[tuple[int, int]] = set()
    retention_guard_frames: set[int] = set()'''

_guard_record_old = '''                        if use_primary_detection:
                            print('''
_guard_record_new = '''                        if use_primary_detection:
                            retention_guard_frames.add(int(frame_indices[f]))
                            print('''

_guard_weight_old = '''                if secondary_link_mode == "raw":'''
_guard_weight_new = '''                transition_uses_primary_detection = bool(
                    t_src in retention_guard_frames
                    or t_tgt in retention_guard_frames
                )
                guarded_secondary_edge_weight = float(os.environ.get(
                    "BIOHUB_RETENTION_GUARD_SECONDARY_EDGE_WEIGHT",
                    str(secondary_edge_weight),
                ))
                active_secondary_edge_weight = (
                    guarded_secondary_edge_weight
                    if transition_uses_primary_detection
                    else secondary_edge_weight
                )
                if transition_uses_primary_detection:
                    print(
                        "BIOHUB_GUARD_EDGE_OWNERSHIP "
                        + json.dumps({
                            "dataset": ds_path.stem,
                            "source_frame": int(t_src),
                            "target_frame": int(t_tgt),
                            "ordinary_p2_ceiling": float(secondary_edge_weight),
                            "active_p2_ceiling": float(active_secondary_edge_weight),
                        }, sort_keys=True),
                        flush=True,
                    )

                if secondary_link_mode == "raw":'''

_guard_low_margin_old = (
    "                            local_weight = secondary_edge_weight * uncertainty"
)
_guard_low_margin_new = (
    "                            local_weight = active_secondary_edge_weight * uncertainty"
)

_s = _apply(
    _s,
    [
        (_guard_state_old, _guard_state_new),
        (_guard_record_old, _guard_record_new),
        (_guard_weight_old, _guard_weight_new),
        (_guard_low_margin_old, _guard_low_margin_new),
    ],
    "guard-aware edge ownership",
)

# 9. EdgeGRAFT exact native-to-fused identity export.
_edgegraft_mapping_replacements = [
    (
        '''import zarr
from tqdm import tqdm''',
        '''import zarr
from scipy.spatial import cKDTree
from tqdm import tqdm''',
    ),
    (
        '''def _save_native_evidence(state, output_path, frames, spacing):''',
        '''def _edgegraft_greedy_native_map(
    native_coords, native_offsets, fused_coords, fused_offsets,
    spacing, max_um=6.0,
):
    mapped = np.full(len(native_coords), -1, np.int64)
    distance = np.full(len(native_coords), np.inf, np.float32)
    frame_count = min(len(native_offsets), len(fused_offsets)) - 1
    for frame in range(frame_count):
        n0, n1 = int(native_offsets[frame]), int(native_offsets[frame + 1])
        f0, f1 = int(fused_offsets[frame]), int(fused_offsets[frame + 1])
        if n1 <= n0 or f1 <= f0:
            continue
        native_um = native_coords[n0:n1, 1:].astype(np.float32) * spacing
        fused_um = fused_coords[f0:f1, 1:].astype(np.float32) * spacing
        k = min(4, len(fused_um))
        query_distance, query_index = cKDTree(fused_um).query(native_um, k=k)
        if k == 1:
            query_distance = query_distance[:, None]
            query_index = query_index[:, None]
        options = []
        for native_row in range(len(native_um)):
            for rank in range(k):
                value = float(query_distance[native_row, rank])
                if value <= max_um:
                    options.append((value, native_row, int(query_index[native_row, rank])))
        used_native = set()
        used_fused = set()
        for value, native_row, fused_row in sorted(options):
            if native_row in used_native or fused_row in used_fused:
                continue
            used_native.add(native_row)
            used_fused.add(fused_row)
            mapped[n0 + native_row] = f0 + fused_row
            distance[n0 + native_row] = value
    return mapped, distance


def _edgegraft_append_graph_ids(path, coords, graph):
    if path is None or not Path(path).is_file():
        return
    by_identity = {}
    for row in graph.node_attrs().iter_rows(named=True):
        key = (
            int(row["t"]), round(float(row["z"]), 5),
            round(float(row["y"]), 5), round(float(row["x"]), 5),
        )
        if key in by_identity:
            raise RuntimeError(f"Duplicate fused graph identity: {key}")
        by_identity[key] = int(row["node_id"])
    graph_ids = np.asarray([
        by_identity[(
            int(round(float(coord[0]))), round(float(coord[1]), 5),
            round(float(coord[2]), 5), round(float(coord[3]), 5),
        )]
        for coord in coords
    ], np.int64)
    with np.load(path, allow_pickle=False) as data:
        arrays = {name: np.asarray(data[name]) for name in data.files}
    arrays["fused_graph_node_id"] = graph_ids
    temporary = Path(path).with_suffix(".edgegraft.tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def _save_native_evidence(
    state, output_path, frames, spacing, fused_detector_coords
):''',
    ),
    (
        '''    dtypes = {
        "source_id": np.int64,''',
        '''    fused_coords = np.asarray(fused_detector_coords, np.float32)
    fused_frames = (
        np.rint(fused_coords[:, 0]).astype(np.int64, copy=False)
        if len(fused_coords) else np.empty(0, np.int64)
    )
    if len(fused_frames) and (
        np.any(fused_frames < 0) or np.any(fused_frames >= frames)
        or np.any(fused_frames[1:] < fused_frames[:-1])
    ):
        raise RuntimeError("Fused detector coordinates are not frame ordered")
    fused_counts = np.bincount(fused_frames, minlength=frames)[:frames]
    fused_offsets = np.concatenate([[0], np.cumsum(fused_counts, dtype=np.int64)])
    mapped_fused, map_distance = _edgegraft_greedy_native_map(
        native_coords, offsets, fused_coords, fused_offsets,
        np.asarray(spacing, np.float32), max_um=6.0,
    )
    dtypes = {
        "source_id": np.int64,''',
    ),
    (
        '''            native_node_coords=native_coords,
            native_frame_offsets=offsets,
            spacing_um=np.asarray(spacing, np.float32),''',
        '''            native_node_coords=native_coords,
            native_frame_offsets=offsets,
            mapped_ab_node=mapped_fused,
            map_distance_um=map_distance,
            ab_node_coords=fused_coords,
            ab_frame_offsets=fused_offsets,
            spacing_um=np.asarray(spacing, np.float32),''',
    ),
    (
        '''        T,
        ds.scale,
    ) if primary_native_state is not None else None''',
        '''        T,
        ds.scale,
        coords,
    ) if primary_native_state is not None else None''',
    ),
    (
        '''        T,
        ds.scale,
    ) if secondary_native_state is not None else None''',
        '''        T,
        ds.scale,
        coords,
    ) if secondary_native_state is not None else None''',
    ),
    (
        '''        graph = build_graph(coords, edges)
        if cfg.use_ilp and graph.num_edges() > 0:''',
        '''        graph = build_graph(coords, edges)
        if use_division_model:
            _edgegraft_append_graph_ids(
                primary_native_evidence_dir / f"{name}.npz"
                if primary_native_evidence_dir is not None else None,
                coords, graph,
            )
            _edgegraft_append_graph_ids(
                secondary_native_evidence_dir / f"{name}.npz"
                if secondary_native_evidence_dir is not None else None,
                coords, graph,
            )
        if cfg.use_ilp and graph.num_edges() > 0:''',
    ),
]
_s = _apply(
    _s, _edgegraft_mapping_replacements,
    "EdgeGRAFT exact native-to-fused identity export",
)

_candidate_old = '            candidates = sorted(\n                [\n                    (probs[i, j], i, j)\n                    for i in range(n_src)\n                    for j in range(n_tgt)\n                    if probs[i, j] > cfg.threshold\n                ],\n                reverse=True,\n            )\n\n            children_count: dict[int, int] = {}\n            parents_count: dict[int, int] = {}\n\n            for prob, i, j in candidates:\n                n_ch = children_count.get(i, 0)\n                n_pa = parents_count.get(j, 0)'

_candidate_new = '            _cand_src, _cand_tgt = np.nonzero(probs > cfg.threshold)\n            _cand_prob = probs[_cand_src, _cand_tgt]\n            _cand_order = np.lexsort((-_cand_tgt, -_cand_src, -_cand_prob))\n            _cand_src = _cand_src[_cand_order]\n            _cand_tgt = _cand_tgt[_cand_order]\n            _cand_prob = _cand_prob[_cand_order]\n\n            children_count: dict[int, int] = {}\n            parents_count: dict[int, int] = {}\n\n            for _cand_row in range(_cand_prob.shape[0]):\n                i = int(_cand_src[_cand_row])\n                j = int(_cand_tgt[_cand_row])\n                prob = _cand_prob[_cand_row]\n                n_ch = children_count.get(i, 0)\n                n_pa = parents_count.get(j, 0)'

# 10. Vectorised candidate extraction.
#
# The frozen loop reads every cell of the (n_src, n_tgt) probability matrix
# from Python -- 635k numpy scalar reads per frame pair on the densest video,
# roughly 57M per video -- to build a list that can never hold more than two
# entries per target: `probs` is a softmax over sources and the candidate
# threshold is 0.48. `probs` itself is left bit-identical (the softmax still
# runs on the GPU in fp32 and is still compared in float64 by numpy), and the
# ordering of `sorted(..., reverse=True)` over `(prob, i, j)` tuples --
# probability descending, then source index descending, then target index
# descending, with no ties possible on the full tuple -- is reproduced exactly
# by one lexsort. Accepted candidates keep the original per-pair
# `np.linalg.norm`, so the emitted distances are unchanged too.
if os.environ.get("BIOHUB_FAST_CANDIDATES", "0") == "1":
    _s = _apply(_s, [(_candidate_old, _candidate_new)], "vectorised candidates")

# 11. Device-level accelerators.
#
# cuDNN autotuning is worth having because every window of a video presents
# the same shape to the same convolutions: the benchmark cost is paid once and
# amortised over hundreds of identical launches. It can select a different
# algorithm than the default heuristic, which is why it is a switch.
#
# The fp16 wrapper is opt-in and unscored. `torch.autocast` keeps softmax and
# the reductions in fp32, but the convolutions and the transformer matmuls
# move to fp16, so both the detection logits and the association logits
# change. It is here because the GPU phase is 14% of the run and halving it is
# worth exactly that much -- measure before trusting it.
_accel_old = (
    '    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
    "    model, window_size, downsample = load_model(weights_path, device)"
)
_accel_new = '''    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if os.environ.get("BIOHUB_CUDNN_BENCHMARK", "0") == "1" and device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        print("cudnn.benchmark: on", flush=True)
    if os.environ.get("BIOHUB_AMP_FP16", "0") == "1":
        print("fp16 autocast: on (detection logits change)", flush=True)
    model, window_size, downsample = load_model(weights_path, device)'''

_amp_old = '''    return coords, all_edges


# =============================================================================
# Prediction loop
# ============================================================================='''
_amp_new = '''    return coords, all_edges


_predict_video_fp32 = predict_video

if os.environ.get("BIOHUB_AMP_FP16", "0") == "1":

    def predict_video(*args, **kwargs):
        """Opt-in mixed-precision wrapper around the frozen predictor."""
        with torch.autocast("cuda", dtype=torch.float16):
            return _predict_video_fp32(*args, **kwargs)


# =============================================================================
# Prediction loop
# ============================================================================='''

_s = _apply(_s, [(_accel_old, _accel_new), (_amp_old, _amp_new)], "device accelerators")

# 12. Uncompressed native evidence.
#
# Each `.npz` is deflated once by the producer and inflated by every stage
# that follows it: BaseGraph repair, the source-cardinality / UniGRAFT /
# live-V2 / ownership chain, EdgeGRAFT and CandidateGRAFT all call np.load on
# the same P1 and P2 tables. The arrays are identical either way, so the
# compression only buys disk. The shim keeps the compressed path for archives
# that would be large enough to threaten the working-directory quota.
_evidence_shim = '''def _biohub_savez(file, **arrays):
    """np.savez / np.savez_compressed, chosen by size and by env switch."""
    if os.environ.get("BIOHUB_UNCOMPRESSED_EVIDENCE", "0") == "1":
        total = sum(int(getattr(value, "nbytes", 0)) for value in arrays.values())
        cap = int(os.environ.get("BIOHUB_UNCOMPRESSED_EVIDENCE_MAX_BYTES", "0") or 0)
        if cap <= 0 or total <= cap:
            return np.savez(file, **arrays)
    return np.savez_compressed(file, **arrays)


def pool_kernel_from_um('''
_s = _apply(_s, [
    ("def pool_kernel_from_um(", _evidence_shim),
    ("    np.savez_compressed(temporary, **arrays)",
     "    _biohub_savez(temporary, **arrays)"),
    ("        np.savez_compressed(\n            handle,",
     "        _biohub_savez(\n            handle,"),
    ("            np.savez_compressed(\n                evidence_handle,",
     "            _biohub_savez(\n                evidence_handle,"),
], "uncompressed evidence")
if _s.count("np.savez_compressed(") != 1:  # only the shim's own fallback survives
    raise RuntimeError("evidence writer rewrite missed a call site")

_ps.write_text(_s)
print(
    "predict_unet_transformer.py: 11 patch groups declared "
    f"(fast candidates={os.environ.get('BIOHUB_FAST_CANDIDATES', '0')}, "
    f"cudnn.benchmark={os.environ.get('BIOHUB_CUDNN_BENCHMARK', '0')}, "
    f"fp16={os.environ.get('BIOHUB_AMP_FP16', '0')})"
)



In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, f"{PKG_DIR}/patch_predictor.py"], check=True, env=dict(os.environ)
)

## 2. Post-processing

Raw predictor graph to submission shard: graph repair, motion re-link, gap
closing, divisions, ownership and component cleanup.


In [ ]:
%%writefile /kaggle/working/biohub_pipeline/postprocess.py
"""Output-graph post-processing for the Biohub cell-tracking submission.

The notebook imports this module once in the parent process; the forked CPU
workers inherit the already-loaded runtimes. Every numeric constant comes from
the CFG JSON emitted by the notebook's CFG cell, so this file has no tuning of
its own.

Pipeline per video graph:
    raw ILP edges -> motion re-link -> single-parent repair -> gap closing
    -> divisions (combined Model-C/V2 decoder, or the safe-geometry fallback)
    -> isolated pruning -> short-track filtering -> line-fit smoothing
"""
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

CFG = json.loads(Path(os.environ["BIOHUB_CFG_JSON"]).read_text())
_PATHS = CFG["paths"]
_G = CFG["graph"]
_D = CFG["division"]
_M = CFG["models"]

TEST_DIR = Path(_PATHS["test_dir"])
SHARD_DIR = Path(_PATHS["shard_dir"])
MODEL_C_EVIDENCE_DIR = Path(_PATHS["model_c_evidence_dir"])
P1_EVIDENCE_DIR = Path(_PATHS["p1_evidence_dir"])
P2_EVIDENCE_DIR = Path(_PATHS["p2_evidence_dir"])

VOXEL_SCALE_UM = tuple(float(value) for value in CFG["voxel_scale_um"])
SCALE = np.asarray(VOXEL_SCALE_UM, dtype=np.float64)

SUBMISSION_COLUMNS = [
    "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id",
]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]

EDGE_MAX_UM = float(_G["edge_max_um"])
ENFORCE_NEXT_FRAME = bool(_G["enforce_next_frame"])
SINGLE_PARENT_REPAIR = bool(_G["single_parent_repair"])
PRUNE_ISOLATED = bool(_G["prune_isolated"])

MOTION_RELINK = bool(_G["motion_relink"])
RELINK_TIGHT_UM = float(_G["relink_tight_um"])
RELINK_JOINT_ASSIGNMENT = bool(_G.get("relink_joint_assignment", False))
RELINK_TIGHT_BONUS_UM = float(_G.get("relink_tight_bonus_um", 0.0))
RELINK_RELAXED_UM = float(_G["relink_relaxed_um"])
RELINK_VELOCITY_WEIGHT = float(_G["relink_velocity_weight"])
RELINK_VELOCITY_AXES = np.asarray(
    _G.get("relink_velocity_weight_axes") or [RELINK_VELOCITY_WEIGHT] * 3, dtype=float
)
RELINK_FRAME_REGISTRATION = bool(_G.get("relink_frame_registration", False))
RELINK_REGISTRATION_WEIGHT = float(_G.get("relink_registration_weight", 1.0))
RELINK_LEARNED_BONUS = float(_G["relink_learned_bonus"])
RELINK_MAX_MATCH_COST = float(_G.get("relink_max_match_cost", 0.0))
RELINK_CAP_INCLUDES_LEARNED = bool(_G.get("relink_cost_cap_includes_learned", False))
RELINK_MARGIN_UM = float(_G.get("relink_margin_um", 0.0))
RELINK_MOTION_STEPS = max(1, int(_G.get("relink_motion_steps", 1)))
RELINK_MOTION_AXES = np.asarray(
    _G.get("relink_motion_axes") or [0.5, 0.5, 0.5], dtype=float
)
RELINK_ORPHAN_PRIOR = bool(_G.get("relink_orphan_prior", False))
RELINK_ORPHAN_BASE_UM = float(_G.get("relink_orphan_base_um", 7.5))
RELINK_ORPHAN_SCALE = float(_G.get("relink_orphan_scale", 1.0))
RELINK_ORPHAN_FLOOR = float(_G.get("relink_orphan_floor", 0.5))
RELINK_MAX_FRAME_NODES = int(_G["relink_max_frame_nodes"])
RELINK_RAW_WEIGHT = float(_G["relink_raw_distance_weight"])
Z_EXTENT_UM = float(_G["motion_feature_z_extent_um"])

GAP_CLOSE = bool(_G["gap_close"])
GAP_CLOSE_UM = float(_G["gap_close_um"])
GAP_DENSITY_ADAPTIVE = bool(_G["gap_density_adaptive"])
GAP_DENSITY_REFERENCE_UM = float(_G["gap_density_reference_um"])
GAP_DENSITY_GAIN = float(_G["gap_density_gain"])
GAP_DENSITY_MAX_STEP_DELTA_UM = float(_G["gap_density_max_step_delta_um"])
GAP_DENSITY_NEIGHBORS = int(_G["gap_density_neighbors"])
GAP_REUSE_EXISTING = bool(_G["gap_reuse_existing"])
GAP_REUSE_UM = float(_G["gap_reuse_um"])
GAP_MAX_ADDED_FRAC = float(_G["gap_max_added_frac"])
GAP_MAX_ADDED_ABS = int(_G["gap_max_added_abs"])
GAP_REFINE_SYNTHETIC = bool(_G["gap_refine_synthetic"])
GAP_REFINE_WIN_Z = int(_G["gap_refine_win_z"])
GAP_REFINE_WIN_YX = int(_G["gap_refine_win_yx"])
GAP_REFINE_MAX_SHIFT_UM = float(_G["gap_refine_max_shift_um"])

DEEPCENTER_GAP_VETO = bool(_G.get("deepcenter_gap_veto", False))
DEEPCENTER_GAP_THRESHOLD = float(_G.get("deepcenter_gap_threshold", 0.25))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(
    _G.get("deepcenter_gap_confirm_min_span_um", 8.5)
)
DEEPCENTER_SCORE_WIN_Z = int(_G.get("deepcenter_score_win_z", 1))
DEEPCENTER_SCORE_WIN_YX = int(_G.get("deepcenter_score_win_yx", 2))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(
    _G.get("deepcenter_score_cache_max_frames", 8)
)
DEEPCENTER_DEVICE = str(_G.get("deepcenter_device", "cpu"))
DEEPCENTER_MODEL_CFG = CFG["models"]["deepcenter"]

ADDONLY_DIVISIONS = bool(_D.get("addonly_divisions", False))
ADDONLY_DIV_MIN_UM = float(_D.get("addonly_div_min_um", 0.5))
ADDONLY_DIV_MAX_UM = float(_D.get("addonly_div_max_um", 10.0))
ADDONLY_DIV_SISTER_MIN_UM = float(_D.get("addonly_div_sister_min_um", 4.0))
ADDONLY_DIV_SISTER_MAX_UM = float(_D.get("addonly_div_sister_max_um", 15.0))
ADDONLY_DIV_MAX_COS = float(_D.get("addonly_div_max_cos", -0.5))
ADDONLY_DIV_BUDGET_FRAC = float(_D.get("addonly_div_budget_frac", 0.002))

SAFE_DIVISIONS = bool(_G["safe_divisions"])
SAFE_DIV_MAX_UM = float(_G["safe_div_max_um"])
SAFE_DIV_SISTER_MAX_UM = float(_G["safe_div_sister_max_um"])
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(_G["safe_div_existing_child_max_um"])
SAFE_DIV_FRAME_FRAC_CAP = float(_G["safe_div_frame_frac_cap"])
SAFE_DIV_GLOBAL_FRAC_CAP = float(_G["safe_div_global_frac_cap"])
SAFE_DIV_SISTER_SCORE_WEIGHT = float(_G["safe_div_sister_score_weight"])

FILTER_SHORT_TRACKS = bool(_G["filter_short_tracks"])
MIN_TRACK_LEN = int(_G["min_track_len"])
KEEP_DIVISION_COMPONENTS = bool(_G["keep_division_components"])
BOUNDARY_TRACK_RESCUE = bool(_G.get("boundary_track_rescue", False))
BOUNDARY_TRACK_MIN_LEN = int(_G.get("boundary_track_min_len", 2))

LINEFIT_SMOOTH = bool(_G["linefit_smooth"])
LINEFIT_WEIGHT = float(_G["linefit_weight"])
LINEFIT_WINDOW = int(_G["linefit_window"])
FRAME_CACHE_MAX = int(_G["frame_cache_max_frames"])
NODE_BUDGET = bool(_G.get("node_budget", False))
NODE_BUDGET_RATIO = float(_G.get("node_budget_ratio", 1.0))
NODE_BUDGET_MAX_DROP_FRAC = float(_G.get("node_budget_max_drop_frac", 0.0))

COMBINED_DECODER = bool(_D["combined_decoder"])
MODEL_C_DIR = Path(_M["model_c"]["dir"])
MODEL_C_THRESHOLD = float(_D["model_c_threshold"])
GBM_THRESHOLD = float(_D["gbm_threshold"])
GBM_RESCUE_DELTA = float(_D["gbm_rescue_delta"])
GBM_STEAL_DELTA = float(_D["gbm_steal_delta"])
SOURCE_CARDINALITY_DIR = Path(_M["source_cardinality"]["dir"])
SOURCE_CARDINALITY_ENABLED = bool(_M["source_cardinality"]["enabled"])

UNIGRAFT_P1P2_DIR = Path(_M["unigraft_p1p2"]["dir"])
UNIGRAFT_P1P2_ENABLED = bool(_M["unigraft_p1p2"]["enabled"])

LIVE_V2_BUNDLE_DIR = Path(_M["live_v2_global_bundle"]["dir"])
LIVE_V2_BUNDLE_ENABLED = bool(_M["live_v2_global_bundle"]["enabled"])

OWNERSHIP_FULL_DIR = Path(_M["ownership_full_population"]["dir"])
OWNERSHIP_FULL_ENABLED = bool(_M["ownership_full_population"]["enabled"])

EDGEGRAFT_DIR = Path(_M["edgegraft"]["dir"])
EDGEGRAFT_ENABLED = bool(_M["edgegraft"]["enabled"])

CANDIDATEGRAFT_DIR = Path(_M["candidategraft"]["dir"])
CANDIDATEGRAFT_ENABLED = bool(_M["candidategraft"]["enabled"])

BASEGRAPH_REPAIR_DIR = Path(_M["basegraph_repair"]["dir"])
BASEGRAPH_REPAIR_ENABLED = bool(_M["basegraph_repair"]["enabled"])

MOTION_CORRECTOR_ENABLED = bool(_M["motion_corrector"]["enabled"])
MOTION_CORRECTOR_PATH = Path(_M["motion_corrector"]["path"])
MOTION_CORRECTOR_STRENGTH = float(_M["motion_corrector"]["strength"])

BASE_STATS = {
    "raw_edges": 0,
    "dropped_nonconsecutive_edges": 0,
    "dropped_long_edges": 0,
    "dropped_multi_parent_edges": 0,
    "motion_relink_edges": 0,
    "motion_relink_tight_edges": 0,
    "motion_relink_relaxed_edges": 0,
    "motion_relink_frames": 0,
    "motion_relink_joint_frames": 0,
    "motion_relink_replaced_raw_edges": 0,
    "motion_relink_fallback_raw": 0,
    "motion_relink_skipped_large_frame": 0,
    "relink_cost_cap_rejected": 0,
    "relink_cost_cap_emptied_frames": 0,
    "relink_margin_declined": 0,
    "addonly_division_candidates": 0,
    "addonly_divisions_added": 0,
    "gap_reuse_stale_skipped": 0,
    "native_evidence_missing": 0,
    "relink_orphan_declined": 0,
    "relink_registered_frames": 0,
    "relink_registration_shift_um_sum": 0.0,
    "motion_corrector_candidates": 0,
    "motion_corrector_abs_residual_sum": 0.0,
    "gap_candidates": 0,
    "gap_pairs_selected": 0,
    "gap_reused_existing": 0,
    "gap_inserted_synthetic": 0,
    "gap_added_nodes": 0,
    "gap_added_edges": 0,
    "gap_skipped_node_cap": 0,
    "gap_density_candidates_expanded": 0,
    "gap_density_candidates_restricted": 0,
    "gap_density_selected_outside_base": 0,
    "gap_refined_synthetic": 0,
    "gap_refine_failed": 0,
    "gap_refine_rejected_shift": 0,
    "deepcenter_gap_checked": 0,
    "deepcenter_gap_accepted": 0,
    "deepcenter_gap_rejected": 0,
    "deepcenter_gap_missing": 0,
    "deepcenter_gap_bypassed_short_span": 0,
    "deepcenter_gap_bypassed_observed_node": 0,
    "safe_division_candidates": 0,
    "safe_divisions_added": 0,
    "safe_division_skipped_cap": 0,
    "pruned_isolated_nodes": 0,
    "short_track_components_removed": 0,
    "short_track_nodes_removed": 0,
    "short_track_edges_removed": 0,
    "boundary_track_rescued_components": 0,
    "boundary_track_rescued_nodes": 0,
    "linefit_smoothed_nodes": 0,
    "linefit_skipped_nodes": 0,
    "node_budget_excess": 0,
    "node_budget_dropped_nodes": 0,
    "node_budget_dropped_edges": 0,
}


# ---------------------------------------------------------------------------
# Speed controls (scheduling and implementation only)
# ---------------------------------------------------------------------------
_S = CFG.get("speed", {})
FAST_GEFF_READER = bool(_S.get("fast_geff_reader", False))
FAST_SHARD_WRITER = bool(_S.get("fast_shard_writer", False))
SKIP_REDUNDANT_RAW_COPIES = bool(_S.get("skip_redundant_raw_copies", False))
STAGE_TIMING = bool(_S.get("stage_timing", False))

# The forked workers pin every math library to one thread, which is right
# while four upgrades share four vCPUs and wrong for the hours where one
# video holds the machine alone. The worker installs a hook here and the
# pipeline calls it between stages.
THREAD_RETUNE_HOOK = None
_THREAD_LIMITER = None
_THREAD_BUDGET = 0


def set_thread_budget(threads: int) -> int:
    """Apply an absolute thread limit to BLAS, OpenMP and torch."""
    global _THREAD_LIMITER, _THREAD_BUDGET
    threads = max(1, int(threads))
    if threads == _THREAD_BUDGET:
        return threads
    try:
        from threadpoolctl import threadpool_limits

        # Each call sets absolute limits, so repeated construction cannot
        # drift; the reference is kept only so the object outlives the call.
        _THREAD_LIMITER = threadpool_limits(limits=threads)
    except Exception:
        _THREAD_LIMITER = None
    try:
        torch.set_num_threads(threads)
    except Exception:
        pass
    for variable in (
        "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
    ):
        os.environ[variable] = str(threads)
    _THREAD_BUDGET = threads
    return threads


def retune_threads() -> None:
    hook = THREAD_RETUNE_HOOK
    if hook is None:
        return
    try:
        hook()
    except Exception:
        pass


def _mark(stats, name: str, started: float) -> float:
    """Record one stage's seconds and hand back the next stage's start."""
    now = time.monotonic()
    if STAGE_TIMING:
        stats[f"t_{name}"] = stats.get(f"t_{name}", 0.0) + (now - started)
    retune_threads()
    return time.monotonic()


STAGE_ORDER = [
    "graph_read", "basegraph", "edge_prefilter", "motion_relink",
    "single_parent", "gap_close", "division", "edgegraft",
    "prune", "component_filters", "candidategraft", "shard_write",
]


def stage_summary(stats: dict) -> str:
    """One compact line: the stages that actually cost something."""
    parts = []
    for name in STAGE_ORDER:
        seconds = float(stats.get(f"t_{name}", 0.0) or 0.0)
        if seconds >= 1.0:
            parts.append(f"{name}={seconds / 60.0:.1f}m")
    return " ".join(parts) if parts else "(all stages under 1s)"


# ---------------------------------------------------------------------------
# Geometry helpers
# ---------------------------------------------------------------------------
def node_point(node: dict) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


class _DCConvBlock3d(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        groups = min(8, out_channels)
        self.block = torch.nn.Sequential(
            torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(groups, out_channels),
            torch.nn.SiLU(inplace=True),
            torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(groups, out_channels),
            torch.nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class _DCDeepCenterUNet3D(torch.nn.Module):
    def __init__(self, in_channels=1, base_channels=24):
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DCConvBlock3d(in_channels, c)
        self.down1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DCConvBlock3d(c, c * 2)
        self.down2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DCConvBlock3d(c * 2, c * 4)
        self.down3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DCConvBlock3d(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DCConvBlock3d(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DCConvBlock3d(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DCConvBlock3d(c * 2, c)
        self.head = torch.nn.Conv3d(c, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        b = self.bottleneck(self.down3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)


_DEEPCENTER_BUNDLE = None
_DEEPCENTER_LOAD_ATTEMPTED = False


def _load_deepcenter_gap_veto():
    global _DEEPCENTER_BUNDLE, _DEEPCENTER_LOAD_ATTEMPTED
    if _DEEPCENTER_LOAD_ATTEMPTED:
        return _DEEPCENTER_BUNDLE
    _DEEPCENTER_LOAD_ATTEMPTED = True
    if not DEEPCENTER_GAP_VETO:
        return None
    checkpoint_path = Path(DEEPCENTER_MODEL_CFG["path"])
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"DeepCenter checkpoint is missing: {checkpoint_path}")
    device = torch.device(DEEPCENTER_DEVICE)
    if device.type == "cpu":
        # Four upgrade workers run concurrently; one BLAS thread each avoids
        # oversubscribing Kaggle's four CPU cores.
        torch.set_num_threads(1)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if int(checkpoint.get("epoch", -1)) != int(DEEPCENTER_MODEL_CFG["expected_epoch"]):
        raise RuntimeError(
            f"DeepCenter epoch mismatch: {checkpoint.get('epoch')} != "
            f"{DEEPCENTER_MODEL_CFG['expected_epoch']}"
        )
    from types import SimpleNamespace
    cfg = SimpleNamespace(**checkpoint.get("config", {}))
    model = _DCDeepCenterUNet3D(
        base_channels=int(getattr(cfg, "base_channels", 24))
    )
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    _DEEPCENTER_BUNDLE = {
        "model": model, "cfg": cfg, "device": device,
    }
    return _DEEPCENTER_BUNDLE


def _dc_pool_frame_xy(volume, factor):
    if factor <= 1:
        return volume.astype(np.float32, copy=False)
    z, y, x = volume.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)
    return cropped.reshape(
        z, y2 // factor, factor, x2 // factor, factor
    ).mean(axis=(2, 4))


def _dc_normalize_dynamic_range(volume, cfg):
    volume = np.asarray(volume, dtype=np.float32)
    lo = float(np.percentile(volume, float(getattr(cfg, "norm_lo_pct", 50.0))))
    hi = float(np.percentile(volume, float(getattr(cfg, "norm_hi_pct", 99.5))))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(volume, dtype=np.float32)
    ratio = (volume - lo) / (hi - lo)
    return np.clip(
        ratio,
        float(getattr(cfg, "norm_clip_lo", -0.5)),
        float(getattr(cfg, "norm_clip_hi", 6.0)),
    ).astype(np.float32)


def _deepcenter_heatmap(dataset, t, frame_cache, heatmap_cache):
    key = (dataset, int(t))
    if key in heatmap_cache:
        return heatmap_cache[key]
    bundle = _load_deepcenter_gap_veto()
    if bundle is None:
        return None
    cfg = bundle["cfg"]
    pool_factor = int(getattr(cfg, "pool_factor", 4))
    volume = read_test_frame(dataset, int(t), frame_cache)
    image = _dc_normalize_dynamic_range(
        _dc_pool_frame_xy(volume, pool_factor), cfg
    )
    with torch.no_grad():
        tensor = torch.from_numpy(image[None, None]).to(
            device=bundle["device"], dtype=torch.float32
        )
        heatmap = torch.sigmoid(bundle["model"](tensor))[0, 0].cpu().numpy()
    heatmap = heatmap.astype(np.float32, copy=False)
    heatmap_cache[key] = heatmap
    while len(heatmap_cache) > max(1, DEEPCENTER_SCORE_CACHE_MAX_FRAMES):
        heatmap_cache.pop(next(iter(heatmap_cache)))
    return heatmap


def deepcenter_accept_gap(dataset, t, point, frame_cache, heatmap_cache, stats):
    stats["deepcenter_gap_checked"] += 1
    heatmap = _deepcenter_heatmap(dataset, int(t), frame_cache, heatmap_cache)
    if heatmap is None or not heatmap.size:
        stats["deepcenter_gap_missing"] += 1
        return True
    cfg = _DEEPCENTER_BUNDLE["cfg"]
    factor = max(1, int(getattr(cfg, "pool_factor", 4)))
    z = int(round(float(point[0])))
    y = int(round(float(point[1]) / factor))
    x = int(round(float(point[2]) / factor))
    z0, z1 = max(0, z - DEEPCENTER_SCORE_WIN_Z), min(
        heatmap.shape[0], z + DEEPCENTER_SCORE_WIN_Z + 1
    )
    y0, y1 = max(0, y - DEEPCENTER_SCORE_WIN_YX), min(
        heatmap.shape[1], y + DEEPCENTER_SCORE_WIN_YX + 1
    )
    x0, x1 = max(0, x - DEEPCENTER_SCORE_WIN_YX), min(
        heatmap.shape[2], x + DEEPCENTER_SCORE_WIN_YX + 1
    )
    patch = heatmap[z0:z1, y0:y1, x0:x1]
    if not patch.size:
        stats["deepcenter_gap_missing"] += 1
        return True
    score = float(np.max(patch))
    if not np.isfinite(score):
        stats["deepcenter_gap_missing"] += 1
        return True
    if score < DEEPCENTER_GAP_THRESHOLD:
        stats["deepcenter_gap_rejected"] += 1
        return False
    stats["deepcenter_gap_accepted"] += 1
    return True


def point_distance_um(a, b) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def edge_distance_um(source: dict, target: dict) -> float:
    return point_distance_um(node_point(source), node_point(target))


def edge_sort_key(edge: dict) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    return (float(prob) if prob is not None else 0.0, -float(edge["distance_um"]))


def _coords_um(nodes_by_id: dict, ids) -> np.ndarray:
    """(n, 3) micrometre positions for `ids`, in the given order."""
    out = np.empty((len(ids), 3), dtype=np.float64)
    for index, node_id in enumerate(ids):
        node = nodes_by_id[node_id]
        out[index, 0] = node["z"]
        out[index, 1] = node["y"]
        out[index, 2] = node["x"]
    return out * SCALE


def _next_node_id(nodes_by_id: dict) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1


def _ids_by_frame(nodes_by_id: dict) -> dict[int, list[int]]:
    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()
    return ids_by_t


# ---------------------------------------------------------------------------
# Raw volume access (only needed to refine synthetic gap nodes)
# ---------------------------------------------------------------------------
def read_test_frame(dataset: str, t: int, frame_cache: dict) -> np.ndarray:
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    frame_shape = tuple(int(value) for value in meta["shape"])[1:]
    dtype = np.dtype(meta["data_type"])
    frame = None
    try:
        import blosc2

        raw = (zarr_path / "0" / "c" / str(t) / "0" / "0" / "0").read_bytes()
        flat = np.frombuffer(blosc2.decompress(raw), dtype=dtype)
        if flat.size == int(np.prod(frame_shape)):
            frame = flat.reshape(frame_shape).copy()
    except Exception:
        frame = None
    if frame is None:
        import zarr

        frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])
    frame_cache[t] = frame
    while len(frame_cache) > FRAME_CACHE_MAX:
        frame_cache.pop(next(iter(frame_cache)))
    return frame


def refine_synthetic_midpoint(dataset, t, midpoint, frame_cache, stats):
    """Snap an interpolated gap node onto the local intensity centroid."""
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = (int(round(value)) for value in midpoint)
        z0, z1 = max(0, z - GAP_REFINE_WIN_Z), min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
        y0, y1 = max(0, y - GAP_REFINE_WIN_YX), min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
        x0, x1 = max(0, x - GAP_REFINE_WIN_YX), min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        weights = np.maximum(patch - float(np.percentile(patch, 20.0)), 0.0)
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        refined = (
            float((weights.sum(axis=(1, 2)) * np.arange(z0, z1)).sum() / total),
            float((weights.sum(axis=(0, 2)) * np.arange(y0, y1)).sum() / total),
            float((weights.sum(axis=(0, 1)) * np.arange(x0, x1)).sum() / total),
        )
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1
            return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined
    except Exception:
        stats["gap_refine_failed"] += 1
        return midpoint


# ---------------------------------------------------------------------------
# Learned motion-cost corrector
# ---------------------------------------------------------------------------
MOTION_FEATURES = [
    "base_cost", "raw_dist", "registered_dist", "motion_dist",
    "abs_dz", "abs_dy", "abs_dx", "abs_reg_dz", "abs_reg_dy", "abs_reg_dx",
    "velocity_z", "velocity_y", "velocity_x", "velocity_mag",
    "shift_z", "shift_y", "shift_x", "shift_mag",
    "density_src", "density_tgt", "z_boundary_src", "z_boundary_tgt",
    "has_predecessor",
]


class _MotionResidual(torch.nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_features, 64), torch.nn.SiLU(), torch.nn.Dropout(0.05),
            torch.nn.Linear(64, 32), torch.nn.SiLU(), torch.nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _load_motion_corrector():
    if not MOTION_CORRECTOR_ENABLED:
        return None
    checkpoint = torch.load(MOTION_CORRECTOR_PATH, map_location="cpu", weights_only=False)
    if list(checkpoint["features"]) != MOTION_FEATURES:
        raise ValueError(f"Unexpected motion-corrector features: {checkpoint['features']}")
    model = _MotionResidual(len(MOTION_FEATURES))
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return {
        "model": model,
        "mean": torch.as_tensor(checkpoint["mean"], dtype=torch.float32),
        "std": torch.as_tensor(checkpoint["std"], dtype=torch.float32),
        "residual_scale": float(checkpoint["residual_scale"]),
    }


MOTION_CORRECTOR = _load_motion_corrector()


@torch.no_grad()
def _motion_cost_residual(features: np.ndarray) -> np.ndarray:
    if MOTION_CORRECTOR is None or features.size == 0:
        return np.zeros(len(features), dtype=np.float64)
    x = torch.from_numpy(np.ascontiguousarray(features, dtype=np.float32))
    x = (x - MOTION_CORRECTOR["mean"]) / MOTION_CORRECTOR["std"]
    scale = MOTION_CORRECTOR["residual_scale"]
    raw = MOTION_CORRECTOR["model"](x)
    return (scale * torch.tanh(raw / scale)).numpy().astype(np.float64)


def _group_min(index: np.ndarray, values: np.ndarray, size: int) -> np.ndarray:
    """Smallest value per group id; +inf for a group with no member."""
    best = np.full(size, np.inf)
    if index.size:
        # Writing in descending value order leaves the smallest per group.
        order = np.argsort(values, kind="stable")[::-1]
        best[index[order]] = values[order]
    return best


def _candidate_position(
    rows: np.ndarray, cols: np.ndarray, want_rows: np.ndarray, want_cols: np.ndarray,
    width: int,
) -> np.ndarray:
    """Index into the candidate arrays of each (row, col) pair, which must exist."""
    key = rows.astype(np.int64) * width + cols
    order = np.argsort(key, kind="stable")
    return order[np.searchsorted(key[order], want_rows.astype(np.int64) * width + want_cols)]


def _sanitized_edge_probs(edges: list[dict]) -> dict[tuple[int, int], float]:
    """Best finite probability per (source, target), squashed into [0, 1]."""
    out: dict[tuple[int, int], float] = {}
    for edge in edges:
        prob = edge.get("edge_prob")
        if prob is None:
            continue
        try:
            value = float(prob)
        except (TypeError, ValueError):
            continue
        if not np.isfinite(value):
            continue
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        key = (int(edge["source_id"]), int(edge["target_id"]))
        if value > out.get(key, -1.0):
            out[key] = value
    return out


# ---------------------------------------------------------------------------
# Motion-aware re-linking
# ---------------------------------------------------------------------------
def _frame_registration(
    source_pos: np.ndarray, target_pos: np.ndarray, gate_um: float
) -> np.ndarray:
    """Global translation between two frames, in um.

    Median displacement from each source to its nearest target, gated so that
    sources whose neighbourhood emptied out do not drag the estimate. The
    median is what makes it robust: it is unaffected by the minority of
    sources whose nearest target is the wrong cell.

    This is the quantity the learned motion corrector calls ``shift`` and was
    trained on; the frozen pipeline passes zeros in its place.
    """
    if len(source_pos) < 5 or len(target_pos) < 5:
        return np.zeros(3)
    distance, index = cKDTree(target_pos).query(source_pos, k=1)
    keep = distance <= gate_um
    if int(keep.sum()) < 5:
        return np.zeros(3)
    return np.median(target_pos[index[keep]] - source_pos[keep], axis=0)


def motion_relink_edges(nodes_by_id: dict, stats, learned_edge_probs=None) -> list[dict]:
    """Rebuild the frame-to-frame linking by motion-predicted bipartite matching.

    Cost for a candidate pair is ``|q_t - q_pred| + w_raw * |q_t - q_s|`` minus
    the learned association bonus and the learned motion residual. Candidates
    are gated by a KD-tree, so the assignment matrix is the only dense object.
    """
    if not MOTION_RELINK or not nodes_by_id:
        return []

    ids_by_t = _ids_by_frame(nodes_by_id)
    if max((len(ids) for ids in ids_by_t.values()), default=0) > RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    probs = learned_edge_probs or {}
    position: dict[int, np.ndarray] = {}
    density: dict[int, np.ndarray] = {}
    for t, ids in ids_by_t.items():
        coords = _coords_um(nodes_by_id, ids)
        position[t] = coords
        counts = cKDTree(coords).query_ball_point(coords, 15.0, return_length=True)
        density[t] = np.maximum(counts.astype(np.float64) - 1.0, 0.0)

    # Position history per node, most recent first. A single entry reproduces
    # the frozen one-step prior exactly.
    history: dict[int, list[np.ndarray]] = {}
    selected: list[dict] = []

    for t in sorted(ids_by_t):
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue

        source_pos = position[t]
        target_pos = position[t + 1]
        velocity = np.zeros_like(source_pos)
        prior = np.zeros_like(source_pos)
        has_predecessor = np.zeros(len(source_ids), dtype=np.float64)
        for index, node_id in enumerate(source_ids):
            past = history.get(node_id)
            if past:
                # `velocity` stays the single previous step, because that is
                # what the motion corrector was trained on. `prior` is the
                # mean of the last `steps` displacements, which telescopes to
                # (now - then) / steps, and only steers the prediction.
                velocity[index] = source_pos[index] - past[0]
                steps = min(len(past), RELINK_MOTION_STEPS)
                prior[index] = (source_pos[index] - past[steps - 1]) / steps
                has_predecessor[index] = 1.0
        shift = (
            _frame_registration(source_pos, target_pos, RELINK_RELAXED_UM)
            if RELINK_FRAME_REGISTRATION
            else np.zeros(3)
        )
        if RELINK_FRAME_REGISTRATION:
            stats["relink_registered_frames"] += 1
            stats["relink_registration_shift_um_sum"] += float(np.linalg.norm(shift))
        predicted = (
            source_pos
            + (RELINK_VELOCITY_AXES if RELINK_MOTION_STEPS == 1 else RELINK_MOTION_AXES)
            * prior
            + RELINK_REGISTRATION_WEIGHT * shift
        )
        velocity_magnitude = np.linalg.norm(velocity, axis=1)
        z_boundary_src = np.clip(source_pos[:, 0] / Z_EXTENT_UM, 0.0, 1.0)
        z_boundary_src = np.minimum(z_boundary_src, 1.0 - z_boundary_src)
        z_boundary_tgt = np.clip(target_pos[:, 0] / Z_EXTENT_UM, 0.0, 1.0)
        z_boundary_tgt = np.minimum(z_boundary_tgt, 1.0 - z_boundary_tgt)

        def assign_pass(source_sel: np.ndarray, target_sel: np.ndarray, gate_um: float,
                        tight_bonus: float = 0.0):
            if source_sel.size == 0 or target_sel.size == 0:
                return np.empty(0, np.int64), np.empty(0, np.int64)
            local_source = source_pos[source_sel]
            local_target = target_pos[target_sel]
            neighbours = cKDTree(local_target).query_ball_point(local_source, gate_um)
            counts = np.fromiter((len(item) for item in neighbours), np.int64, len(neighbours))
            if not counts.any():
                return np.empty(0, np.int64), np.empty(0, np.int64)
            rows = np.repeat(np.arange(len(source_sel), dtype=np.int64), counts)
            cols = np.fromiter(
                (index for item in neighbours for index in item), np.int64, int(counts.sum())
            )
            delta = local_target[cols] - local_source[rows]
            raw = np.linalg.norm(delta, axis=1)
            inside = raw <= gate_um
            rows, cols, delta, raw = rows[inside], cols[inside], delta[inside], raw[inside]
            if rows.size == 0:
                return np.empty(0, np.int64), np.empty(0, np.int64)

            global_source = source_sel[rows]
            global_target = target_sel[cols]
            motion = np.linalg.norm(
                local_target[cols] - predicted[global_source], axis=1
            )
            probability = np.fromiter(
                (
                    probs.get((source_ids[s], target_ids[g]), 0.0)
                    for s, g in zip(global_source, global_target)
                ),
                np.float64,
                rows.size,
            )
            base_cost = motion + RELINK_RAW_WEIGHT * raw
            values = base_cost - RELINK_LEARNED_BONUS * probability

            if MOTION_CORRECTOR is not None:
                absolute = np.abs(delta)
                # registered_dist / abs_reg_* / shift_* are real features of
                # the trained corrector, not padding. Frozen code passed
                # duplicates of raw and four zero columns.
                registered_delta = delta - shift
                registered = np.linalg.norm(registered_delta, axis=1)
                features = np.column_stack([
                    base_cost, raw, registered, motion,
                    absolute, np.abs(registered_delta),
                    velocity[global_source], velocity_magnitude[global_source],
                    np.tile(shift, (rows.size, 1)),
                    np.full(rows.size, float(np.linalg.norm(shift))),
                    density[t][global_source], density[t + 1][global_target],
                    z_boundary_src[global_source], z_boundary_tgt[global_target],
                    has_predecessor[global_source],
                ])
                residual = _motion_cost_residual(features) * MOTION_CORRECTOR_STRENGTH
                values = values - residual
                stats["motion_corrector_candidates"] += int(rows.size)
                stats["motion_corrector_abs_residual_sum"] += float(np.abs(residual).sum())

            if tight_bonus > 0.0:
                # Preference, not a gate: a short match is cheaper, but the
                # solver may still decline it to keep a cell that has no other
                # option. Applied after the corrector and before the cap, so
                # the geometric cap never sees the discount.
                values = values - tight_bonus * (raw <= RELINK_TIGHT_UM)

            # ---- the refusal -------------------------------------
            # linear_sum_assignment pairs every source that has any target
            # inside the gate. Dropping the candidates whose geometry is
            # implausible before the solve is what lets a source stay open:
            # the assignment then simply has nothing to give it.
            if RELINK_MAX_MATCH_COST > 0.0:
                judged = values if RELINK_CAP_INCLUDES_LEARNED else base_cost
                keep = judged <= RELINK_MAX_MATCH_COST
                rejected = int(keep.size - keep.sum())
                if rejected:
                    stats["relink_cost_cap_rejected"] += rejected
                    rows, cols, values = rows[keep], cols[keep], values[keep]
                    if rows.size == 0:
                        stats["relink_cost_cap_emptied_frames"] += 1
                        return np.empty(0, np.int64), np.empty(0, np.int64)

            big = gate_um * 1000.0 + 1.0
            cost = np.full((source_sel.size, target_sel.size), big, dtype=np.float64)
            cost[rows, cols] = values

            if RELINK_ORPHAN_PRIOR:
                # HOCT's variable appearance cost, per target: a column whose
                # best evidence is weak is cheaper to leave unmatched than to
                # hand to whichever body happens to be nearest. Implemented as
                # one slack row per target, so the assignment itself decides.
                surviving = probability[keep] if RELINK_MAX_MATCH_COST > 0.0 else probability
                best = np.zeros(target_sel.size, np.float64)
                np.maximum.at(best, cols, surviving)
                # HOCT: w_a(i) = base * (1 - p_appear(i)), and a target whose
                # best association evidence is strong is unlikely to be an
                # appearance -- so declining it must be expensive, and
                # declining a target nothing is confident about must be free.
                orphan = RELINK_ORPHAN_BASE_UM * np.clip(
                    RELINK_ORPHAN_FLOOR
                    + (1.0 - RELINK_ORPHAN_FLOOR) * RELINK_ORPHAN_SCALE * best,
                    0.0, 1.0,
                )
                cost = np.vstack([cost, np.tile(orphan, (source_sel.size, 1))])

            row_index, col_index = linear_sum_assignment(cost)
            if RELINK_ORPHAN_PRIOR:
                real = row_index < source_sel.size
                stats["relink_orphan_declined"] += int((~real).sum())
                row_index, col_index = row_index[real], col_index[real]
            matched = cost[row_index, col_index] < big
            row_index, col_index = row_index[matched], col_index[matched]

            # ---- the second refusal ------------------------------------
            # A wrong link costs the metric twice - the false positive, and
            # the true edge it displaced - while a refusal costs once. So a
            # match is only worth taking when it beats its runner-up by more
            # than `margin`: the best alternative for that source, and the
            # best alternative for that target, are both checked. The solve
            # is already globally optimal, so this is not second-guessing the
            # assignment; it is declining the ones it had no evidence for.
            if RELINK_MARGIN_UM > 0.0 and row_index.size:
                chosen = _candidate_position(
                    rows, cols, row_index, col_index, target_sel.size
                )
                others = np.ones(rows.size, dtype=bool)
                others[chosen] = False
                runner_source = _group_min(rows[others], values[others], source_sel.size)
                runner_target = _group_min(cols[others], values[others], target_sel.size)
                slack = (
                    np.minimum(runner_source[row_index], runner_target[col_index])
                    - values[chosen]
                )
                confident = slack >= RELINK_MARGIN_UM
                stats["relink_margin_declined"] += int((~confident).sum())
                row_index, col_index = row_index[confident], col_index[confident]

            return source_sel[row_index], target_sel[col_index]

        open_sources = np.ones(len(source_ids), dtype=bool)
        open_targets = np.ones(len(target_ids), dtype=bool)
        if RELINK_JOINT_ASSIGNMENT:
            # One solve over the relaxed candidate set. The tight gate becomes
            # a discount rather than a first-come claim, so a short match can
            # no longer strand a fast cell that had no alternative.
            schedule = (("joint", RELINK_RELAXED_UM, RELINK_TIGHT_BONUS_UM),)
        else:
            schedule = (("tight", RELINK_TIGHT_UM, 0.0),
                        ("relaxed", RELINK_RELAXED_UM, 0.0))
        for pass_name, gate_um, tight_bonus in schedule:
            matched_source, matched_target = assign_pass(
                np.flatnonzero(open_sources), np.flatnonzero(open_targets), gate_um,
                tight_bonus,
            )
            if matched_source.size == 0:
                continue
            open_sources[matched_source] = False
            open_targets[matched_target] = False

            raw = np.linalg.norm(target_pos[matched_target] - source_pos[matched_source], axis=1)
            if pass_name == "joint":
                # Report by which gate the accepted match actually fell in, so
                # these stay comparable with the two-pass notebooks.
                inside_tight = int((raw <= RELINK_TIGHT_UM).sum())
                stats["motion_relink_tight_edges"] += inside_tight
                stats["motion_relink_relaxed_edges"] += int(matched_source.size) - inside_tight
                stats["motion_relink_joint_frames"] += 1
            else:
                stats[f"motion_relink_{pass_name}_edges"] += int(matched_source.size)
            motion = np.linalg.norm(
                target_pos[matched_target] - predicted[matched_source], axis=1
            )
            for offset in range(matched_source.size):
                source_id = source_ids[int(matched_source[offset])]
                target_id = target_ids[int(matched_target[offset])]
                selected.append({
                    "source_id": source_id,
                    "target_id": target_id,
                    "edge_prob": probs.get((source_id, target_id), 0.0),
                    "distance_um": float(raw[offset]),
                    "motion_distance_um": float(motion[offset]),
                    "motion_relinked": 1,
                    "motion_pass": pass_name,
                })
                source_index = int(matched_source[offset])
                history[target_id] = [
                    source_pos[source_index],
                    *history.get(source_ids[source_index], [])[:RELINK_MOTION_STEPS - 1],
                ]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected)
    return selected


# ---------------------------------------------------------------------------
# Single-frame gap closing
# ---------------------------------------------------------------------------
def close_single_frame_gaps(
    nodes_by_id, edges, stats, dataset=None, frame_cache=None,
    deepcenter_enabled=False,
):
    """Bridge one-frame track interruptions with a reused or synthetic node."""
    if not GAP_CLOSE or not edges:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    all_ids_by_t = _ids_by_frame(nodes_by_id)
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)
    for bucket in (ends_by_t, starts_by_t, isolated_by_t):
        for ids in bucket.values():
            ids.sort()

    max_synthetic = min(
        GAP_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_MAX_ADDED_FRAC))) if GAP_MAX_ADDED_FRAC > 0 else 0,
    )
    next_id = _next_node_id(nodes_by_id)
    frame_cache = {} if frame_cache is None else frame_cache
    deepcenter_heatmap_cache = {}
    used_starts: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict] = []

    spacing_cache: dict[int, np.ndarray] = {}

    def frame_spacing(t: int) -> np.ndarray:
        """Median distance to the nearest neighbours, per node of frame `t`."""
        cached = spacing_cache.get(t)
        if cached is not None:
            return cached
        ids = all_ids_by_t.get(t, [])
        if len(ids) <= 1:
            result = np.full(len(ids), GAP_DENSITY_REFERENCE_UM)
        else:
            coords = _coords_um(nodes_by_id, ids)
            k = min(len(ids), max(2, GAP_DENSITY_NEIGHBORS + 1))
            distances = np.atleast_2d(cKDTree(coords).query(coords, k=k)[0])[:, 1:]
            distances = np.where(np.isfinite(distances), distances, np.nan)
            with np.errstate(invalid="ignore"):
                result = np.nanmedian(distances, axis=1)
            result = np.where(np.isfinite(result), result, GAP_DENSITY_REFERENCE_UM)
        spacing_cache[t] = result
        return result

    isolated_cache: dict[int, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}

    def isolated_pool(t: int):
        cached = isolated_cache.get(t)
        if cached is None:
            ids = np.asarray(isolated_by_t.get(t, []), dtype=np.int64)
            cached = (ids, _coords_um(nodes_by_id, ids), np.ones(ids.size, dtype=bool))
            isolated_cache[t] = cached
        return cached

    threshold_um = GAP_CLOSE_UM * 2.0
    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = [node_id for node_id in starts_by_t.get(t + 2, []) if node_id not in used_starts]
        if not end_ids or not start_ids:
            continue

        distance = cdist(
            _coords_um(nodes_by_id, end_ids), _coords_um(nodes_by_id, start_ids)
        )
        adaptive = np.full_like(distance, threshold_um)
        if GAP_DENSITY_ADAPTIVE:
            end_index = {node_id: i for i, node_id in enumerate(all_ids_by_t[t])}
            start_index = {node_id: i for i, node_id in enumerate(all_ids_by_t[t + 2])}
            source_spacing = frame_spacing(t)[[end_index[i] for i in end_ids]]
            target_spacing = frame_spacing(t + 2)[[start_index[j] for j in start_ids]]
            local = 0.5 * (source_spacing[:, None] + target_spacing[None, :])
            step = np.clip(
                GAP_DENSITY_GAIN * (local - GAP_DENSITY_REFERENCE_UM),
                -GAP_DENSITY_MAX_STEP_DELTA_UM,
                GAP_DENSITY_MAX_STEP_DELTA_UM,
            )
            adaptive = threshold_um + 2.0 * step

        base_allowed = distance <= threshold_um
        allowed = distance <= adaptive
        stats["gap_density_candidates_expanded"] += int((allowed & ~base_allowed).sum())
        stats["gap_density_candidates_restricted"] += int((base_allowed & ~allowed).sum())
        stats["gap_candidates"] += int(allowed.sum())
        if not allowed.any():
            continue

        big = float(np.max(adaptive)) * 1000.0 + 1.0
        row_index, col_index = linear_sum_assignment(np.where(allowed, distance, big))

        for r, c in zip(row_index, col_index):
            if not allowed[r, c]:
                continue
            source_id = end_ids[int(r)]
            target_id = start_ids[int(c)]
            if source_id in outgoing or target_id in used_starts:
                continue
            if not base_allowed[r, c]:
                stats["gap_density_selected_outside_base"] += 1

            source = nodes_by_id[source_id]
            target = nodes_by_id[target_id]
            mid_t = int(source["t"]) + 1
            mid_point = (
                (float(source["z"]) + float(target["z"])) / 2.0,
                (float(source["y"]) + float(target["y"])) / 2.0,
                (float(source["x"]) + float(target["x"])) / 2.0,
            )

            middle_id = None
            if GAP_REUSE_EXISTING:
                pool_ids, pool_coords, alive = isolated_pool(mid_t)
                if alive.any():
                    reuse = np.where(
                        alive,
                        np.linalg.norm(pool_coords - np.asarray(mid_point) * SCALE, axis=1),
                        np.inf,
                    )
                    while True:
                        best = int(np.argmin(reuse))
                        if reuse[best] > GAP_REUSE_UM:
                            break
                        candidate = int(pool_ids[best])
                        # `isolated_by_t` is a snapshot taken before any gap
                        # was closed. A node that has been linked since - as
                        # another gap's start, or as its midpoint - would take
                        # a second parent here, and the final contract rejects
                        # in-degree 2. Start frames are visited before midpoint
                        # frames, so this is reachable whenever orphans are
                        # common.
                        if candidate in incoming or candidate in outgoing:
                            alive[best] = False
                            reuse[best] = np.inf
                            stats["gap_reuse_stale_skipped"] += 1
                            continue
                        middle_id = candidate
                        alive[best] = False
                        stats["gap_reused_existing"] += 1
                        break

            if middle_id is None:
                if synthetic_added >= max_synthetic:
                    stats["gap_skipped_node_cap"] += 1
                    continue
                middle_id = next_id
                next_id += 1
                refined = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                nodes_by_id[middle_id] = {
                    "node_id": middle_id,
                    "t": mid_t,
                    "z": refined[0],
                    "y": refined[1],
                    "x": refined[2],
                    "gap_synthetic": 1,
                }
                synthetic_added += 1
                stats["gap_inserted_synthetic"] += 1

            middle = nodes_by_id[middle_id]
            gap_span_um = float(distance[r, c])
            synthetic_middle = int(middle.get("gap_synthetic", 0)) == 1
            if deepcenter_enabled and not synthetic_middle:
                stats["deepcenter_gap_bypassed_observed_node"] += 1
            elif (
                deepcenter_enabled
                and synthetic_middle
                and gap_span_um < DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM
            ):
                stats["deepcenter_gap_bypassed_short_span"] += 1
            elif (
                deepcenter_enabled
                and synthetic_middle
                and not deepcenter_accept_gap(
                    dataset, mid_t, node_point(middle), frame_cache,
                    deepcenter_heatmap_cache, stats,
                )
            ):
                nodes_by_id.pop(middle_id, None)
                synthetic_added = max(0, synthetic_added - 1)
                stats["gap_inserted_synthetic"] = max(
                    0, stats["gap_inserted_synthetic"] - 1
                )
                continue

            new_edges.append({
                "source_id": source_id,
                "target_id": middle_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(source, middle),
                "gap_closed": 1,
            })
            new_edges.append({
                "source_id": middle_id,
                "target_id": target_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(middle, target),
                "gap_closed": 1,
            })
            outgoing.update((source_id, middle_id))
            incoming.update((middle_id, target_id))
            used_starts.add(target_id)
            stats["gap_pairs_selected"] += 1
            stats["gap_added_edges"] += 2

    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, ([*edges, *new_edges] if new_edges else edges)


def add_only_divisions(nodes_by_id, edges, stats):
    """Give a one-child source a second daughter where the geometry is a division.

    Adds edges only: no node is created, a target must be unclaimed, a source
    must have exactly one child, and a lineage may fork once. Candidates are
    ranked by cos(existing child, candidate) because daughters separate in
    near-opposite directions - median -0.75 over the 151 training divisions,
    against 0 for an arbitrary neighbour.
    """
    if not ADDONLY_DIVISIONS or not edges or not nodes_by_id:
        return edges

    children: dict[int, list[int]] = {}
    incoming: set[int] = set()
    for edge in edges:
        children.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))
        incoming.add(int(edge["target_id"]))

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    for edge in edges:
        source_id, target_id = int(edge["source_id"]), int(edge["target_id"])
        if source_id in parent and target_id in parent:
            left, right = find(source_id), find(target_id)
            if left != right:
                parent[left] = right
    forked = {find(i) for i, kids in children.items() if len(kids) > 1 and i in parent}

    ids_by_t = _ids_by_frame(nodes_by_id)
    proposals: list[tuple[float, int, int]] = []
    for t in sorted(ids_by_t):
        if t + 1 not in ids_by_t:
            continue
        sources = [
            i for i in ids_by_t[t]
            if len(children.get(i, ())) == 1 and find(i) not in forked
        ]
        free = [i for i in ids_by_t[t + 1] if i not in incoming]
        if not sources or not free:
            continue
        free_coords = _coords_um(nodes_by_id, free)
        source_coords = _coords_um(nodes_by_id, sources)
        near = cKDTree(free_coords).query_ball_point(source_coords, ADDONLY_DIV_MAX_UM)
        for index, source_id in enumerate(sources):
            if not near[index]:
                continue
            child = nodes_by_id.get(children[source_id][0])
            if child is None or int(child["t"]) != t + 1:
                continue
            child_vector = np.asarray(node_point(child)) * SCALE - source_coords[index]
            child_distance = float(np.linalg.norm(child_vector))
            if not ADDONLY_DIV_MIN_UM <= child_distance <= ADDONLY_DIV_MAX_UM:
                continue
            offsets = np.asarray(near[index], dtype=np.int64)
            vectors = free_coords[offsets] - source_coords[index]
            distances = np.linalg.norm(vectors, axis=1)
            sisters = np.linalg.norm(
                free_coords[offsets] - (source_coords[index] + child_vector), axis=1
            )
            cosines = (vectors @ child_vector) / np.maximum(
                distances * child_distance, 1e-9
            )
            keep = (
                (distances >= ADDONLY_DIV_MIN_UM)
                & (cosines <= ADDONLY_DIV_MAX_COS)
                & (sisters >= ADDONLY_DIV_SISTER_MIN_UM)
                & (sisters <= ADDONLY_DIV_SISTER_MAX_UM)
            )
            for offset in np.flatnonzero(keep):
                proposals.append(
                    (float(cosines[offset]), source_id, free[int(offsets[offset])])
                )

    stats["addonly_division_candidates"] = len(proposals)
    if not proposals:
        return edges

    budget = max(0, int(round(len(nodes_by_id) * ADDONLY_DIV_BUDGET_FRAC)))
    proposals.sort()
    added: list[dict] = []
    used_sources: set[int] = set()
    used_targets: set[int] = set()
    for _, source_id, target_id in proposals:
        if len(added) >= budget:
            break
        if source_id in used_sources or target_id in used_targets:
            continue
        if find(source_id) in forked or find(target_id) in forked:
            continue
        used_sources.add(source_id)
        used_targets.add(target_id)
        forked.update((find(source_id), find(target_id)))
        added.append({
            "source_id": source_id,
            "target_id": target_id,
            "edge_prob": None,
            "distance_um": edge_distance_um(
                nodes_by_id[source_id], nodes_by_id[target_id]
            ),
            "addonly_division": 1,
        })
    stats["addonly_divisions_added"] = len(added)
    return [*edges, *added] if added else edges


# ---------------------------------------------------------------------------
# Geometry-only division fallback (used for the anytime baseline shard)
# ---------------------------------------------------------------------------
def add_safe_divisions_postlink(nodes_by_id, edges, stats):
    if not SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges

    out_by_source: dict[int, list[dict]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))

    ids_by_t = _ids_by_frame(nodes_by_id)
    existing_edges = {(int(e["source_id"]), int(e["target_id"])) for e in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict] = []
    used_targets: set[int] = set()
    used_sources: set[int] = set()

    for t in sorted(ids_by_t):
        if t + 1 not in ids_by_t:
            continue
        source_ids = [i for i in ids_by_t[t] if len(out_by_source.get(i, ())) == 1]
        candidate_ids = [
            i for i in ids_by_t[t + 1] if i not in incoming and i not in used_targets
        ]
        if not source_ids or not candidate_ids:
            continue

        candidate_coords = _coords_um(nodes_by_id, candidate_ids)
        candidate_tree = cKDTree(candidate_coords)
        source_coords = _coords_um(nodes_by_id, source_ids)
        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float]] = []

        for index, source_id in enumerate(source_ids):
            existing_child_id = int(out_by_source[source_id][0]["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_position = np.asarray(node_point(existing_child)) * SCALE
            if float(np.linalg.norm(child_position - source_coords[index])) > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
            near = candidate_tree.query_ball_point(source_coords[index], SAFE_DIV_MAX_UM)
            if not near:
                continue
            near = np.asarray(near, dtype=np.int64)
            parent_distance = np.linalg.norm(candidate_coords[near] - source_coords[index], axis=1)
            sister_distance = np.linalg.norm(candidate_coords[near] - child_position, axis=1)
            keep = (parent_distance <= SAFE_DIV_MAX_UM) & (sister_distance <= SAFE_DIV_SISTER_MAX_UM)
            for offset in np.flatnonzero(keep):
                candidate_id = candidate_ids[int(near[offset])]
                if (source_id, candidate_id) in existing_edges:
                    continue
                proposals.append((
                    float(parent_distance[offset])
                    + SAFE_DIV_SISTER_SCORE_WEIGHT * float(sister_distance[offset]),
                    source_id,
                    candidate_id,
                    float(parent_distance[offset]),
                ))

        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_distance in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            if source_id in used_sources:
                continue
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_distance,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            used_sources.add(source_id)
            added_this_frame += 1

    if not added:
        return edges
    stats["safe_divisions_added"] = len(added)
    return [*edges, *added]


# ---------------------------------------------------------------------------
# Component filtering and trajectory smoothing
# ---------------------------------------------------------------------------
def filter_short_track_components(nodes_by_id, edges, stats):
    if not FILTER_SHORT_TRACKS or MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        if source_id in parent and target_id in parent:
            root_a, root_b = find(source_id), find(target_id)
            if root_a != root_b:
                parent[root_a] = root_b
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    t_min_global = t_max_global = None
    if BOUNDARY_TRACK_RESCUE and nodes_by_id:
        boundary_times = [int(node["t"]) for node in nodes_by_id.values()]
        t_min_global = min(boundary_times)
        t_max_global = max(boundary_times)

    keep: set[int] = set()
    for members in components.values():
        divides = KEEP_DIVISION_COMPONENTS and any(out_count.get(i, 0) >= 2 for i in members)
        is_long_enough = len(members) >= MIN_TRACK_LEN
        is_boundary_truncated = False
        if (
            BOUNDARY_TRACK_RESCUE
            and not is_long_enough
            and not divides
            and len(members) >= BOUNDARY_TRACK_MIN_LEN
        ):
            member_times = [int(nodes_by_id[node_id]["t"]) for node_id in members]
            span = max(member_times) - min(member_times) + 1
            touches_start = min(member_times) == t_min_global
            touches_end = max(member_times) == t_max_global
            is_boundary_truncated = span < MIN_TRACK_LEN and (touches_start or touches_end)

        if is_long_enough or divides or is_boundary_truncated:
            keep.update(members)
            if is_boundary_truncated:
                stats["boundary_track_rescued_components"] += 1
                stats["boundary_track_rescued_nodes"] += len(members)

    if not keep or len(keep) == len(nodes_by_id):
        return nodes_by_id, edges

    kept_nodes = {i: node for i, node in nodes_by_id.items() if i in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(
        1 for members in components.values() if not (set(members) & keep)
    )
    stats["short_track_nodes_removed"] = len(nodes_by_id) - len(kept_nodes)
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def enforce_node_budget(nodes_by_id, edges, stats, detected_nodes: int):
    """Trim the weakest components when post-processing inflated the node count.

    The adjusted edge Jaccard scales the raw Jaccard by
    ``1 - alpha * (T_pred - T_true) / T_true``, so every node the pipeline adds
    beyond what the detector actually found is paid for at the full weight of
    the dominant metric term. The detector count is the only unbiased estimate
    of ``T_true`` available at test time, so it is the budget. Components that
    carry a fork are never dropped: a division is worth more than the handful
    of nodes it costs.
    """
    if not NODE_BUDGET or detected_nodes <= 0 or not nodes_by_id:
        return nodes_by_id, edges

    budget = int(round(detected_nodes * NODE_BUDGET_RATIO))
    excess = len(nodes_by_id) - budget
    stats["node_budget_excess"] = int(max(0, excess))
    if excess <= 0:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        if source_id in parent and target_id in parent:
            root_a, root_b = find(source_id), find(target_id)
            if root_a != root_b:
                parent[root_a] = root_b
        out_count[source_id] = out_count.get(source_id, 0) + 1

    members: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        members.setdefault(find(node_id), []).append(node_id)

    probability_sum: dict[int, float] = {}
    probability_count: dict[int, int] = {}
    for edge in edges:
        root = find(int(edge["source_id"]))
        value = edge.get("edge_prob")
        try:
            value = float(value) if value is not None else 0.0
        except (TypeError, ValueError):
            value = 0.0
        probability_sum[root] = probability_sum.get(root, 0.0) + value
        probability_count[root] = probability_count.get(root, 0) + 1

    ranked = sorted(
        (
            (
                any(out_count.get(node_id, 0) >= 2 for node_id in group),
                len(group),
                probability_sum.get(root, 0.0) / max(probability_count.get(root, 0), 1),
                root,
            )
            for root, group in members.items()
        )
    )

    max_drop = int(round(len(nodes_by_id) * NODE_BUDGET_MAX_DROP_FRAC))
    dropped: set[int] = set()
    for has_fork, size, _probability, root in ranked:
        if has_fork or len(dropped) >= excess or len(dropped) + size > max_drop:
            continue
        dropped.update(members[root])
    if not dropped:
        stats["node_budget_dropped_nodes"] = 0
        return nodes_by_id, edges

    kept_nodes = {i: node for i, node in nodes_by_id.items() if i not in dropped}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["node_budget_dropped_nodes"] = len(dropped)
    stats["node_budget_dropped_edges"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(nodes_by_id, edges, stats):
    """Blend each node towards a local linear fit; topology is untouched.

    Neighbourhoods only extend through unambiguous single links, so with a
    window of W there are at most (W + 1)^2 distinct shapes. Nodes are grouped
    by shape and each group is fitted in one batched least-squares.
    """
    if not LINEFIT_SMOOTH or LINEFIT_WEIGHT <= 0 or LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    # `None` marks "not exactly one link", which stops a walk just like a gap.
    predecessor: dict[int, int | None] = {}
    successor: dict[int, int | None] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None or int(target["t"]) != int(source["t"]) + 1:
            continue
        successor[source_id] = None if source_id in successor else target_id
        predecessor[target_id] = None if target_id in predecessor else source_id

    ids = sorted(nodes_by_id)
    index_of = {node_id: index for index, node_id in enumerate(ids)}
    coords = np.empty((len(ids), 3), dtype=np.float64)
    for index, node_id in enumerate(ids):
        node = nodes_by_id[node_id]
        coords[index] = (node["z"], node["y"], node["x"])

    shapes: dict[tuple[int, int], list[list[int]]] = {}
    for index, node_id in enumerate(ids):
        row = [index]
        counts = []
        for links in (predecessor, successor):
            current = node_id
            steps = 0
            while steps < LINEFIT_WINDOW:
                nearby = links.get(current)
                if nearby is None or nearby not in index_of:
                    break
                current = nearby
                row.append(index_of[current])
                steps += 1
            counts.append(steps)
        if len(row) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue
        shapes.setdefault((counts[0], counts[1]), []).append(row)

    weight = float(np.clip(LINEFIT_WEIGHT, 0.0, 1.0))
    # Every fit reads the original positions; results are applied at the end.
    smoothed = coords.copy()
    touched: list[np.ndarray] = []
    for (back, forward), rows in shapes.items():
        offsets = np.array(
            [0, *range(-1, -back - 1, -1), *range(1, forward + 1)], dtype=np.float64
        )
        centered = offsets - offsets.mean()
        denominator = float(centered @ centered)
        if denominator <= 0.0:
            stats["linefit_skipped_nodes"] += len(rows)
            continue
        members = np.asarray(rows, dtype=np.int64)
        window = coords[members]                                # (m, k, 3)
        mean = window.mean(axis=1)
        # Closed-form degree-1 least squares evaluated at offset 0.
        slope = np.einsum("k,mkc->mc", centered, window - mean[:, None, :]) / denominator
        fitted = mean - slope * offsets.mean()
        usable = np.isfinite(fitted).all(axis=1)
        stats["linefit_skipped_nodes"] += int((~usable).sum())
        targets = members[usable, 0]
        smoothed[targets] = (1.0 - weight) * coords[targets] + weight * fitted[usable]
        touched.append(targets)

    updated = np.concatenate(touched) if touched else np.empty(0, dtype=np.int64)
    for index in updated:
        node = nodes_by_id[ids[int(index)]]
        point = smoothed[int(index)]
        node["z"], node["y"], node["x"] = float(point[0]), float(point[1]), float(point[2])
    stats["linefit_smoothed_nodes"] = int(updated.size)
    return nodes_by_id


# ---------------------------------------------------------------------------
# Division decoder (Model C + division GBM + source-cardinality head)
# ---------------------------------------------------------------------------
DIVISION_RUNTIME = None
EDGEGRAFT_RUNTIME = None
CANDIDATEGRAFT_RUNTIME = None
BASEGRAPH_REPAIR_RUNTIME = None


def _load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Unable to load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def init_basegraph_repair_runtime():
    """Load the deterministic upstream graph policy once before worker fork."""
    global BASEGRAPH_REPAIR_RUNTIME
    if BASEGRAPH_REPAIR_RUNTIME is not None or not BASEGRAPH_REPAIR_ENABLED:
        return BASEGRAPH_REPAIR_RUNTIME
    module = _load_module(
        "biohub_basegraph_candidate_fusion_runtime",
        BASEGRAPH_REPAIR_DIR / "basegraph_candidate_fusion_runtime_v1.py",
    )
    BASEGRAPH_REPAIR_RUNTIME = module.BaseGraphCandidateFusionRuntime()
    return BASEGRAPH_REPAIR_RUNTIME


def init_edgegraft_runtime():
    """Load exact-parity V3 CPU gates once; forked workers inherit them."""
    global EDGEGRAFT_RUNTIME
    if EDGEGRAFT_RUNTIME is not None or not EDGEGRAFT_ENABLED:
        return EDGEGRAFT_RUNTIME
    edgegraft_dir = str(EDGEGRAFT_DIR)
    if edgegraft_dir not in sys.path:
        sys.path.insert(0, edgegraft_dir)
    module = _load_module(
        "biohub_edgegraft_v3_runtime",
        EDGEGRAFT_DIR / "edgegraft_v3_deploy_runtime_v5.py",
    )
    EDGEGRAFT_RUNTIME = module.EdgeGraftV3Runtime(
        EDGEGRAFT_DIR / "ranker", EDGEGRAFT_DIR / "metric",
    )
    return EDGEGRAFT_RUNTIME


def init_candidategraft_runtime():
    """Load the final known-only add-only gate once before worker fork."""
    global CANDIDATEGRAFT_RUNTIME
    if CANDIDATEGRAFT_RUNTIME is not None or not CANDIDATEGRAFT_ENABLED:
        return CANDIDATEGRAFT_RUNTIME
    module = _load_module(
        "biohub_candidategraft_direct_runtime",
        CANDIDATEGRAFT_DIR / "candidategraft_direct_runtime_v1.py",
    )
    CANDIDATEGRAFT_RUNTIME = module.CandidateGraftDirectRuntime(
        CANDIDATEGRAFT_DIR, EDGEGRAFT_DIR,
    )
    return CANDIDATEGRAFT_RUNTIME


def init_division_runtime():
    """Load the decoder once in the parent; forked workers inherit it."""
    global DIVISION_RUNTIME
    if DIVISION_RUNTIME is not None or not COMBINED_DECODER:
        return DIVISION_RUNTIME

    gbm = _load_module(
        "biohub_division_gbm_runtime", MODEL_C_DIR / "division_gbm_runtime.py"
    ).DivisionGBMRuntime(MODEL_C_DIR)
    legacy = _load_module(
        "biohub_model_c_safe_runtime", MODEL_C_DIR / "model_c_safe_runtime.py"
    ).CombinedModelCPrimaryRuntime(gbm, MODEL_C_DIR, threshold=MODEL_C_THRESHOLD)

    if not SOURCE_CARDINALITY_ENABLED:
        DIVISION_RUNTIME = legacy
        return DIVISION_RUNTIME

    if str(SOURCE_CARDINALITY_DIR) not in sys.path:
        sys.path.insert(0, str(SOURCE_CARDINALITY_DIR))
    primary_runtime = _load_module(
        "biohub_source_cardinality_graph_runtime",
        SOURCE_CARDINALITY_DIR / "source_cardinality_graph_runtime.py",
    ).SourceCardinalityGraphRuntime(
        gbm, SOURCE_CARDINALITY_DIR, MODEL_C_DIR, legacy_fallback_runtime=legacy
    )
    if not UNIGRAFT_P1P2_ENABLED:
        DIVISION_RUNTIME = primary_runtime
        return DIVISION_RUNTIME

    runtime_path = UNIGRAFT_P1P2_DIR / "unigraft_p1p2_late_ensemble_runtime.py"
    head_path = UNIGRAFT_P1P2_DIR / "p1p2_only_source_cardinality_head.pt"
    if not runtime_path.is_file() or not head_path.is_file():
        raise FileNotFoundError(
            f"Incomplete independent UG2 dataset: runtime={runtime_path} head={head_path}"
        )
    module = _load_module("biohub_unigraft_p1p2_late_runtime", runtime_path)
    ug12_runtime = module.IndependentP1P2LateEnsembleRuntime(
        primary_runtime, UNIGRAFT_P1P2_DIR
    )
    if not LIVE_V2_BUNDLE_ENABLED:
        DIVISION_RUNTIME = ug12_runtime
        return DIVISION_RUNTIME

    helper_path = LIVE_V2_BUNDLE_DIR / "live_v2_global_bundle_runtime_v1.py"
    if not helper_path.is_file():
        raise FileNotFoundError(f"Missing live V2 global bundle runtime: {helper_path}")
    helper = _load_module("biohub_live_v2_global_bundle_runtime", helper_path)
    live_runtime = helper.LiveV2GlobalBundleRuntime(ug12_runtime)
    if not OWNERSHIP_FULL_ENABLED:
        DIVISION_RUNTIME = live_runtime
        return DIVISION_RUNTIME
    ownership_path = OWNERSHIP_FULL_DIR / "ownership_full_population_runtime_v1.py"
    if not ownership_path.is_file():
        raise FileNotFoundError(f"Missing full-population ownership runtime: {ownership_path}")
    ownership = _load_module("biohub_full_population_ownership_runtime", ownership_path)
    DIVISION_RUNTIME = ownership.FullPopulationOwnershipRuntime(
        live_runtime, OWNERSHIP_FULL_DIR
    )
    return DIVISION_RUNTIME


def _native_evidence_ready(dataset) -> bool:
    """Whether this video has its P1/P2/Model-C native evidence.

    A video that crosses the division-helper deadline is predicted without the
    helper models, so no .npz is written for it - all three are gated by the
    same flag in the predictor. The stages below refine a graph; none of them
    is a prerequisite for one, so a missing archive should cost those stages,
    not the video.
    """
    return dataset is not None and all(
        (root / f"{dataset}.npz").exists()
        for root in (P1_EVIDENCE_DIR, P2_EVIDENCE_DIR, MODEL_C_EVIDENCE_DIR)
    )


def _evidence(root: Path, dataset: str) -> Path:
    path = root / f"{dataset}.npz"
    if not path.exists():
        raise FileNotFoundError(f"Missing native evidence: {path}")
    return path


def apply_division_decoder(dataset, nodes_by_id, edges, stats):
    if DIVISION_RUNTIME is None:
        init_division_runtime()
    return DIVISION_RUNTIME.apply(
        TEST_DIR / f"{dataset}.zarr",
        _evidence(MODEL_C_EVIDENCE_DIR, dataset),
        _evidence(P1_EVIDENCE_DIR, dataset),
        _evidence(P2_EVIDENCE_DIR, dataset),
        nodes_by_id,
        edges,
        stats,
        spacing=VOXEL_SCALE_UM,
        v2_threshold=GBM_THRESHOLD,
        v2_rescue_delta=GBM_RESCUE_DELTA,
        v2_steal_delta=GBM_STEAL_DELTA,
    )


# ---------------------------------------------------------------------------
# Post-EdgeGRAFT cleanup and final validation
# ---------------------------------------------------------------------------


def cleanup_after_edgegraft(nodes_by_id, edges, stats):
    """Reassert the continuation contract before destructive graph cleanup.

    Division sources are allowed to use the wider UG geometry. The 14 um
    continuation limit is applied only to one-child sources, so this pass does
    not silently delete an accepted UG1/UG2 fork.
    """
    source_degree = defaultdict(int)
    for edge in edges:
        source_degree[int(edge["source_id"])] += 1
    fork_sources = {source for source, degree in source_degree.items() if degree >= 2}

    by_pair: dict[tuple[int, int], dict] = {}
    for original in edges:
        edge = dict(original)
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            stats["post_edgegraft_dropped_dangling_edges"] += 1
            continue
        if ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["post_edgegraft_dropped_nonconsecutive_edges"] += 1
            continue
        distance = edge_distance_um(source, target)
        edge["distance_um"] = distance
        if EDGE_MAX_UM > 0 and distance > EDGE_MAX_UM:
            if source_id not in fork_sources:
                stats["post_edgegraft_dropped_long_continuations"] += 1
                continue
            stats["post_edgegraft_kept_long_division_edges"] += 1
        pair = (source_id, target_id)
        previous = by_pair.get(pair)
        if previous is None or edge_sort_key(edge) > edge_sort_key(previous):
            by_pair[pair] = edge
        else:
            stats["post_edgegraft_dropped_duplicate_edges"] += 1

    cleaned = list(by_pair.values())
    best_by_target: dict[int, dict] = {}
    for edge in cleaned:
        target_id = int(edge["target_id"])
        previous = best_by_target.get(target_id)
        if previous is None or edge_sort_key(edge) > edge_sort_key(previous):
            best_by_target[target_id] = edge
    if len(best_by_target) != len(cleaned):
        stats["post_edgegraft_dropped_multi_parent_edges"] += (
            len(cleaned) - len(best_by_target)
        )
    cleaned = list(best_by_target.values())

    incoming = defaultdict(int)
    outgoing = defaultdict(int)
    for edge in cleaned:
        incoming[int(edge["target_id"])] += 1
        outgoing[int(edge["source_id"])] += 1
    if any(value > 1 for value in incoming.values()):
        raise RuntimeError("Post-EdgeGRAFT cleanup left in-degree > 1")
    if any(value > 2 for value in outgoing.values()):
        raise RuntimeError("Post-EdgeGRAFT cleanup left out-degree > 2")
    stats["post_edgegraft_contract_edges"] = len(cleaned)
    return cleaned


def validate_final_graph_contract(nodes_by_id, edges, stats):
    """Validate the written topology and report distances after line fitting."""
    incoming = defaultdict(int)
    outgoing = defaultdict(int)
    final_long_edges = 0
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            raise RuntimeError("Final graph contains a dangling edge")
        if ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            raise RuntimeError("Final graph contains a nonconsecutive edge")
        incoming[target_id] += 1
        outgoing[source_id] += 1
        final_long_edges += int(
            EDGE_MAX_UM > 0 and edge_distance_um(source, target) > EDGE_MAX_UM
        )
    if any(value > 1 for value in incoming.values()):
        raise RuntimeError("Final graph contains in-degree > 1")
    if any(value > 2 for value in outgoing.values()):
        raise RuntimeError("Final graph contains out-degree > 2")
    stats["final_edges_over_max_um_after_linefit"] = final_long_edges

# ---------------------------------------------------------------------------
# Full graph filter
# ---------------------------------------------------------------------------
def filter_output_graph(
    nodes_by_id, raw_edges, dataset=None, division_mode="combined",
    raw_node_attrs=None, raw_edge_attrs=None,
):
    stats = defaultdict(int, BASE_STATS)
    evidence_ready = division_mode != "combined" or _native_evidence_ready(dataset)
    if division_mode == "combined" and not evidence_ready:
        # Past the helper deadline. Keep the combined graph and skip only the
        # stages that read the archives; the alternative is losing the video
        # to the geometry-only baseline shard.
        stats["native_evidence_missing"] = 1
    if division_mode == "combined" and BASEGRAPH_REPAIR_ENABLED and evidence_ready:
        if dataset is None:
            raise RuntimeError("BaseGraph repair needs a dataset id")
        if BASEGRAPH_REPAIR_RUNTIME is None:
            init_basegraph_repair_runtime()
        basegraph_started = time.monotonic()
        nodes_by_id, raw_edges, basegraph_stats = BASEGRAPH_REPAIR_RUNTIME.apply(
            nodes_by_id,
            raw_edges,
            _evidence(P1_EVIDENCE_DIR, dataset),
            _evidence(P2_EVIDENCE_DIR, dataset),
        )
        stats.update({f"basegraph_{key}": value for key, value in basegraph_stats.items()})
        stats["basegraph_seconds"] = time.monotonic() - basegraph_started
        stats["basegraph_upstream_candidate_fusion"] = 1
        raw_node_attrs = {node_id: dict(node) for node_id, node in nodes_by_id.items()}
        raw_edge_attrs = [dict(edge) for edge in raw_edges]
    else:
        stats["basegraph_skipped_for_fallback"] = int(BASEGRAPH_REPAIR_ENABLED)
    stats["raw_edges"] = len(raw_edges)
    if STAGE_TIMING and "basegraph_seconds" in stats:
        stats["t_basegraph"] = float(stats["basegraph_seconds"])
    _stage_t = time.monotonic()
    detected_nodes = len(nodes_by_id)
    if raw_node_attrs is None:
        raw_node_attrs = {node_id: dict(node) for node_id, node in nodes_by_id.items()}
    if raw_edge_attrs is None:
        raw_edge_attrs = [dict(edge) for edge in raw_edges]

    edges: list[dict] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance = edge_distance_um(source, target)
        edge["distance_um"] = distance
        if EDGE_MAX_UM > 0 and distance > EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge)

    _stage_t = _mark(stats, "edge_prefilter", _stage_t)
    if MOTION_RELINK:
        motion_edges = motion_relink_edges(nodes_by_id, stats, _sanitized_edge_probs(edges))
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1

    _stage_t = _mark(stats, "motion_relink", _stage_t)
    if SINGLE_PARENT_REPAIR and edges:
        best_by_target: dict[int, dict] = {}
        for edge in edges:
            target_id = int(edge["target_id"])
            previous = best_by_target.get(target_id)
            if previous is None or edge_sort_key(edge) > edge_sort_key(previous):
                best_by_target[target_id] = edge
        kept = {id(edge) for edge in best_by_target.values()}
        stats["dropped_multi_parent_edges"] = len(edges) - len(kept)
        edges = [edge for edge in edges if id(edge) in kept]

    _stage_t = _mark(stats, "single_parent", _stage_t)
    nodes_by_id, edges = close_single_frame_gaps(
        nodes_by_id,
        edges,
        stats,
        dataset=dataset,
        frame_cache={},
        deepcenter_enabled=(division_mode == "combined" and DEEPCENTER_GAP_VETO),
    )

    _stage_t = _mark(stats, "gap_close", _stage_t)
    if division_mode == "combined" and COMBINED_DECODER and evidence_ready:
        if dataset is None:
            raise RuntimeError("The combined decoder needs a dataset id")
        edges = apply_division_decoder(dataset, nodes_by_id, edges, stats)
    else:
        edges = add_safe_divisions_postlink(nodes_by_id, edges, stats)

    _stage_t = _mark(stats, "division", _stage_t)
    # EdgeGRAFT is an upgrade-only ownership stage. It runs exactly once,
    # after UG1/UG2 has installed protected forks and before any destructive
    # component cleanup or line-fit smoothing.
    if division_mode == "combined" and EDGEGRAFT_ENABLED and evidence_ready:
        if dataset is None:
            raise RuntimeError("EdgeGRAFT needs a dataset id")
        if EDGEGRAFT_RUNTIME is None:
            init_edgegraft_runtime()
        edgegraft_started = time.monotonic()
        edges, edgegraft_stats = EDGEGRAFT_RUNTIME.apply(
            dataset, raw_node_attrs, raw_edge_attrs, nodes_by_id, edges,
            _evidence(P1_EVIDENCE_DIR, dataset),
            _evidence(P2_EVIDENCE_DIR, dataset),
        )
        stats.update({
            f"edgegraft_{key}": value
            for key, value in edgegraft_stats.items()
        })
        stats["edgegraft_seconds"] = time.monotonic() - edgegraft_started
        stats["edgegraft_v3_full_population_atomic_upgrade"] = 1
        edges = cleanup_after_edgegraft(nodes_by_id, edges, stats)
    else:
        stats["edgegraft_skipped_for_fallback"] = int(EDGEGRAFT_ENABLED)

    _stage_t = _mark(stats, "edgegraft", _stage_t)
    if PRUNE_ISOLATED:
        incident = {int(e["source_id"]) for e in edges} | {int(e["target_id"]) for e in edges}
        if incident:
            kept_nodes = {i: node for i, node in nodes_by_id.items() if i in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [
                edge for edge in edges
                if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id
            ]

    _stage_t = _mark(stats, "prune", _stage_t)
    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    nodes_by_id, edges = enforce_node_budget(nodes_by_id, edges, stats, detected_nodes)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)
    _stage_t = _mark(stats, "component_filters", _stage_t)

    # Exact replay placement: after all destructive cleanup on the same final
    # retained-node population used by the 175-video promotion gate.
    if division_mode == "combined" and CANDIDATEGRAFT_ENABLED and evidence_ready:
        if dataset is None:
            raise RuntimeError("CandidateGRAFT needs a dataset id")
        if CANDIDATEGRAFT_RUNTIME is None:
            init_candidategraft_runtime()
        candidate_started = time.monotonic()
        edges, candidate_stats = CANDIDATEGRAFT_RUNTIME.apply(
            dataset, raw_node_attrs, raw_edge_attrs, nodes_by_id, edges,
            _evidence(P1_EVIDENCE_DIR, dataset),
            _evidence(P2_EVIDENCE_DIR, dataset),
        )
        stats.update({
            f"candidategraft_{key}": value
            for key, value in candidate_stats.items()
        })
        stats["candidategraft_seconds"] = time.monotonic() - candidate_started
        stats["candidategraft_direct_add_only"] = 1
    else:
        stats["candidategraft_skipped_for_fallback"] = int(CANDIDATEGRAFT_ENABLED)
    _stage_t = _mark(stats, "candidategraft", _stage_t)
    edges = add_only_divisions(nodes_by_id, edges, stats)
    _stage_t = _mark(stats, "addonly_divisions", _stage_t)
    validate_final_graph_contract(nodes_by_id, edges, stats)
    return nodes_by_id, edges, dict(stats)


# ---------------------------------------------------------------------------
# geff -> submission shard
# ---------------------------------------------------------------------------
def _columnar_nodes_edges(graph):
    """Same dicts as the per-row reader, built one column at a time.

    `iter_rows(named=True)` materialises a Python dict per row inside polars
    and then the caller rebuilds it; on a 71k-node graph that is 140k throwaway
    dicts. Reading each column once and zipping the Python scalars produces
    identical values. Returns None if the table does not look as expected, so
    the frozen reader stays the fallback.
    """
    try:
        node_table = graph.node_attrs()
        edge_table = graph.edge_attrs()
        node_ids = [int(value) for value in node_table["node_id"].to_list()]
        frames = [int(value) for value in node_table["t"].to_list()]
        depths = [float(value) for value in node_table["z"].to_list()]
        rows = [float(value) for value in node_table["y"].to_list()]
        columns = [float(value) for value in node_table["x"].to_list()]
        sources = [int(value) for value in edge_table["source_id"].to_list()]
        targets = [int(value) for value in edge_table["target_id"].to_list()]
        if "edge_prob" in list(edge_table.columns):
            probabilities = [
                None if value is None else float(value)
                for value in edge_table["edge_prob"].to_list()
            ]
        else:
            probabilities = [None] * len(sources)
    except Exception:
        return None

    nodes = {
        node_id: {"node_id": node_id, "t": t, "z": z, "y": y, "x": x}
        for node_id, t, z, y, x in zip(node_ids, frames, depths, rows, columns)
    }
    edges = [
        {"source_id": source, "target_id": target, "edge_prob": probability}
        for source, target, probability in zip(sources, targets, probabilities)
    ]
    return nodes, edges


def graph_nodes_edges(path: Path):
    import tracksdata as td

    graph = td.graph.IndexedRXGraph.from_geff(path)
    if isinstance(graph, tuple):
        graph = graph[0]

    if FAST_GEFF_READER:
        result = _columnar_nodes_edges(graph)
        if result is not None:
            return result

    nodes: dict[int, dict] = {}
    for row in graph.node_attrs().iter_rows(named=True):
        node_id = int(row["node_id"])
        nodes[node_id] = {
            "node_id": node_id,
            "t": int(row["t"]),
            "z": float(row["z"]),
            "y": float(row["y"]),
            "x": float(row["x"]),
        }
    edges = []
    for row in graph.edge_attrs().iter_rows(named=True):
        prob = row.get("edge_prob")
        edges.append({
            "source_id": int(row["source_id"]),
            "target_id": int(row["target_id"]),
            "edge_prob": None if prob is None else float(prob),
        })
    return nodes, edges


def write_dataset_shard(dataset: str, nodes_by_id: dict, edges: list[dict]):
    destination = SHARD_DIR / f"{dataset}.csv"
    temporary = destination.with_suffix(".csv.tmp")
    row_id = 0
    out_degree: dict[int, int] = {}

    # csv.DictWriter builds and formats a dict per row; 140k of those is a
    # measurable slice of a shard write. The fast path emits the same bytes --
    # same column order, same CRLF terminator, same int formatting -- and is
    # skipped outright if the dataset id would need quoting.
    if FAST_SHARD_WRITER and not any(mark in dataset for mark in ',"\r\n'):
        lines = [",".join(CSV_COLUMNS)]
        for node_id in sorted(nodes_by_id):
            node = nodes_by_id[node_id]
            lines.append(
                f"{row_id},{dataset},node,{int(node['node_id'])},{int(node['t'])},"
                f"{max(0, int(round(float(node['z']))))},"
                f"{max(0, int(round(float(node['y']))))},"
                f"{max(0, int(round(float(node['x']))))},-1,-1"
            )
            row_id += 1
        for edge in edges:
            source_id = int(edge["source_id"])
            target_id = int(edge["target_id"])
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            lines.append(
                f"{row_id},{dataset},edge,-1,-1,-1,-1,-1,{source_id},{target_id}"
            )
            row_id += 1
            out_degree[source_id] = out_degree.get(source_id, 0) + 1
        with temporary.open("w", newline="") as handle:
            handle.write("\r\n".join(lines))
            handle.write("\r\n")
        temporary.replace(destination)
        return len(nodes_by_id), len(edges), sum(v >= 2 for v in out_degree.values())

    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        for node_id in sorted(nodes_by_id):
            node = nodes_by_id[node_id]
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(node["node_id"]),
                "t": int(node["t"]),
                "z": max(0, int(round(float(node["z"])))),
                "y": max(0, int(round(float(node["y"])))),
                "x": max(0, int(round(float(node["x"])))),
                "source_id": -1,
                "target_id": -1,
            })
            row_id += 1
        for edge in edges:
            source_id = int(edge["source_id"])
            target_id = int(edge["target_id"])
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": source_id,
                "target_id": target_id,
            })
            row_id += 1
            out_degree[source_id] = out_degree.get(source_id, 0) + 1
    temporary.replace(destination)
    return len(nodes_by_id), len(edges), sum(v >= 2 for v in out_degree.values())


def emergency_shard(geff_path: Path) -> dict:
    """Last-resort shard built straight from the producer graph.

    Reachable only when the combined upgrade and the baseline pass have both
    failed for a video. It enforces nothing but the submission contract --
    consecutive frames, one parent per node, at most two children per node --
    because a degraded shard still scores and a missing one makes the whole
    submission unassemblable.
    """
    dataset = Path(geff_path).stem
    nodes_by_id, raw_edges = graph_nodes_edges(Path(geff_path))

    best_by_target: dict[int, dict] = {}
    for original in raw_edges:
        source = nodes_by_id.get(int(original["source_id"]))
        target = nodes_by_id.get(int(original["target_id"]))
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        edge = dict(original)
        edge["distance_um"] = edge_distance_um(source, target)
        if EDGE_MAX_UM > 0 and edge["distance_um"] > EDGE_MAX_UM:
            continue
        target_id = int(edge["target_id"])
        previous = best_by_target.get(target_id)
        if previous is None or edge_sort_key(edge) > edge_sort_key(previous):
            best_by_target[target_id] = edge

    by_source: dict[int, list[dict]] = {}
    for edge in best_by_target.values():
        by_source.setdefault(int(edge["source_id"]), []).append(edge)
    edges = []
    for group in by_source.values():
        group.sort(key=edge_sort_key, reverse=True)
        edges.extend(group[:2])

    incident = {int(e["source_id"]) for e in edges} | {int(e["target_id"]) for e in edges}
    nodes_by_id = {i: node for i, node in nodes_by_id.items() if i in incident}
    edges = [
        edge for edge in edges
        if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id
    ]
    if not nodes_by_id:
        raise AssertionError(f"{dataset}: emergency shard is empty")

    node_count, edge_count, divisions = write_dataset_shard(dataset, nodes_by_id, edges)
    return {
        "dataset": dataset,
        "raw_nodes": node_count,
        "nodes": node_count,
        "edges": edge_count,
        "division_like_sources": divisions,
        "edge_to_node_ratio": edge_count / max(node_count, 1),
        "division_mode": "emergency",
        "process_seconds": 0.0,
    }


def process_graph(geff_path: Path, division_mode: str) -> dict:
    """Post-process one prediction graph and atomically replace its shard."""
    import time

    dataset = Path(geff_path).stem
    read_started = time.monotonic()
    nodes_by_id, raw_edges = graph_nodes_edges(Path(geff_path))
    read_seconds = time.monotonic() - read_started
    if SKIP_REDUNDANT_RAW_COPIES:
        # filter_output_graph builds these itself, before it mutates anything,
        # and the BaseGraph repair overwrites both on the combined path. The
        # copies made here were never read.
        raw_node_attrs = None
        raw_edge_attrs = None
    else:
        raw_node_attrs = {node_id: dict(node) for node_id, node in nodes_by_id.items()}
        raw_edge_attrs = [dict(edge) for edge in raw_edges]
    raw_nodes = len(nodes_by_id)

    started = time.monotonic()
    nodes_by_id, edges, stats = filter_output_graph(
        nodes_by_id, raw_edges, dataset=dataset, division_mode=division_mode,
        raw_node_attrs=raw_node_attrs, raw_edge_attrs=raw_edge_attrs,
    )
    seconds = time.monotonic() - started
    if not nodes_by_id:
        raise AssertionError(f"{dataset}: post-processing removed every node")

    write_started = time.monotonic()
    node_count, edge_count, divisions = write_dataset_shard(dataset, nodes_by_id, edges)
    if STAGE_TIMING:
        stats["t_graph_read"] = read_seconds
        stats["t_shard_write"] = time.monotonic() - write_started
    stats.update({
        "dataset": dataset,
        "raw_nodes": raw_nodes,
        "nodes": node_count,
        "edges": edge_count,
        "division_like_sources": divisions,
        "edge_to_node_ratio": edge_count / max(node_count, 1),
        "division_mode": division_mode,
        "process_seconds": seconds,
    })
    return stats


## 3. Runtime accelerator

Hot helpers inside the frozen artifact runtimes are replaced only where the
replacement reproduces the original bit for bit on an adversarial battery.
Anything that fails stays frozen and is named in the log.


In [ ]:
%%writefile /kaggle/working/biohub_pipeline/runtime_accel.py
"""Verified drop-in replacements for the hot helpers inside frozen artifacts.

Nothing here changes what the pipeline computes. Every replacement reproduces
the original's arithmetic in the same order and at the same precision, and
every one is checked against the function it replaces -- on adversarial
inputs, requiring bit-identical results -- before it is installed. A helper
that does not match is left frozen and reported, so a different artifact
build degrades to the original speed instead of silently producing a
different graph.

The precision caveat is not hypothetical. `_augment_geometry` and
`_augment_trajectory` hold positions as float64, but the component runtime
holds them as float32, and `np.linalg.norm` on a float32 array does its sum
*and its square root* in float32. A float64 scalar rewrite is more accurate
and therefore wrong: it moved 7045 of 7309 candidate rows in the first
version of this file. Hence `_norm_like`, and hence a check battery that
covers both dtypes.

Profile that motivates this (EdgeGRAFT V3 apply() on a synthetic 71k-node
video, cProfile, one core):

    total                       233.2 s
      np.linalg.norm              61.2 s   8,121,934 calls on 3-vectors
      _augment_trajectory         80.2 s
      _rows (component)           57.3 s
      _model_context              38.6 s     246,844 calls
      _cosine                     34.2 s   1,649,521 calls
      _decisions                  31.3 s   pandas .loc per group
      _augment_geometry           22.8 s
      _linear_velocity            20.1 s     142,026 calls, 37% never read
      _rows_frame                 14.6 s
"""
from __future__ import annotations

import math

import numpy as np

INSTALLED: list[str] = []
REJECTED: list[str] = []
SKIPPED: list[str] = []

_F64 = np.dtype(np.float64)


# ---------------------------------------------------------------------------
# Norm primitives
#
# np.linalg.norm(v) on a real 1-D array is sqrt(v.dot(v)) evaluated in v's own
# dtype. Rounding a float64 square root down to float32 is correctly rounded
# (53 >= 2*24 + 2), so _norm_like reproduces it exactly for every real float
# width while skipping norm's ravel / complex / ord dispatch.
# ---------------------------------------------------------------------------
def _norm_like(vector):
    squared = vector.dot(vector)
    return squared.dtype.type(math.sqrt(float(squared)))


def _norm3(a: float, b: float, c: float) -> float:
    return math.sqrt(a * a + b * b + c * c)


def _is_f64_triple(vector) -> bool:
    return (
        type(vector) is np.ndarray
        and vector.dtype == _F64
        and vector.shape == (3,)
    )


# ---------------------------------------------------------------------------
# Verification harness
# ---------------------------------------------------------------------------
def _vectors(seed: int, count: int, dtype):
    rng = np.random.default_rng(seed)
    stack = np.vstack([
        rng.normal(0, 30, (count, 3)),
        rng.normal(0, 1e-7, (max(count // 4, 1), 3)),
        rng.normal(0, 1e7, (max(count // 4, 1), 3)),
        np.zeros((3, 3)),
        np.array([[3.0, 4.0, 0.0], [-0.0, 0.0, 0.0], [1e-30, 1e-30, 1e-30]]),
    ])
    return stack.astype(dtype)


def _identical(left, right) -> bool:
    """Exact equality, with NaN equal to NaN and dtype significant."""
    if isinstance(left, float) and isinstance(right, float):
        return left == right or (math.isnan(left) and math.isnan(right))
    left = np.asarray(left)
    right = np.asarray(right)
    if left.shape != right.shape or left.dtype != right.dtype:
        return False
    return bool(np.array_equal(left, right, equal_nan=True))


def _install(module, name: str, make, check) -> bool:
    """Swap module.name for make(original), but only if check() passes."""
    original = getattr(module, name, None)
    if original is None:
        SKIPPED.append(f"{module.__name__}.{name} (absent)")
        return False
    replacement = make(original)
    try:
        if not check(original, replacement):
            REJECTED.append(f"{module.__name__}.{name}")
            return False
    except Exception as error:
        REJECTED.append(f"{module.__name__}.{name} ({type(error).__name__}: {error})")
        return False
    setattr(module, name, replacement)
    INSTALLED.append(f"{module.__name__}.{name}")
    return True


def _check_pairs(original, replacement) -> bool:
    """Both dtypes, both orders, degenerate vectors, non-contiguous views."""
    for dtype in (np.float64, np.float32):
        left = _vectors(11, 2500, dtype)
        right = _vectors(12, 2500, dtype)
        for a, b in zip(left, right):
            if not _identical(original(a, b), replacement(a, b)):
                return False
        zero = np.zeros(3, dtype)
        for a, b in ((zero, right[0]), (left[0], zero), (zero, zero)):
            if not _identical(original(a, b), replacement(a, b)):
                return False
        strided = np.ascontiguousarray(left[:20]).T[:, ::2].T
        for a in strided:
            if not _identical(original(a, right[0]), replacement(a, right[0])):
                return False
    mixed_a = _vectors(13, 200, np.float64)
    mixed_b = _vectors(14, 200, np.float32)
    for a, b in zip(mixed_a, mixed_b):
        if not _identical(original(a, b), replacement(a, b)):
            return False
    # Sweep the denominator across every plausible guard epsilon. Without
    # this the probe below cannot tell a 1e-6 guard from a 1e-8 one: random
    # vectors never land in the window between them.
    for dtype in (np.float64, np.float32):
        for denominator in np.geomspace(1e-14, 1e-2, 160):
            magnitude = float(np.sqrt(denominator))
            for a, b in (
                (np.array([magnitude, 0.0, 0.0], dtype),
                 np.array([magnitude, 0.0, 0.0], dtype)),
                (np.array([magnitude, 0.0, 0.0], dtype),
                 np.array([0.0, magnitude, 0.0], dtype)),
                (np.array([magnitude / 3.0, magnitude, -magnitude], dtype),
                 np.array([magnitude, magnitude / 7.0, magnitude], dtype)),
            ):
                if not _identical(original(a, b), replacement(a, b)):
                    return False
    # Ratios far outside the +-4 clip, for the speed-ratio contract.
    for dtype in (np.float64, np.float32):
        for scale in (1e-6, 1e-3, 1.0, 1e3, 1e6):
            a = np.array([1.0, 2.0, 3.0], dtype)
            b = (a * scale).astype(dtype)
            if not _identical(original(a, b), replacement(a, b)):
                return False
            if not _identical(original(b, a), replacement(b, a)):
                return False
    return True


# ---------------------------------------------------------------------------
# _cosine
# ---------------------------------------------------------------------------
def _make_cosine(epsilon: float):
    def make(original):
        def _cosine(left, right) -> float:
            if _is_f64_triple(left) and _is_f64_triple(right):
                a0, a1, a2 = left.tolist()
                b0, b1, b2 = right.tolist()
                denominator = _norm3(a0, a1, a2) * _norm3(b0, b1, b2)
                if denominator > epsilon:
                    return (a0 * b0 + a1 * b1 + a2 * b2) / denominator
                return 0.0
            # Anything else keeps the original expression verbatim, with
            # only norm swapped out: on float32 inputs the final division is
            # a float32 division under NEP 50 weak promotion, and widening
            # first moves the result.
            try:
                denominator = float(_norm_like(left) * _norm_like(right))
            except Exception:
                return original(left, right)
            if denominator > epsilon:
                return float(np.dot(left, right) / denominator)
            return 0.0
        return _cosine
    return make


# ---------------------------------------------------------------------------
# _speed_ratio
# ---------------------------------------------------------------------------
def _make_speed_ratio(original):
    def _speed_ratio(first, second) -> float:
        if _is_f64_triple(first) and _is_f64_triple(second):
            a0, a1, a2 = first.tolist()
            b0, b1, b2 = second.tolist()
            value = math.log(
                (_norm3(b0, b1, b2) + 1e-3) / (_norm3(a0, a1, a2) + 1e-3)
            )
            return -4.0 if value < -4.0 else (4.0 if value > 4.0 else value)
        try:
            ratio = (_norm_like(second) + 1e-3) / (_norm_like(first) + 1e-3)
        except Exception:
            return original(first, second)
        return float(np.clip(np.log(ratio), -4, 4))
    return _speed_ratio


# ---------------------------------------------------------------------------
# _linear_velocity -- avoid np.stack over a list of dict lookups
# ---------------------------------------------------------------------------
def _make_linear_velocity(original):
    def _linear_velocity(chain, position, backward):
        length = len(chain)
        if length < 2:
            return np.zeros(3, np.float64), 0.0
        first = position[chain[0]]
        if type(first) is not np.ndarray or first.dtype != _F64 or first.shape != (3,):
            return original(chain, position, backward)
        times = np.arange(length, dtype=np.float64)
        if backward:
            times = -times
        values = np.empty((length, 3), np.float64)
        for index, node in enumerate(chain):
            values[index] = position[node]
        centered = times - times.mean()
        denominator = float(np.dot(centered, centered))
        mean = values.mean(axis=0)
        slope = (
            centered[:, None] * (values - mean)
        ).sum(axis=0) / max(denominator, 1e-8)
        fitted = mean + centered[:, None] * slope
        residual = float(np.sqrt(np.mean(np.sum((values - fitted) ** 2, axis=1))))
        return slope, residual
    return _linear_velocity


def _check_linear_velocity(original, replacement) -> bool:
    rng = np.random.default_rng(31)
    for dtype in (np.float64, np.float32):
        position = {index: rng.normal(0, 40, 3).astype(dtype) for index in range(60)}
        for length in range(0, 9):
            for _ in range(30):
                chain = rng.choice(60, size=length, replace=False).tolist()
                for backward in (True, False):
                    a_slope, a_fit = original(chain, position, backward)
                    b_slope, b_fit = replacement(chain, position, backward)
                    if not _identical(a_slope, b_slope) or not _identical(a_fit, b_fit):
                        return False
    return True


# ---------------------------------------------------------------------------
# _rows_frame -- build the DataFrame column-wise instead of from N dicts
# ---------------------------------------------------------------------------
def _make_rows_frame(original):
    from operator import itemgetter

    import pandas as pd

    def _rows_frame(rows):
        if not rows:
            return original(rows)
        # Column order is the dict insertion order of the original record:
        # the four fixed keys, then the features in their own order.
        names = list(rows[0]["features"])
        reference = tuple(names)
        # One itemgetter call per row pulls all ~180 feature values at C
        # speed; a per-column pass costs 180 dict lookups per row instead.
        # The guard below is 0.3 s on 121k rows and makes the shortcut safe.
        if any(tuple(row["features"]) != reference for row in rows):
            return original(rows)
        count = len(rows)
        columns = {
            "component": np.fromiter(
                (int(r["component"]) for r in rows), np.int64, count),
            "source": np.fromiter(
                (int(r["source"]) for r in rows), np.int64, count),
            "target": np.fromiter(
                (int(r["target"]) for r in rows), np.int64, count),
            "native_probability": np.fromiter(
                (float(r["native_probability"]) for r in rows), np.float64, count),
        }
        if names:
            getter = itemgetter(*names)
            if len(names) == 1:
                block = np.fromiter(
                    (float(getter(r["features"])) for r in rows), np.float64, count
                ).reshape(count, 1)
            else:
                block = np.array(
                    [getter(r["features"]) for r in rows], dtype=np.float64
                )
            for index, name in enumerate(names):
                if name in columns:
                    return original(rows)
                columns[name] = block[:, index]
        return pd.DataFrame(
            columns, columns=["component", "source", "target",
                              "native_probability", *names]
        )
    return _rows_frame


def _check_rows_frame(original, replacement) -> bool:
    rng = np.random.default_rng(41)
    rows = []
    for _ in range(80):
        rows.append({
            "component": int(rng.integers(0, 9)),
            "source": int(rng.integers(1, 999)),
            "target": int(rng.integers(1, 999)),
            "native_probability": float(rng.random()),
            "features": {
                "alpha": float(rng.normal()), "beta": float(rng.normal()),
                "gamma": int(rng.integers(0, 3)), "delta": float("nan"),
                "epsilon": float(np.float32(rng.normal())),
            },
        })
    left = original(rows)
    right = replacement(rows)
    if list(left.columns) != list(right.columns) or len(left) != len(right):
        return False
    if not left.index.equals(right.index):
        return False
    for column in left.columns:
        if left[column].dtype != right[column].dtype:
            return False
        if not _identical(left[column].to_numpy(), right[column].to_numpy()):
            return False
    ragged = rows[:5] + [dict(rows[5], features={"alpha": 1.0})]
    if not list(original(ragged).columns) == list(replacement(ragged).columns):
        return False
    return bool(original([]).equals(replacement([])))




def _n64(vector) -> float:
    """float(np.linalg.norm(v)) for a float64 1-D array, without the dispatch."""
    return math.sqrt(float(vector.dot(vector)))


# ---------------------------------------------------------------------------
# _augment_geometry -- same statements, scalar norms, no per-row temp dict
# ---------------------------------------------------------------------------
def _make_augment_geometry(original):
    import edgegraft_v15_runtime_v1 as v15

    def _augment_geometry(rows, nodes, edges):
        if not rows or not nodes:
            return
        position = v15._position
        spacing = v15.SPACING
        cosine = v15._cosine
        incoming, outgoing = v15._edge_sets(edges)
        physical = {node: position(value) * spacing for node, value in nodes.items()}
        all_xyz = np.stack(list(physical.values()))
        lower = all_xyz.min(axis=0)
        span = np.maximum(all_xyz.max(axis=0) - lower, 1e-6)
        zeros = np.zeros(3, np.float64)
        for row in rows:
            source = int(row["source"])
            target = int(row["target"])
            source_xyz = physical.get(source)
            target_xyz = physical.get(target)
            if source_xyz is None or target_xyz is None:
                continue
            displacement = target_xyz - source_xyz
            distance = _n64(displacement)
            unit = displacement / max(distance, 1e-6)
            source_norm = (source_xyz - lower) / span
            target_norm = (target_xyz - lower) / span
            previous = next(iter(incoming[source])) if len(incoming[source]) == 1 else None
            following = next(iter(outgoing[target])) if len(outgoing[target]) == 1 else None
            in_vector = (
                source_xyz - physical[previous] if previous in physical else zeros
            )
            out_vector = (
                physical[following] - target_xyz if following in physical else zeros
            )
            values = row["features"]
            values["geom_dz_um"] = displacement[0]
            values["geom_dy_um"] = displacement[1]
            values["geom_dx_um"] = displacement[2]
            values["geom_unit_z"] = unit[0]
            values["geom_unit_y"] = unit[1]
            values["geom_unit_x"] = unit[2]
            values["geom_distance_um_exact"] = distance
            values["geom_source_z_norm"] = source_norm[0]
            values["geom_source_y_norm"] = source_norm[1]
            values["geom_source_x_norm"] = source_norm[2]
            values["geom_target_z_norm"] = target_norm[0]
            values["geom_target_y_norm"] = target_norm[1]
            values["geom_target_x_norm"] = target_norm[2]
            values["geom_incoming_valid"] = float(previous in physical)
            values["geom_outgoing_valid"] = float(following in physical)
            for prefix, vector in (("geom_incoming", in_vector),
                                   ("geom_outgoing", out_vector)):
                magnitude = _n64(vector)
                residual = displacement - vector
                values[prefix + "_dz_um"] = vector[0]
                values[prefix + "_dy_um"] = vector[1]
                values[prefix + "_dx_um"] = vector[2]
                values[prefix + "_speed_um"] = magnitude
                values[prefix + "_candidate_cosine"] = cosine(displacement, vector)
                values[prefix + "_residual_dz_um"] = residual[0]
                values[prefix + "_residual_dy_um"] = residual[1]
                values[prefix + "_residual_dx_um"] = residual[2]
                values[prefix + "_residual_um"] = _n64(residual)
    return _augment_geometry


# ---------------------------------------------------------------------------
# _augment_trajectory -- 37% of the velocity fits it computes are never read
# ---------------------------------------------------------------------------
def _make_augment_trajectory(original):
    import edgegraft_v15_runtime_v1 as v15

    horizon_keys = {}

    def _augment_trajectory(rows, nodes, edges):
        if not rows:
            return
        position_of = v15._position
        spacing = v15.SPACING
        cosine = v15._cosine
        linear_velocity = v15._linear_velocity
        horizon = v15.HORIZON
        keys = horizon_keys.get(horizon)
        if keys is None:
            keys = [
                (f"traj_forward_h{h}_valid", f"traj_forward_h{h}_residual",
                 f"traj_forward_h{h}_cosine", f"traj_reverse_h{h}_valid",
                 f"traj_reverse_h{h}_residual", f"traj_reverse_h{h}_cosine")
                for h in range(1, horizon + 1)
            ]
            horizon_keys[horizon] = keys

        position = {node: position_of(value) * spacing for node, value in nodes.items()}
        history, future = v15._chains(nodes, edges)
        history_velocity = {}
        history_fit = {}
        future_velocity = {}
        future_fit = {}
        # The original fits every node twice; a row only ever reads the history
        # of its source and the future of its target.
        need_history = set()
        need_future = set()
        for row in rows:
            need_history.add(int(row["source"]))
            need_future.add(int(row["target"]))
        for node in need_history:
            if node in position:
                history_velocity[node], history_fit[node] = linear_velocity(
                    history[node], position, True
                )
        for node in need_future:
            if node in position:
                future_velocity[node], future_fit[node] = linear_velocity(
                    future[node], position, False
                )

        zeros = np.zeros(3, np.float64)
        for row in rows:
            source = int(row["source"])
            target = int(row["target"])
            source_pos = position.get(source)
            target_pos = position.get(target)
            if source_pos is None or target_pos is None:
                continue
            displacement = target_pos - source_pos
            source_history = history[source]
            target_future = future[target]
            source_velocity = history_velocity[source]
            target_velocity = future_velocity[target]
            values = row["features"]
            values["traj_history_len"] = len(source_history) - 1
            values["traj_future_len"] = len(target_future) - 1
            values["traj_history_fit_residual"] = history_fit[source]
            values["traj_future_fit_residual"] = future_fit[target]
            values["traj_forward_linear_residual"] = _n64(
                target_pos - (source_pos + source_velocity)
            )
            values["traj_reverse_linear_residual"] = _n64(
                source_pos - (target_pos - target_velocity)
            )
            values["traj_velocity_agreement"] = _n64(source_velocity - target_velocity)
            values["traj_velocity_cosine"] = cosine(source_velocity, target_velocity)
            values["traj_candidate_history_cosine"] = cosine(displacement, source_velocity)
            values["traj_candidate_future_cosine"] = cosine(displacement, target_velocity)
            acceleration = zeros
            if len(source_history) >= 3:
                first = position[source_history[0]] - position[source_history[1]]
                previous = position[source_history[1]] - position[source_history[2]]
                acceleration = first - previous
            values["traj_acceleration_residual"] = _n64(
                target_pos - (source_pos + source_velocity + 0.5 * acceleration)
            )
            history_len = len(source_history)
            future_len = len(target_future)
            for offset, names in enumerate(keys, start=1):
                if history_len > offset:
                    velocity = (source_pos - position[source_history[offset]]) / offset
                    values[names[0]] = 1.0
                    values[names[1]] = _n64(target_pos - (source_pos + velocity))
                    values[names[2]] = cosine(displacement, velocity)
                else:
                    values[names[0]] = 0.0
                    values[names[1]] = 0.0
                    values[names[2]] = 0.0
                if future_len > offset:
                    velocity = (position[target_future[offset]] - target_pos) / offset
                    values[names[3]] = 1.0
                    values[names[4]] = _n64(source_pos - (target_pos - velocity))
                    values[names[5]] = cosine(displacement, velocity)
                else:
                    values[names[3]] = 0.0
                    values[names[4]] = 0.0
                    values[names[5]] = 0.0
    return _augment_trajectory


def _graph_fixture(seed: int):
    """A small graph with gaps, forks, and rows pointing at random pairs."""
    rng = np.random.default_rng(seed)
    frames, per_frame = 8, 9
    nodes, edges = {}, []
    node_id = 1
    previous_ids = []
    for frame in range(frames):
        current = []
        for _ in range(per_frame):
            nodes[node_id] = {
                "node_id": node_id, "t": frame,
                "z": float(rng.normal(30, 12)), "y": float(rng.normal(400, 90)),
                "x": float(rng.normal(400, 90)),
            }
            current.append(node_id)
            node_id += 1
        if previous_ids:
            for a, b in zip(previous_ids, current):
                if rng.random() < 0.85:
                    edges.append({"source_id": a, "target_id": b,
                                  "edge_prob": float(rng.random())})
            if rng.random() < 0.6:      # a fork, so some sources have 2 children
                edges.append({"source_id": previous_ids[0], "target_id": current[-1],
                              "edge_prob": float(rng.random())})
        previous_ids = current
    ids = sorted(nodes)
    rows = []
    for _ in range(150):
        rows.append({
            "component": int(rng.integers(0, 5)),
            "source": int(rng.choice(ids)), "target": int(rng.choice(ids)),
            "native_probability": float(rng.random()),
            "features": {"seed_feature": float(rng.random())},
        })
    return rows, nodes, edges


def _check_row_augmenter(original, replacement) -> bool:
    import copy

    for seed in range(5):
        rows, nodes, edges = _graph_fixture(seed)
        left = copy.deepcopy(rows)
        right = copy.deepcopy(rows)
        original(left, nodes, edges)
        replacement(right, nodes, edges)
        if len(left) != len(right):
            return False
        for a, b in zip(left, right):
            if list(a["features"].keys()) != list(b["features"].keys()):
                return False
            for key in a["features"]:
                if not _identical(float(a["features"][key]), float(b["features"][key])):
                    return False
    return True


# ---------------------------------------------------------------------------
# EdgeGraftV15Runtime._decisions -- one pandas groupby with a .loc scalar
# lookup per group becomes one stable argsort plus flat numpy slices.
# ---------------------------------------------------------------------------
def _make_decisions(original):
    import zlib

    import pandas as pd

    import edgegraft_v15_runtime_v1 as v15

    def _decisions(self, stem, frame, final_edges, protected):
        if frame.empty or "rank_score" not in frame:
            return original(self, stem, frame, final_edges, protected)
        scores = frame["rank_score"].to_numpy(np.float64)
        if not np.isfinite(scores).all():
            # idxmax skips NaN; do not reimplement that corner.
            return original(self, stem, frame, final_edges, protected)
        targets = frame["target"].to_numpy()
        sources = frame["source"].to_numpy()
        components = frame["component"].to_numpy()

        incoming, outgoing = v15._edge_sets(final_edges)
        # groupby(sort=False) yields groups in order of first appearance, and a
        # stable argsort puts each group's rows in their original order, so the
        # first element of a group block is that group's first appearance.
        order = np.argsort(targets, kind="stable")
        grouped = targets[order]
        starts = np.flatnonzero(
            np.concatenate(([True], grouped[1:] != grouped[:-1]))
        )
        ends = np.concatenate((starts[1:], [len(grouped)]))
        first_index = order[starts]
        fold = int(zlib.crc32(stem.encode()) % 5)

        records = []
        for block in np.argsort(first_index, kind="stable"):
            rows = order[starts[block]:ends[block]]
            target = int(targets[rows[0]])
            if len(incoming[target]) != 1:
                continue
            current_source = next(iter(incoming[target]))
            local = scores[rows]
            winner_row = int(rows[int(np.argmax(local))])
            source = int(sources[winner_row])
            if source == current_source:
                continue
            ordered = np.sort(local)
            runner_up = float(ordered[-2]) if ordered.size > 1 else 0.0
            baseline = local[sources[rows] == current_source]
            current_present = bool(baseline.size)
            current_score = float(baseline.max()) if current_present else np.nan
            source_target = (
                next(iter(outgoing[source])) if len(outgoing[source]) == 1 else -1
            )
            remove_count = 1 + int(source_target >= 0 and source_target != target)
            touched = {source, target, current_source}
            if source_target >= 0:
                touched.add(int(source_target))
            top_score = float(scores[winner_row])
            records.append({
                "dataset": stem, "fold": fold,
                "component": int(components[winner_row]), "source": source,
                "target": target, "current_source": current_source,
                "source_current_target": int(source_target),
                "topology": "2to1" if remove_count == 2 else "1to1",
                "remove_count": remove_count,
                "top_score": top_score,
                "runner_up_score": runner_up,
                "top_margin": float(top_score - runner_up),
                "current_present": current_present,
                "current_score": current_score,
                "advantage": (
                    float(top_score - current_score) if current_present else np.nan
                ),
                "protected": bool(touched & protected),
            })
        return pd.DataFrame(records)
    return _decisions


def _check_decisions(original, replacement) -> bool:
    import pandas as pd

    rng = np.random.default_rng(61)
    for seed in range(4):
        _rows, nodes, edges = _graph_fixture(seed)
        ids = sorted(nodes)
        count = 300
        frame = pd.DataFrame({
            "component": rng.integers(0, 6, count).astype(np.int64),
            "source": rng.choice(ids, count).astype(np.int64),
            "target": rng.choice(ids, count).astype(np.int64),
            "rank_score": rng.random(count),
            "native_probability": rng.random(count),
        })
        # duplicate scores and a duplicate (source, target) to exercise ties
        frame.loc[frame.index[:40], "rank_score"] = 0.5
        protected = {int(i) for i in rng.choice(ids, 5)}

        class _Fake:
            pass

        left = original(_Fake(), "6bba_05db0fb1", frame, edges, protected)
        right = replacement(_Fake(), "6bba_05db0fb1", frame, edges, protected)
        if list(left.columns) != list(right.columns) or len(left) != len(right):
            return False
        for column in left.columns:
            a = left[column].to_numpy()
            b = right[column].to_numpy()
            if a.dtype != b.dtype:
                return False
            if a.dtype.kind in "fc":
                if not np.array_equal(a, b, equal_nan=True):
                    return False
            elif not np.array_equal(a, b):
                return False
    return True


# ---------------------------------------------------------------------------
# _model_context -- 246k calls, each rebuilding 20 f-string keys
# ---------------------------------------------------------------------------
_ZERO_F32 = np.zeros(3, np.float32)
_CONTEXT_SUFFIXES = (
    "_prev_valid", "_prev_probability", "_prev_margin", "_prev_winner",
    "_prev_distance", "_future_valid", "_future_probability", "_future_margin",
    "_future_winner", "_future_distance", "_prev_current_residual",
    "_current_future_residual", "_prev_current_cosine",
    "_current_future_cosine", "_prev_current_speed_ratio",
    "_current_future_speed_ratio", "_path_min", "_path_mean",
    "_path_geomean", "_context_both",
)


def _make_model_context(original):
    import edgegraft_component_runtime_v1 as component

    key_cache: dict[str, tuple] = {}

    def _model_context(prefix, source, target, current_probability, nodes,
                       incoming, outgoing):
        keys = key_cache.get(prefix)
        if keys is None:
            keys = tuple(prefix + suffix for suffix in _CONTEXT_SUFFIXES)
            key_cache[prefix] = keys
        cosine = component._cosine
        speed_ratio = component._speed_ratio

        current = nodes[target] - nodes[source]
        previous = incoming.get(source)
        future = outgoing.get(target)
        previous_valid = float(previous is not None and previous[0] in nodes)
        future_valid = float(future is not None and future[1] in nodes)
        previous_delta = (
            nodes[source] - nodes[previous[0]] if previous_valid else _ZERO_F32
        )
        future_delta = (
            nodes[future[1]] - nodes[target] if future_valid else _ZERO_F32
        )
        previous_probability = previous[2] if previous_valid else 0.0
        future_probability = future[2] if future_valid else 0.0
        both = previous_valid and future_valid
        values = {
            keys[0]: previous_valid,
            keys[1]: previous_probability,
            keys[2]: previous[2] - previous[3] if previous_valid else 0.0,
            keys[3]: previous[4] if previous_valid else 0.0,
            keys[4]: previous[5] if previous_valid else 0.0,
            keys[5]: future_valid,
            keys[6]: future_probability,
            keys[7]: future[2] - future[3] if future_valid else 0.0,
            keys[8]: future[4] if future_valid else 0.0,
            keys[9]: future[5] if future_valid else 0.0,
            keys[10]: (
                float(_norm_like(current - previous_delta))
                if previous_valid else 0.0
            ),
            keys[11]: (
                float(_norm_like(future_delta - current)) if future_valid else 0.0
            ),
            keys[12]: cosine(previous_delta, current) if previous_valid else 0.0,
            keys[13]: cosine(current, future_delta) if future_valid else 0.0,
            keys[14]: (
                speed_ratio(previous_delta, current) if previous_valid else 0.0
            ),
            keys[15]: speed_ratio(current, future_delta) if future_valid else 0.0,
            keys[16]: (
                min(previous_probability, current_probability, future_probability)
                if both else 0.0
            ),
            keys[17]: (
                (previous_probability + current_probability + future_probability) / 3.0
                if both else 0.0
            ),
            keys[18]: (
                float(np.cbrt(max(
                    previous_probability * current_probability * future_probability,
                    0.0,
                ))) if both else 0.0
            ),
            keys[19]: previous_valid * future_valid,
        }
        return values, (
            int(previous[0]) if previous_valid else -1,
            int(future[1]) if future_valid else -1,
        )
    return _model_context


def _context_fixture(seed: int):
    rng = np.random.default_rng(seed)
    spacing = np.array([1.625, 0.40625, 0.40625])
    nodes = {
        index: (rng.normal(0, 40, 3) * spacing).astype(np.float32)
        for index in range(40)
    }
    incoming, outgoing = {}, {}
    for index in range(40):
        if rng.random() < 0.8:
            incoming[index] = (
                int(rng.integers(0, 45)), index, float(rng.random()),
                float(rng.random()), float(rng.integers(0, 2)),
                float(rng.random() * 10),
            )
        if rng.random() < 0.8:
            outgoing[index] = (
                index, int(rng.integers(0, 45)), float(rng.random()),
                float(rng.random()), float(rng.integers(0, 2)),
                float(rng.random() * 10),
            )
    return nodes, incoming, outgoing


def _check_model_context(original, replacement) -> bool:
    rng = np.random.default_rng(51)
    for seed in range(6):
        nodes, incoming, outgoing = _context_fixture(seed)
        for _ in range(300):
            source = int(rng.integers(0, 40))
            target = int(rng.integers(0, 40))
            probability = float(rng.random())
            for prefix in ("ctx_p1", "ctx_p2"):
                left = original(prefix, source, target, probability,
                                nodes, incoming, outgoing)
                right = replacement(prefix, source, target, probability,
                                    nodes, incoming, outgoing)
                if left[1] != right[1]:
                    return False
                if list(left[0].keys()) != list(right[0].keys()):
                    return False
                for key in left[0]:
                    if not _identical(float(left[0][key]), float(right[0][key])):
                        return False
    return True


# ---------------------------------------------------------------------------
# Discovery and installation
#
# The replacements are offered to *every* loaded artifact runtime, not just
# EdgeGRAFT. These modules were written in one style and share helper names,
# but only EdgeGRAFT is available to test against offline, so nothing is
# assumed: a candidate is installed only where its self-check reproduces the
# module's own function bit for bit. A module whose `_cosine` uses a different
# epsilon, or whose `_augment_geometry` computes different features, simply
# fails the check and stays frozen.
# ---------------------------------------------------------------------------
_COSINE_EPSILONS = (1e-8, 1e-6, 1e-10, 1e-12, 0.0)


def _artifact_modules():
    """Loaded modules that live inside an attached dataset, newest first."""
    import sys

    found = []
    for name, module in list(sys.modules.items()):
        if module is None or name.startswith(("numpy", "scipy", "pandas", "torch")):
            continue
        path = getattr(module, "__file__", None) or ""
        lowered = path.replace("\\", "/").lower()
        if "/kaggle/input/" in lowered or "/datasets/" in lowered:
            found.append(module)
    return found


def _install_cosine(module) -> bool:
    """Install a cosine whose epsilon is chosen by what the check accepts."""
    original = getattr(module, "_cosine", None)
    if original is None:
        return False
    for epsilon in _COSINE_EPSILONS:
        replacement = _make_cosine(epsilon)(original)
        try:
            if _check_pairs(original, replacement):
                setattr(module, "_cosine", replacement)
                INSTALLED.append(f"{module.__name__}._cosine (eps={epsilon:g})")
                return True
        except Exception:
            continue
    REJECTED.append(f"{module.__name__}._cosine")
    return False


_CANDIDATES = (
    ("_speed_ratio", _make_speed_ratio, _check_pairs),
    ("_linear_velocity", _make_linear_velocity, _check_linear_velocity),
    ("_model_context", _make_model_context, _check_model_context),
    ("_rows_frame", _make_rows_frame, _check_rows_frame),
    ("_augment_geometry", _make_augment_geometry, _check_row_augmenter),
    ("_augment_trajectory", _make_augment_trajectory, _check_row_augmenter),
)


def install(verbose: bool = True) -> dict:
    """Install every replacement whose self-check passes. Idempotent."""
    INSTALLED.clear()
    REJECTED.clear()
    SKIPPED.clear()

    modules = _artifact_modules()
    for module in modules:
        if hasattr(module, "_cosine"):
            _install_cosine(module)
        for name, make, check in _CANDIDATES:
            if hasattr(module, name):
                _install(module, name, make, check)
        for attribute in vars(module).values():
            # Only where the class defines _decisions itself: subclasses that
            # merely inherit it would otherwise each get a shadowing copy.
            if (
                isinstance(attribute, type)
                and getattr(attribute, "__module__", None) == module.__name__
                and "_decisions" in vars(attribute)
            ):
                _install(attribute, "_decisions", _make_decisions, _check_decisions)

    report = {
        "modules_scanned": len(modules),
        "installed": list(INSTALLED),
        "rejected": list(REJECTED),
        "skipped": list(SKIPPED),
    }
    if verbose:
        print(f"runtime_accel: {len(INSTALLED)} installed, {len(REJECTED)} rejected "
              f"across {len(modules)} artifact modules")
        for name in INSTALLED:
            print(f"  + {name}")
        for name in REJECTED:
            print(f"  - REJECTED, left frozen: {name}")
    return report


# ---------------------------------------------------------------------------
# Function-level timing for the runtimes this repo cannot see
#
# The stage timer says the division chain costs N minutes; it cannot say which
# function inside it does. This wraps every function in the loaded artifact
# modules with a counter, at roughly a microsecond per call, so one profiled
# video names the targets for the next round.
# ---------------------------------------------------------------------------
_PROFILE: dict[str, list] = {}
_PROFILING = False
_PROFILE_RESTORE: list[tuple] = []


def profile_modules() -> int:
    """Wrap every artifact-module function with a wall-clock counter."""
    global _PROFILING
    import time
    import types

    if _PROFILING:
        return len(_PROFILE)
    _PROFILING = True
    wrapped = 0

    stack: list[float] = []

    def make(label, function):
        # [cumulative, calls, exclusive]. Exclusive time is what names a
        # target: a wrapper high on cumulative time is usually just the
        # caller of the one that actually costs something.
        record = _PROFILE.setdefault(label, [0.0, 0, 0.0])

        def timed(*args, **kwargs):
            started = time.perf_counter()
            stack.append(0.0)
            try:
                return function(*args, **kwargs)
            finally:
                elapsed = time.perf_counter() - started
                children = stack.pop()
                record[0] += elapsed
                record[1] += 1
                record[2] += elapsed - children
                if stack:
                    stack[-1] += elapsed
        timed.__name__ = getattr(function, "__name__", label)
        return timed

    for module in _artifact_modules():
        for name, value in list(vars(module).items()):
            # __name__ covers the artifact's own functions; the accelerator's
            # own module covers the replacements installed above, which would
            # otherwise hide inside their caller's self time.
            if isinstance(value, types.FunctionType) and value.__module__ in (
                module.__name__, __name__
            ):
                _PROFILE_RESTORE.append((module, name, value))
                setattr(module, name, make(f"{module.__name__}.{name}", value))
                wrapped += 1
            elif isinstance(value, type) and getattr(value, "__module__", None) == module.__name__:
                for attribute, method in list(vars(value).items()):
                    if isinstance(method, types.FunctionType) and method.__module__ in (
                        module.__name__, __name__
                    ):
                        _PROFILE_RESTORE.append((value, attribute, method))
                        setattr(value, attribute,
                                make(f"{module.__name__}.{name}.{attribute}", method))
                        wrapped += 1
    return wrapped


def unprofile_modules() -> int:
    """Put every wrapped function back, so only the profiled video pays."""
    global _PROFILING
    count = len(_PROFILE_RESTORE)
    for holder, name, original in reversed(_PROFILE_RESTORE):
        try:
            setattr(holder, name, original)
        except Exception:
            pass
    _PROFILE_RESTORE.clear()
    _PROFILING = False
    return count


def profile_report(limit: int = 25) -> list[tuple[str, float, float, int]]:
    """(name, exclusive seconds, cumulative seconds, calls), heaviest first."""
    rows = [
        (name, value[2], value[0], value[1])
        for name, value in _PROFILE.items() if value[1]
    ]
    rows.sort(key=lambda row: -row[1])
    return rows[:limit]


def profile_reset() -> None:
    """Zero the counters. Safe while wrapped: the wrappers hold references to
    the record lists, so clearing the dict would silently orphan them."""
    if _PROFILING:
        for value in _PROFILE.values():
            value[0] = 0.0
            value[1] = 0
            value[2] = 0.0
    else:
        _PROFILE.clear()


## 4. Schedule

Videos are ranked by voxel count and dealt to the GPU shards longest-first; the
geometry-only baseline is produced only for videos the combined upgrade could
not finish; each worker retunes its thread budget between stages.


In [ ]:
"""Streaming inference: cost-ordered GPU producers -> CPU upgrades.

The modelling is `.9525` verbatim. What changes is when work starts and how
many threads it is allowed to use.
"""
import csv
import multiprocessing as mp
import queue as stdlib_queue
import signal
import subprocess
import time
import traceback

import pandas as pd

if os.name != "posix":
    raise RuntimeError("The forked worker pool requires Kaggle's POSIX runtime")

sys.path.insert(0, str(PKG_DIR))
import postprocess as pp

pp.init_division_runtime()
pp.init_edgegraft_runtime()
pp.init_candidategraft_runtime()
pp.init_basegraph_repair_runtime()

RUNTIME = CFG["runtime"]
SPEED = CFG["speed"]

# The artifact runtimes are all imported by now, so this is the one moment
# where every one of them can be seen at once -- and it is before the worker
# pool forks, so the workers inherit the installed replacements.
import runtime_accel

ACCEL_REPORT = {"installed": [], "rejected": [], "modules_scanned": 0}
if SPEED["runtime_accel"]:
    ACCEL_REPORT = runtime_accel.install()
else:
    print("runtime accelerator: disabled by CFG")
METHOD = "unet_transformer"
SHARD_DIR = Path(CFG["paths"]["shard_dir"])
DEADLINE = (
    NOTEBOOK_START + RUNTIME["hard_limit_seconds"] - RUNTIME["finalize_reserve_seconds"]
)
CPU_TOTAL = int(SPEED["thread_pool_total"] or 0) or (os.cpu_count() or 4)

# --------------------------------------------------------------------------
# Cost model
#
# Both halves of the run scale with the size of the volume: the GPU sweeps one
# window per frame, and the CPU upgrade scales with the node count, which is
# the volume times a density the detector has not measured yet. Raw voxels is
# the only estimate available before any work is done, and it only has to be
# good enough to rank four videos.
# --------------------------------------------------------------------------
def _video_cost(stem: str) -> float:
    for meta_path in (
        TEST_DIR / f"{stem}.zarr" / "0" / "zarr.json",
        TEST_DIR / f"{stem}.zarr" / "0" / ".zarray",
    ):
        try:
            shape = json.loads(meta_path.read_text())["shape"]
        except Exception:
            continue
        cost = 1.0
        for extent in shape:
            cost *= float(extent)
        return cost
    return 1.0


VIDEO_COST = {stem: _video_cost(stem) for stem in TEST_STEMS}
COST_ORDER = sorted(TEST_STEMS, key=lambda stem: (-VIDEO_COST[stem], stem))

# Which video, if any, gets the function-level profiler. The smallest one is
# the cheapest place to buy the breakdown, and the code path is the same.
_PROFILE_MODE = str(SPEED["function_profile"]).strip().lower()
PROFILE_ALL = _PROFILE_MODE == "all"
PROFILE_TARGET = COST_ORDER[-1] if (_PROFILE_MODE == "smallest" and COST_ORDER) else ""
if _PROFILE_MODE != "off":
    print(f"function profile: {_PROFILE_MODE} "
          f"(target={'every video' if PROFILE_ALL else PROFILE_TARGET or 'none'})")

# --------------------------------------------------------------------------
# GPU producers
# --------------------------------------------------------------------------
predict_cmd = [
    sys.executable, "scripts/predict_unet_transformer.py",
    "--data-dir", str(TEST_DIR),
    "--split", "0",
    "--weights", P1_WEIGHTS_REL,
    "--unet-batch-size", str(CFG["detection"]["unet_batch_size"]),
    "--det-threshold", str(CFG["detection"]["threshold"]),
    "--ilp-edge-weight", str(CFG["ilp"]["edge_weight"]),
    "--ilp-appearance-weight", str(CFG["ilp"]["appearance"]),
    "--ilp-disappearance-weight", str(CFG["ilp"]["disappearance"]),
    "--ilp-division-weight", str(CFG["ilp"]["division"]),
]
if CFG["ilp"]["use"]:
    predict_cmd.append("--use-ilp")
slice_spec = str(RUNTIME["slice"]).strip()

visible = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
cuda_tokens = (
    [token.strip() for token in visible.split(",") if token.strip()]
    if visible and visible != "-1"
    else [str(index) for index in range(torch.cuda.device_count())]
)
shard_count = min(RUNTIME["gpu_workers"], len(cuda_tokens), len(TEST_STEMS))
sharded = shard_count >= 2 and not slice_spec

if not sharded:
    shard_stems = [list(TEST_STEMS)]
elif SPEED["longest_first"]:
    # Longest-processing-time-first: deal each video, largest first, to the
    # shard with the least work queued. The frozen `--slice i::n` handed the
    # heaviest video to whichever shard the sorted stem order happened to put
    # it in, and on the local set that was last.
    load = [0.0] * shard_count
    shard_stems = [[] for _ in range(shard_count)]
    for stem in COST_ORDER:
        index = min(range(shard_count), key=lambda k: (load[k], k))
        shard_stems[index].append(stem)
        load[index] += VIDEO_COST[stem]
else:
    shard_stems = [list(TEST_STEMS[index::shard_count]) for index in range(shard_count)]

helper_deadline_epoch = time.time() + max(0.0, DEADLINE - time.monotonic())

predict_processes: dict[int, subprocess.Popen] = {}
predict_commands: dict[int, list[str]] = {}
predict_methods: list[str] = []
predict_started = time.monotonic()

for index, stems in enumerate(shard_stems):
    if not stems:
        continue
    method = f"{METHOD}_gpu{index}" if sharded else METHOD
    splits_path = REPO_DIR / f"kaggle_test_splits_{index}.json"
    splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": stems}]))
    command = [*predict_cmd, "--splits", splits_path.name]
    if sharded:
        command += ["--method", method]
    if slice_spec:
        command += ["--slice", slice_spec]
    environment = {
        **os.environ,
        "PYTHONPATH": "src",
        "CUDA_VISIBLE_DEVICES": cuda_tokens[index],
        "BIOHUB_HELPER_DEADLINE_EPOCH": str(helper_deadline_epoch),
    }
    predict_methods.append(method)
    predict_commands[index] = command
    predict_processes[index] = subprocess.Popen(command, cwd=REPO_DIR, env=environment)

print(f"GPU producers: {len(predict_processes)} on CUDA {cuda_tokens[:len(predict_processes)]}")
for index, stems in enumerate(shard_stems):
    if stems:
        total = sum(VIDEO_COST[stem] for stem in stems) / 1e9
        print(f"  gpu{index}: {', '.join(stems)}  ({total:,.1f} Gvoxel)")

# --------------------------------------------------------------------------
# CPU upgrade workers
# --------------------------------------------------------------------------
CPU_WORKERS = max(1, int(RUNTIME["cpu_workers"] or 0) or CPU_TOTAL)
_while_predicting = int(RUNTIME["cpu_workers_while_predicting"] or 0)
CPU_WORKERS_WHILE_PREDICTING = max(
    1,
    min(_while_predicting or max(1, CPU_TOTAL - len(predict_processes)), CPU_WORKERS),
)


class _UpgradeDeadline(TimeoutError):
    pass


def _on_alarm(signum, frame):
    raise _UpgradeDeadline("upgrade reached the submission-safe deadline")


def _thread_budget(active_tasks, live_producers) -> int:
    """Threads for one task, given what else is currently running.

    The GPU producers are not idle while they wait on the device -- they
    decode zarr, run the ILP, and build graphs -- so each one holds back a
    core for as long as it lives.
    """
    if not SPEED["thread_boost"]:
        return 1
    active = max(1, int(active_tasks.value))
    reserved = max(0, int(live_producers.value))
    return max(1, (CPU_TOTAL - reserved) // active)


def _worker_main(worker_id, task_queue, result_queue, active_tasks, live_producers) -> None:
    signal.signal(signal.SIGALRM, _on_alarm)
    pp.THREAD_RETUNE_HOOK = lambda: pp.set_thread_budget(
        _thread_budget(active_tasks, live_producers)
    )
    pp.set_thread_budget(1)
    while True:
        task = task_queue.get()
        if task is None:
            return
        dataset, graph_path, mode = task
        started = time.monotonic()
        remaining = DEADLINE - started
        if remaining <= 0:
            result_queue.put({"worker_id": worker_id, "dataset": dataset, "mode": mode,
                              "status": "deadline", "error": "no time left"})
            continue
        with active_tasks.get_lock():
            active_tasks.value += 1
        signal.setitimer(signal.ITIMER_REAL, max(remaining, 1.0))
        try:
            pp.retune_threads()
            profiling = (
                (PROFILE_ALL or dataset == PROFILE_TARGET) and mode == "combined"
            )
            if profiling:
                runtime_accel.profile_reset()
                runtime_accel.profile_modules()
            try:
                if mode == "emergency":
                    row = pp.emergency_shard(Path(graph_path))
                else:
                    row = pp.process_graph(Path(graph_path), mode)
            finally:
                if profiling:
                    # Unwrap immediately: the rest of this worker's videos
                    # must not pay for one video's diagnostics.
                    profile = runtime_accel.profile_report(25)
                    runtime_accel.unprofile_modules()
            result_queue.put({"worker_id": worker_id, "dataset": dataset, "mode": mode,
                              "status": "ok", "seconds": time.monotonic() - started,
                              "row": row,
                              "profile": profile if profiling else None})
        except _UpgradeDeadline as error:
            result_queue.put({"worker_id": worker_id, "dataset": dataset, "mode": mode,
                              "status": "deadline", "error": str(error)})
        except BaseException as error:
            result_queue.put({"worker_id": worker_id, "dataset": dataset, "mode": mode,
                              "status": "error", "error": f"{type(error).__name__}: {error}",
                              "traceback": traceback.format_exc(limit=12)})
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0.0)
            pp.set_thread_budget(1)
            with active_tasks.get_lock():
                active_tasks.value -= 1


context = mp.get_context("fork")
result_queue = context.Queue()
task_queues = [context.Queue(maxsize=1) for _ in range(CPU_WORKERS)]
active_tasks = context.Value("i", 0)
live_producers = context.Value("i", len(predict_processes))
workers = []
for worker_id in range(CPU_WORKERS):
    process = context.Process(
        target=_worker_main,
        args=(worker_id, task_queues[worker_id], result_queue, active_tasks, live_producers),
        name=f"biohub-upgrade-{worker_id}",
        daemon=False,
    )
    process.start()
    workers.append(process)
worker_task: dict[int, tuple[str, str] | None] = {
    worker_id: None for worker_id in range(CPU_WORKERS)
}
print(
    f"CPU upgrade workers: {CPU_WORKERS} ({CPU_WORKERS_WHILE_PREDICTING} while the GPU runs) "
    f"| {CPU_TOTAL} vCPU | thread boost={SPEED['thread_boost']}"
)

# --------------------------------------------------------------------------
# Scheduler
#
# One queue of (dataset, mode) work. `combined` is the real pipeline;
# `baseline` and `emergency` exist only to guarantee that every video ends the
# run with a shard, and are queued on demand rather than up front.
# --------------------------------------------------------------------------
stats_by_dataset: dict[str, dict] = {}
FUNCTION_PROFILES: dict[str, list] = {}
graph_by_dataset: dict[str, Path] = {}
shard_done: set[str] = set()
attempted: dict[str, set[str]] = {}
upgraded: list[str] = []
evidence_roots = [Path(CFG["paths"][key]) for key in
                  ("model_c_evidence_dir", "p1_evidence_dir", "p2_evidence_dir")]
MODE_AFTER = {"combined": "baseline", "baseline": "emergency", "emergency": None}
EAGER_BASELINE = not SPEED["lazy_baseline"]


def _ready_graphs() -> dict[str, Path]:
    ready = {}
    for method in predict_methods:
        for marker in (REPO_DIR / "predictions").glob(f"*/{method}/split_0/*.ready"):
            graph = marker.with_suffix(".geff")
            if graph.exists() and marker.stem in TEST_STEMS:
                ready[marker.stem] = graph
    return ready


def _has_evidence(dataset: str) -> bool:
    return all((root / f"{dataset}.npz").exists() for root in evidence_roots)


def _queue(dataset: str, mode: str, worker_id: int) -> None:
    attempted.setdefault(dataset, set()).add(mode)
    worker_task[worker_id] = (dataset, mode)
    task_queues[worker_id].put((dataset, str(graph_by_dataset[dataset]), mode))


def _next_mode(dataset: str, producer_done: bool) -> str | None:
    """The most valuable pass this video has not been given yet, or None."""
    tried = attempted.setdefault(dataset, set())
    if "combined" not in tried:
        if _has_evidence(dataset):
            return "combined"
        # The division helpers write no .npz past their own deadline, so a
        # video can wait for evidence that will never arrive. Once the
        # producers have exited, what is on disk is all there will ever be:
        # retire the combined pass so the video falls through to a baseline
        # instead of being re-queued for it on every tick.
        if not producer_done:
            return None
        tried.add("combined")
    if dataset in shard_done:
        return None
    if "baseline" not in tried:
        return "baseline"
    if RUNTIME["emergency_shard"] and "emergency" not in tried:
        return "emergency"
    return None


def _collect_results() -> None:
    while True:
        try:
            result = result_queue.get_nowait()
        except stdlib_queue.Empty:
            return
        worker_task[int(result["worker_id"])] = None
        dataset = str(result["dataset"])
        mode = str(result.get("mode", "combined"))
        if result["status"] == "ok":
            row = result["row"]
            stats_by_dataset[dataset] = row
            shard_done.add(dataset)
            if mode == "combined":
                upgraded.append(dataset)
            print(
                f"[{mode} {len(shard_done):3d}/{len(TEST_STEMS)}] {dataset}: "
                f"{result['seconds'] / 60:.1f}m "
                f"nodes={row.get('nodes', 0):,} edges={row.get('edges', 0):,} "
                f"wall={(time.monotonic() - NOTEBOOK_START) / 3600:.2f}h",
                flush=True,
            )
            if SPEED["stage_timing"]:
                print(f"           stages: {pp.stage_summary(row)}", flush=True)
            if result.get("profile"):
                FUNCTION_PROFILES[dataset] = result["profile"]
                print(f"           function profile ({len(result['profile'])} rows) "
                      f"captured for {dataset}", flush=True)
        else:
            print(f"[fallback] {dataset}: {mode} {result['status']} "
                  f"{result.get('error', '')}", flush=True)
            if result.get("traceback"):
                print(result["traceback"], flush=True)


producer_finished_at = None
while True:
    graph_by_dataset.update(_ready_graphs())

    _collect_results()
    for worker_id, process in enumerate(workers):
        if process.exitcode is not None and worker_task[worker_id] is not None:
            dataset, mode = worker_task[worker_id]
            print(f"[worker exit] {dataset}: {mode} did not return", flush=True)
            worker_task[worker_id] = None

    producer_done = all(process.poll() is not None for process in predict_processes.values())
    if producer_done and producer_finished_at is None:
        producer_finished_at = time.monotonic()
        with live_producers.get_lock():
            live_producers.value = 0
    else:
        alive = sum(1 for process in predict_processes.values() if process.poll() is None)
        if alive != live_producers.value:
            with live_producers.get_lock():
                live_producers.value = alive
    remaining = DEADLINE - time.monotonic()

    busy_datasets = {task[0] for task in worker_task.values() if task is not None}
    pending: list[tuple[str, str]] = []
    for dataset in graph_by_dataset:
        if dataset in busy_datasets:
            continue
        mode = _next_mode(dataset, producer_done)
        if mode is None:
            continue
        if EAGER_BASELINE and mode == "combined" and dataset not in shard_done:
            mode = "baseline"
        pending.append((dataset, mode))
    # Longest-processing-time-first: the video that owns the makespan must not
    # wait behind three short ones for a worker.
    pending.sort(key=lambda item: (-VIDEO_COST.get(item[0], 0.0), item[0]))

    if remaining > RUNTIME["min_upgrade_seconds"] and pending:
        active_limit = CPU_WORKERS if producer_done else CPU_WORKERS_WHILE_PREDICTING
        idle = [
            worker_id for worker_id in range(active_limit)
            if worker_task[worker_id] is None and workers[worker_id].is_alive()
        ]
        for worker_id, (dataset, mode) in zip(idle, pending):
            _queue(dataset, mode, worker_id)

    busy = any(task is not None for task in worker_task.values())
    complete = producer_done and len(shard_done) == len(TEST_STEMS)
    if complete and not busy and not pending:
        break
    if remaining <= 0 and producer_done and not busy:
        break
    if producer_done and not busy and not pending and not complete:
        break
    time.sleep(2.0)

_collect_results()
for worker_id, task_queue in enumerate(task_queues):
    if workers[worker_id].is_alive():
        task_queue.put(None)
for process in workers:
    process.join(timeout=10)
    if process.is_alive():
        process.terminate()
        process.join(timeout=5)

for index, process in predict_processes.items():
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, predict_commands[index])

# Anything the pool could not finish is rescued here, in this process, so a
# single failed video cannot cost the whole submission.
pp.set_thread_budget(CPU_TOTAL)
signal.signal(signal.SIGALRM, _on_alarm)
HARD_STOP = NOTEBOOK_START + RUNTIME["hard_limit_seconds"] - 120.0
for dataset in sorted(set(TEST_STEMS) - shard_done):
    graph = graph_by_dataset.get(dataset)
    if graph is None:
        raise RuntimeError(f"Producer finished but no graph was written for {dataset}")
    for mode in ("baseline", "emergency"):
        if mode == "emergency" and not RUNTIME["emergency_shard"]:
            continue
        budget = HARD_STOP - time.monotonic()
        if budget <= 0:
            print(f"[rescue] {dataset}: out of time before {mode}", flush=True)
            break
        signal.setitimer(signal.ITIMER_REAL, budget)
        try:
            stats_by_dataset[dataset] = (
                pp.emergency_shard(graph) if mode == "emergency"
                else pp.process_graph(graph, mode)
            )
            shard_done.add(dataset)
            print(f"[rescue] {dataset}: {mode} shard written", flush=True)
            break
        except BaseException as error:
            print(f"[rescue] {dataset}: {mode} failed - "
                  f"{type(error).__name__}: {error}", flush=True)
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0.0)

missing = sorted(set(TEST_STEMS) - shard_done)
if missing:
    raise RuntimeError(f"Producer finished but shards are missing: {missing[:10]}")

# --------------------------------------------------------------------------
# Submission
# --------------------------------------------------------------------------
row_id = node_rows = edge_rows = 0
with SUBMISSION_PATH.open("w", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=pp.CSV_COLUMNS)
    writer.writeheader()
    for dataset in sorted(TEST_STEMS):
        shard = SHARD_DIR / f"{dataset}.csv"
        if not shard.exists():
            raise FileNotFoundError(f"Missing shard: {shard}")
        with shard.open(newline="") as handle:
            for row in csv.DictReader(handle):
                row["id"] = row_id
                writer.writerow(row)
                row_id += 1
                node_rows += row["row_type"] == "node"
                edge_rows += row["row_type"] == "edge"

assert node_rows > 0 and row_id == node_rows + edge_rows

predict_hours = ((producer_finished_at or time.monotonic()) - predict_started) / 3600.0
stats = pd.DataFrame(
    [stats_by_dataset[dataset] for dataset in sorted(stats_by_dataset)]
).sort_values("dataset").reset_index(drop=True)
stats["experiment_tag"] = CFG["experiment_tag"]
stats["predict_hours"] = predict_hours
stats["upgraded_total"] = len(upgraded)
stats["wall_hours"] = (time.monotonic() - NOTEBOOK_START) / 3600.0
stats.to_csv(RUN_STATS_PATH, index=False)

print(f"\n{SUBMISSION_PATH}: {row_id:,} rows ({node_rows:,} nodes, {edge_rows:,} edges)")
print(f"Combined-decoder upgrades: {len(upgraded)}/{len(TEST_STEMS)} "
      f"| GPU {predict_hours:.2f}h | wall {(time.monotonic() - NOTEBOOK_START) / 3600:.2f}h")

if CFG["graph"]["relink_joint_assignment"] and "motion_relink_joint_frames" in stats.columns:
    print(f"\njoint assignment: tight bonus "
          f"{CFG['graph']['relink_tight_bonus_um']} um, "
          f"{int(stats['motion_relink_joint_frames'].fillna(0).sum()):,} frame pairs solved once")
    print(stats[["dataset", "motion_relink_tight_edges",
                 "motion_relink_relaxed_edges", "relink_cost_cap_rejected"]]
          .to_string(index=False))
    print("  tight/relaxed here count the gate each accepted match fell in, "
          "not which pass produced it")
    print("  relink_cost_cap_rejected is NOT comparable with the two-pass "
          "notebooks: one solve screens the whole 10 um candidate set, where "
          "the tight pass only ever screened 6 um, so the count rises by "
          "orders of magnitude without a single extra edge being lost")

_cap = CFG["graph"]["relink_max_match_cost"]
if _cap > 0 and "relink_cost_cap_rejected" in stats.columns:
    _rejected = int(stats["relink_cost_cap_rejected"].fillna(0).sum())
    _kept = int(stats["motion_relink_edges"].fillna(0).sum())
    print(f"refusal: cap {_cap} um rejected {_rejected:,} candidate assignments; "
          f"{_kept:,} edges kept")
    print(stats[["dataset", "relink_cost_cap_rejected", "motion_relink_edges",
                 "motion_relink_tight_edges", "motion_relink_relaxed_edges"]]
          .to_string(index=False))

evidence_bytes = sum(
    path.stat().st_size for root in evidence_roots for path in root.glob("*.npz")
)
print(f"native evidence on disk: {evidence_bytes / 1e9:.2f} GB "
      f"(uncompressed={SPEED['uncompressed_evidence']}, "
      f"cap={SPEED['uncompressed_evidence_max_mb']} MB per archive)")

if ACCEL_REPORT["installed"] or ACCEL_REPORT["rejected"]:
    print(f"\n--- runtime accelerator: {len(ACCEL_REPORT['installed'])} installed, "
          f"{len(ACCEL_REPORT['rejected'])} rejected across "
          f"{ACCEL_REPORT['modules_scanned']} artifact modules ---")

for dataset, profile in FUNCTION_PROFILES.items():
    print(f"\n--- function profile: {dataset} (self seconds, artifact runtimes) ---")
    print(f"{'self':>8} {'cumulative':>11} {'calls':>12}  function")
    for name, self_seconds, cumulative, calls in profile:
        print(f"{self_seconds:8.2f} {cumulative:11.2f} {calls:12,}  {name}")
    frame = pd.DataFrame(
        profile, columns=["function", "self_seconds", "cumulative_seconds", "calls"]
    )
    frame.to_csv(Path(CFG["paths"]["working_dir"]) / f"function_profile_{dataset}.csv",
                 index=False)

if SPEED["stage_timing"]:
    print("\n--- where the CPU upgrade time went (minutes) ---")
    timing = [column for column in stats.columns if column.startswith("t_")]
    if timing:
        table = stats[["dataset", *timing]].copy()
        for column in timing:
            table[column] = (table[column].astype(float) / 60.0).round(2)
        table = table.rename(columns={column: column[2:] for column in timing})
        totals = table[[c for c in table.columns if c != "dataset"]].sum()
        print(table.to_string(index=False))
        print("\ntotal per stage:")
        print(totals.sort_values(ascending=False).head(12).to_string())


## 5. Local scoring

Structural check, then the competition metric when a labelled `train/` is
attached: `adjusted edge Jaccard + 0.1 * division Jaccard`. Per-video rows go to
`local_metrics.csv`.


In [ ]:
"""Scoring cell: competition metric first, structural checks second.

Two modes, selected automatically:

* **Scored** â€” a ground-truth `.geff` exists for the predicted videos (point
  `BIOHUB_TEST_DIR` / `CFG["metrics"]["gt_dir"]` at the labelled `train/`
  split). The submission is rebuilt into tracksdata graphs and evaluated with
  the official metric: node matching at 7 um, adjusted edge Jaccard with the
  node-count penalty, division Jaccard over the local division windows, and
  `score = adj_edge_jaccard + 0.1 * division_jaccard`.
* **Audit** â€” no ground truth (the real test run). Every quantity the metric
  reacts to is still reported per video: predicted node count against the raw
  detector count (the node-penalty proxy), fork count, edge/node ratio,
  synthetic-node share and track-length distribution.
"""
import numpy as np
import pandas as pd

METRICS = CFG["metrics"]
MAX_MATCH_UM = float(METRICS["max_match_um"])
ALPHA = float(METRICS["adjustment_alpha"])
DIVISION_WEIGHT = float(METRICS["division_weight"])

submission = pd.read_csv(SUBMISSION_PATH)

# ---------------------------------------------------------------- structure
if list(submission.columns) != ["id", *pp.SUBMISSION_COLUMNS]:
    raise AssertionError(f"Unexpected columns: {list(submission.columns)}")
if not submission["id"].is_monotonic_increasing or submission["id"].duplicated().any():
    raise AssertionError("Row ids must be unique and increasing")

nodes = submission[submission["row_type"] == "node"]
edges = submission[submission["row_type"] == "edge"]
if len(nodes) + len(edges) != len(submission):
    raise AssertionError("Unknown row_type present")

for dataset, group in submission.groupby("dataset", sort=False):
    group_nodes = group[group["row_type"] == "node"]
    group_edges = group[group["row_type"] == "edge"]
    known = set(group_nodes["node_id"])
    if group_nodes["node_id"].duplicated().any():
        raise AssertionError(f"{dataset}: duplicate node_id")
    if not set(group_edges["source_id"]).issubset(known):
        raise AssertionError(f"{dataset}: edge source outside its own node set")
    if not set(group_edges["target_id"]).issubset(known):
        raise AssertionError(f"{dataset}: edge target outside its own node set")
    if (group_nodes[["t", "z", "y", "x"]] < 0).any().any():
        raise AssertionError(f"{dataset}: negative node coordinate")

in_degree = edges.groupby(["dataset", "target_id"]).size()
out_degree = edges.groupby(["dataset", "source_id"]).size()
if len(in_degree) and int(in_degree.max()) > 1:
    raise AssertionError("A node has more than one parent")
if len(out_degree) and int(out_degree.max()) > 2:
    raise AssertionError("A node has more than two children")

print(
    f"structure ok | {len(nodes):,} nodes, {len(edges):,} edges, "
    f"{len(submission['dataset'].unique())} videos, "
    f"{int((out_degree >= 2).sum()):,} forks"
)


# ------------------------------------------------------------------- audit
def _time_span(group_nodes, group_edges):
    """Mean number of nodes per weakly connected component."""
    parent = {int(node_id): int(node_id) for node_id in group_nodes["node_id"]}

    def find(node_id):
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    for source_id, target_id in zip(group_edges["source_id"], group_edges["target_id"]):
        root_a, root_b = find(int(source_id)), find(int(target_id))
        if root_a != root_b:
            parent[root_a] = root_b
    sizes = {}
    for node_id in parent:
        root = find(node_id)
        sizes[root] = sizes.get(root, 0) + 1
    return (np.mean(list(sizes.values())) if sizes else 0.0), len(sizes)


run_stats = pd.read_csv(RUN_STATS_PATH).set_index("dataset")
audit_rows = []
for dataset, group in submission.groupby("dataset", sort=True):
    group_nodes = group[group["row_type"] == "node"]
    group_edges = group[group["row_type"] == "edge"]
    forks = int((group_edges.groupby("source_id").size() >= 2).sum())
    mean_component, components = _time_span(group_nodes, group_edges)
    raw_nodes = float(run_stats.loc[dataset, "raw_nodes"]) if dataset in run_stats.index else float("nan")
    synthetic = float(run_stats.loc[dataset, "gap_inserted_synthetic"]) if dataset in run_stats.index else float("nan")
    node_ratio = len(group_nodes) / raw_nodes if raw_nodes > 0 else float("nan")
    audit_rows.append({
        "dataset": dataset,
        "nodes": len(group_nodes),
        "edges": len(group_edges),
        "forks": forks,
        "raw_detected_nodes": raw_nodes,
        "node_ratio_vs_detected": node_ratio,
        # Same shape as the metric's penalty, with the detector count standing
        # in for the organisers' T_true estimate.
        "penalty_proxy": (
            max(0.0, 1.0 - ALPHA * (node_ratio - 1.0)) if node_ratio == node_ratio else float("nan")
        ),
        "synthetic_nodes": synthetic,
        "edges_per_node": len(group_edges) / max(len(group_nodes), 1),
        "components": components,
        "mean_component_nodes": mean_component,
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(Path(CFG["paths"]["working_dir"]) / "submission_audit.csv", index=False)
print("\n--- metric-aware audit (per video) ---")
print(audit.to_string(index=False, float_format=lambda value: f"{value:,.3f}"))
print(
    "\ntotals: nodes={:,} edges={:,} forks={:,} | node/detected={:.3f} "
    "| penalty proxy={:.4f} | forks per 1k nodes={:.2f}".format(
        int(audit["nodes"].sum()), int(audit["edges"].sum()), int(audit["forks"].sum()),
        audit["nodes"].sum() / max(audit["raw_detected_nodes"].sum(), 1),
        max(0.0, 1.0 - ALPHA * (audit["nodes"].sum() / max(audit["raw_detected_nodes"].sum(), 1) - 1.0)),
        1000.0 * audit["forks"].sum() / max(audit["nodes"].sum(), 1),
    )
)


# ------------------------------------------------------- competition metric
def _ground_truth_dir():
    configured = str(METRICS.get("gt_dir", "")).strip()
    if configured:
        return Path(configured)
    for candidate in (TEST_DIR, Path(CFG["paths"]["competition_dir"]) / "train"):
        if candidate.exists() and any(candidate.glob("*.geff")):
            return candidate
    return TEST_DIR


def _estimated_nodes(geff_path):
    """`estimated_number_of_nodes` from the GEFF metadata extras (else NaN)."""
    try:
        meta = json.loads((Path(geff_path) / "zarr.json").read_text())
        extra = meta["attributes"]["geff"].get("extra") or {}
        value = extra.get("estimated_number_of_nodes")
        return float(value) if value is not None else float("nan")
    except Exception:
        return float("nan")


def _predicted_graph(group):
    import polars as pl
    import tracksdata as td

    graph = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        graph.add_node_attr_key(key, pl.Float64, -999999.0)
    group_nodes = group[group["row_type"] == "node"]
    node_ids = graph.bulk_add_nodes([
        {"t": int(row.t), "z": float(row.z), "y": float(row.y), "x": float(row.x)}
        for row in group_nodes.itertuples()
    ])
    remap = dict(zip(group_nodes["node_id"].astype(int), node_ids))
    group_edges = group[group["row_type"] == "edge"]
    if len(group_edges):
        graph.bulk_add_edges([
            {"source_id": remap[int(row.source_id)], "target_id": remap[int(row.target_id)]}
            for row in group_edges.itertuples()
        ])
    return graph


GT_DIR = _ground_truth_dir()
scored_datasets = [
    dataset for dataset in sorted(submission["dataset"].unique())
    if (GT_DIR / f"{dataset}.geff").exists()
]
limit = int(METRICS.get("max_videos", 0) or 0)
if limit > 0:
    scored_datasets = scored_datasets[:limit]

if not METRICS.get("enabled", True):
    print("\nlocal scoring disabled in CFG['metrics'].")
elif not scored_datasets:
    print(
        f"\nno ground truth under {GT_DIR} â€” audit only.\n"
        "For a scored dry run set CFG['paths']['test_dir'] (or BIOHUB_TEST_DIR) to the\n"
        "labelled train/ directory and CFG['runtime']['slice'] to a handful of videos."
    )
else:
    sys.path.insert(0, str(REPO_DIR / "src"))
    import tracksdata as td
    from biohub_tracking.metrics import (
        evaluate as evaluate_pair,
        node_recall,
        per_sample_metrics,
        summarise,
    )

    rows = []
    for dataset in scored_datasets:
        gt_path = GT_DIR / f"{dataset}.geff"
        gt_graph = td.graph.IndexedRXGraph.from_geff(gt_path)
        if isinstance(gt_graph, tuple):
            gt_graph = gt_graph[0]
        predicted = _predicted_graph(submission[submission["dataset"] == dataset])
        result = evaluate_pair(
            predicted, gt_graph,
            scale=tuple(CFG["voxel_scale_um"]),
            max_distance=MAX_MATCH_UM,
        )
        row = per_sample_metrics(
            result, _estimated_nodes(gt_path), node_recall(predicted, gt_graph),
        )
        row["dataset"] = dataset
        rows.append(row)
        print(
            f"  {dataset}: edge J={row['edge_jaccard']:.4f} adj={row['adj_edge_jaccard']:.4f} "
            f"div TP/FP/FN={row['division_tp']}/{row['division_fp']}/{row['division_fn']} "
            f"node recall={row['node_recall']:.4f} node ratio={row['total_node_ratio']:+.3f}",
            flush=True,
        )

    per_video = pd.DataFrame(rows)
    per_video.to_csv(METRICS["report_path"], index=False)
    summary = summarise(rows)
    division_jaccard = summary["division_jaccard"]
    score = summary["adj_edge_jaccard"] + DIVISION_WEIGHT * (
        0.0 if division_jaccard != division_jaccard else division_jaccard
    )
    print(
        "\n=== competition metric ({} videos) ===\n"
        "adjusted edge Jaccard : {:.4f}\n"
        "raw edge Jaccard      : {:.4f}\n"
        "division Jaccard      : {:.4f}   (TP={} FP={} FN={})\n"
        "node recall           : {:.4f}\n"
        "score                 : {:.4f}   = adj_edge + {:g} * division".format(
            summary["n"], summary["adj_edge_jaccard"], summary["edge_jaccard"],
            float("nan") if division_jaccard != division_jaccard else division_jaccard,
            summary["division_tp"], summary["division_fp"], summary["division_fn"],
            summary["node_recall"], score, DIVISION_WEIGHT,
        )
    )
    print(f"per-video metrics written to {METRICS['report_path']}")
